# Suficiencia energética del SIN 2026-2030 y expansión de mínimo costo

**Tutorial del Trabajo Final · Taller de Programación en Python (MECA 4110), 2026-2 · Universidad de los Andes**
Carlos Andrés Terrassa Vallejo (código 200518213) · Esteban Darío Herrera León (código 202424968) · Jair Armando Castañeda Rodríguez (código 202616631)

Este tutorial responde qué expansión de mínimo costo necesita el Sistema Interconectado Nacional (SIN) de Colombia para que su energía firme cubra la demanda proyectada en cada período del Cargo por Confiabilidad, de diciembre de 2026 a noviembre de 2030. Para eso mide cuánta energía firme falta en cada período y resuelve un modelo de optimización con variables enteras. Todo el código está en este notebook, y los datos vienen en la carpeta `datos_tutorial/` del .zip: corre de principio a fin sin internet.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cterrassa/suficiencia-sin/blob/main/Tutorial_Suficiencia_SIN.ipynb) [![Abrir en Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/cterrassa/suficiencia-sin/HEAD?urlpath=lab/tree/Tutorial_Suficiencia_SIN.ipynb) · [Versión web, ya ejecutada](https://cterrassa.github.io/suficiencia-sin/) · [App interactiva](https://cterrassa.github.io/suficiencia-sin/app/) · [Repositorio](https://github.com/cterrassa/suficiencia-sin)

**Contenido**

1. [Contexto y motivación](#contexto): el problema, la pregunta, los antecedentes y por qué Python
2. [Datos](#datos): instalación, fuentes, el índice de El Niño, limpieza, revisión y uniones
3. [Metodología y desarrollo](#metodologia): homologación con RapidFuzz, costos con la fuente de cada parámetro, balance de energía firme y modelo de expansión
4. [Resultados y visualización](#resultados): siete figuras, la respuesta a la pregunta y sus limitaciones
5. [Uso de IA para ir más allá de la metodología](#mas-alla): la consulta, la respuesta completa, lo que va más allá de lo que recomendaron los profesores y dos mejoras con su efecto

[Cómo atendimos la evaluación de la propuesta](#evaluacion) · [Declaración de uso de IA](#ia) · [Referencias](#referencias)

**Cómo está hecho.** Cada paso sigue el mismo orden: un texto que explica qué se va a hacer y por qué, el código, y una lectura del resultado. Las funciones propias tienen docstring, y la tabla de las funciones y los métodos de cada sección, con los parámetros que usamos y por qué, está en `docs/funciones_y_metodos.md`, en el .zip y en el [repositorio](https://github.com/cterrassa/suficiencia-sin/blob/main/docs/funciones_y_metodos.md). Tres ejercicios que se revisan solos, con sus soluciones, están en el notebook `Ejercicios_Suficiencia_SIN.ipynb`, que viene en el .zip.

**Qué se necesita.** Python 3.13 o 3.14 con los paquetes de la sección 2.1, y el notebook al lado de la carpeta `datos_tutorial/`, como vienen en el .zip. No hace falta internet mientras esa carpeta esté completa; solo si falta algún archivo, el notebook lo descarga del repositorio público en GitHub y comprueba su huella SHA-256. Con los botones de Colab o de Binder corre en el navegador, sin instalar nada, y para mover los supuestos del modelo sin correr código está la [app interactiva](https://cterrassa.github.io/suficiencia-sin/app/).

<a id="contexto" name="contexto"></a>
## 1. Contexto y motivación

En esta sección planteamos el problema, la pregunta con la que lo abordamos, lo que ya dice la literatura y por qué escogimos Python. De aquí salen los tres insumos que buscamos en la sección 2.

### El problema

El SIN depende del agua. De los 22,8 GW de capacidad efectiva que operaban o estaban en pruebas en octubre de 2026 (sin contar las plantas pequeñas que probablemente son generación distribuida), 13,0 GW eran hidráulicos (sección 3.3).

Cuando llega El Niño, llueve menos en los Andes colombianos y bajan los caudales de los ríos que alimentan los embalses (Poveda et al., 2011), de modo que el sistema queda en manos del agua embalsada y de las térmicas. El precio lo refleja: en los meses de hidrología seca, el precio de bolsa real promedia el doble que en los normales (sección 2.5), y Ríos et al. (2024) muestran que ese encarecimiento del mercado mayorista termina trasladándose a las tarifas de los usuarios.

Para asegurar que haya con qué atender la demanda en un año seco, la CREG creó en 2006 el Cargo por Confiabilidad (Resolución 071 de 2006). Cada planta tiene una energía firme, la ENFICC: la que puede garantizar con la hidrología crítica de la serie histórica, si es hidráulica, o con su disponibilidad y su combustible, si es térmica. En subastas, el sistema compra obligaciones de energía firme (OEF) para cada período de diciembre a noviembre; quien las vende recibe un pago fijo y, a cambio, se compromete a entregar esa energía cuando el precio de bolsa supera el precio de escasez. Por eso, preguntarse si el sistema es suficiente es preguntarse por su energía firme: si la ENFICC del parque, más la de los proyectos comprometidos en las subastas, cubre la demanda que proyecta la UPME.

Tenemos cuatro razones para dudarlo:
- **Los proyectos llegan tarde.** De la OEF que la subasta de 2019 asignó a plantas nuevas, el 47% entró tarde o no ha entrado (sección 3.1).
- **La demanda subió.** Para 2029-2030, la proyección media de la UPME de julio de 2026 es 5% mayor que la de enero de 2025, con la que la CREG convocó la última subasta (Documento CREG 901 202 de 2025; sección 2.5).
- **El operador advierte un faltante.** Según la prensa, el Boletín 364 de XM estima hasta 7.669 GWh de faltante de energía firme en 2026-2027.
- **Hay un El Niño fuerte en curso.** El índice ONI llegó a 2,46 °C en agosto-octubre de 2026 (sección 2.3).

Un faltante de energía firme no significa un apagón seguro, porque en un año de lluvias normales la hidráulica genera más que su energía firme. Significa exposición a un racionamiento en el próximo año seco. Y como construir toma años (una térmica de ciclo abierto que se decida hoy no opera antes de diciembre de 2028; sección 3.2), cualquier decisión de expansión para 2028-2030 tiene que tomarse ya.

### La pregunta

Todo el tutorial gira alrededor de una sola pregunta:

> **¿Qué expansión de mínimo costo necesita el Sistema Interconectado Nacional (SIN) para que su energía firme cubra la demanda proyectada por la UPME en cada período del Cargo por Confiabilidad entre 2026-2027 y 2029-2030?**

**Qué abarca:**
- **Unidad:** el SIN de Colombia; no incluye las zonas no interconectadas.
- **Período:** los cuatro períodos del Cargo por Confiabilidad, de diciembre de 2026 a noviembre de 2030, con datos al 4 de noviembre de 2026.
- **Variables:** la energía firme del parque (ENFICC) y de los proyectos con OEF de las subastas, con sus retrasos; la demanda total proyectada por la UPME (revisión de julio de 2026), en sus escenarios medio y alto, y los costos y supuestos de cada tecnología, cada uno con su fuente (sección 3.2).
- **Alcance:** energía firme, la que el parque garantiza con la hidrología crítica, y como contraste un año seco y un año normal como los observados desde 2010. No incluye la potencia hora a hora ni la red de transmisión; el precio de bolsa queda como contexto (sección 2.5).

**Cómo la respondemos.** En tres pasos, cada uno insumo del siguiente: medimos cuánta energía firme falta en cada período (sección 3.3); con esa brecha, el modelo de expansión escoge qué construir y cuándo (sección 3.4), y su solución da el costo y la energía que no alcanza a cubrirse a tiempo. La sección 4.8 responde siguiendo esos mismos pasos.

**Tres hidrologías.** El balance principal usa la ENFICC, es decir, la energía que el parque garantiza con la hidrología crítica de la serie histórica: es la regla del Cargo por Confiabilidad y el criterio con el que dimensionamos la expansión. Como contraste, en la sección 3.3 calculamos dos años con la generación hidráulica observada desde 2010: uno seco, como los de 2015-2016, 2019-2020 y 2023-2024, que empezaron con El Niño en curso, y uno normal. Así se distingue cuánto de la brecha responde a un criterio de prudencia y cuánto a un riesgo real en un año seco típico. Los tres escenarios (base, estrés y optimista) varían la demanda y los retrasos de los proyectos.

### Antecedentes

Nos apoyamos en la literatura sobre el diseño del mercado de energía firme, sobre su desempeño en Colombia, sobre la planeación oficial y sobre cómo emparejar registros que no comparten un código.

- **El diseño del mercado de energía firme.** Cramton y Stoft (2007) describen el mecanismo que Colombia adoptó en 2006: obligaciones de energía firme que se venden en una subasta de reloj descendente y funcionan como una opción de compra con precio de ejercicio igual al precio de escasez. Cramton, Ockenfels y Stoft (2013) explican por qué un mercado de energía con precios topados necesita un mecanismo así para pagar la capacidad que solo se usa en la escasez.
- **Cómo le ha ido.** Harbord y Pagnozzi (2014) revisan las subastas de Colombia y de Nueva Inglaterra como lecciones para el diseño británico. Juvinao Navarro (2021) concluye que la inestabilidad regulatoria del cargo redistribuyó rentas hacia los grupos de interés mejor organizados y recomienda un regulador más independiente.
- **La planeación oficial.** La UPME publica el Plan Indicativo de Expansión de la Generación (PIEG 2025-2039), que optimiza la expansión de largo plazo y trae su propio balance de energía firme. Este tutorial usa sus costos, sus factores firmes y su tasa de descuento. Los modelos de expansión de mínimo costo con variables enteras son la herramienta estándar de esa planeación (Koltsaklis y Dagoumas, 2018).
- **Emparejar registros sin un código común** es un problema clásico de la estadística. Fellegi y Sunter (1969) lo formalizan como una decisión entre aceptar el par, rechazarlo o revisarlo a mano: la misma lógica de los dos umbrales de la sección 3.1.

**Qué agrega este trabajo.** Un balance de energía firme por período, reproducible desde datos públicos, con tres piezas que no aparecen juntas en los documentos oficiales: los retrasos medidos en las subastas reales, gracias a la homologación (sección 3.1); la oferta física del parque, y no solo las obligaciones asignadas (sección 3.3), y una expansión que respeta las fechas en que cada tecnología puede llegar (sección 3.4).

### ¿Por qué Python?

El problema reúne cuatro tareas que normalmente viven en herramientas distintas: descargar datos de varias API, emparejar nombres de plantas sin un código común, optimizar con variables enteras y mostrar los resultados.

Cada una tiene su herramienta habitual (GAMS para optimizar, Stata para limpiar, Excel para graficar); lo que gana Python es juntarlas en un mismo flujo reproducible, sin pasar archivos a mano de una a otra:
- **`requests`** baja del repositorio público los archivos de datos que falten (en el pipeline del proyecto, también consulta la API de XM). **`pandas`** limpia las tablas mensuales y valida cada unión (`merge(..., validate=...)`), para que una llave repetida no duplique filas sin aviso.
- **`rapidfuzz`** calcula en C++, con todos los núcleos del procesador, los puntajes de similitud entre miles de nombres (`process.cdist`), y la matriz completa evita recorrer los pares uno por uno.
- **`scipy.optimize.milp`** resuelve el modelo de expansión con HiGHS, un solver de código abierto que no pide las licencias de GAMS o CPLEX; **`numpy`** arma la matriz del problema.
- **`plotly`** produce figuras interactivas, que se pueden explorar pasando el cursor.

La sección 2.1 dice cómo instalar cada paquete, quién lo desarrolla y dónde está su documentación.

**Lo que sigue.** Para responder la pregunta necesitamos tres insumos: la energía firme del parque y de los proyectos (XM), la demanda proyectada (UPME) y los costos de las tecnologías (UPME y otras fuentes). En la sección 2 los conseguimos, los revisamos y los dejamos listos.

<a id="datos" name="datos"></a>
## 2. Datos

Aquí conseguimos los tres insumos que pide la pregunta (energía firme, demanda proyectada y costos) con datos de XM, la UPME, NOAA y el DANE y las reglas de la CREG, y los dejamos limpios, revisados y unidos para las secciones 3 y 4. Las fuentes de los costos y supuestos están en la sección 3.2.

### 2.1 Instalación y configuración

Antes de tocar los datos, explicamos cómo preparar el entorno, en el navegador o en su equipo, y cargamos todos los paquetes en una sola celda.

**En el navegador.** Los botones de la portada lo abren en Colab, que pide una cuenta de Google y en el que la primera celda instala RapidFuzz, o en Binder, que no pide cuenta y arma el entorno con las versiones de la tabla; la primera vez tarda unos minutos en arrancar.

**En su equipo.** Python 3.13 o 3.14 (lo probamos con 3.14.7 en Windows 11 y con 3.13 en Linux) y JupyterLab, o VS Code con su extensión de Jupyter. Internet no hace falta si el notebook está al lado de la carpeta `datos_tutorial/`, como viene en el .zip. En Windows (PowerShell):

```powershell
python -m venv $env:USERPROFILE\.venvs\suficiencia_sin
& $env:USERPROFILE\.venvs\suficiencia_sin\Scripts\Activate.ps1
python -m pip install pandas==3.0.3 numpy==2.5.4 requests==2.34.2 rapidfuzz==3.14.6 scipy==1.18.1 plotly==7.1.0 jupyterlab
python -m jupyter lab
```

Si PowerShell no deja correr `Activate.ps1`, se habilita solo para esa ventana con `Set-ExecutionPolicy -Scope Process Bypass`. En macOS y Linux los pasos son los mismos, con `python3 -m venv` y `source ~/.venvs/suficiencia_sin/bin/activate`, pero no los probamos.

| Paquete | Versión | Para qué lo usamos | Quién lo desarrolla | Documentación oficial |
|---|---|---|---|---|
| pandas | 3.0.3 | Tablas: lectura, limpieza, uniones y agregaciones | pandas Development Team (proyecto comunitario) | [pandas.pydata.org/docs](https://pandas.pydata.org/docs/) |
| NumPy | 2.5.4 | Vectores y matrices del modelo de expansión y de las simulaciones de la sección 5 | NumPy Developers | [numpy.org/doc/stable](https://numpy.org/doc/stable/) |
| requests | 2.34.2 | La descarga de los archivos de datos que falten | Creado por Kenneth Reitz; hoy en la Python Software Foundation | [requests.readthedocs.io](https://requests.readthedocs.io/) |
| RapidFuzz | 3.14.6 | Similitud de nombres en la homologación | Max Bachmann | [rapidfuzz.github.io/RapidFuzz](https://rapidfuzz.github.io/RapidFuzz/) |
| SciPy | 1.18.1 | `milp`, que llama al solver HiGHS | SciPy Developers; HiGHS nació en la Universidad de Edimburgo | [scipy.optimize.milp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html) y [HiGHS](https://highs.dev/) |
| Plotly | 7.1.0 | Figuras interactivas | Plotly (empresa) | [plotly.com/python](https://plotly.com/python/) |

La primera celda de código solo actúa en Colab, donde instala RapidFuzz con `pip`. La segunda importa todo lo que usa el notebook, primero la biblioteca estándar de Python y después los paquetes de la tabla.

In [1]:
# En Google Colab: instalar RapidFuzz, el único paquete de la tabla que Colab no trae
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz==3.14.6"], check=True)
    print("Colab: RapidFuzz instalado")

In [2]:
# Biblioteca estándar
import hashlib
import json
import os
import re
import time
import unicodedata
from importlib.metadata import PackageNotFoundError, version
from io import StringIO
from pathlib import Path

# Paquetes de la tabla de la sección 2.1
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from IPython.display import display
from plotly.subplots import make_subplots
from rapidfuzz import fuzz, process
from scipy.optimize import Bounds, LinearConstraint, milp

SEPARADORES = str.maketrans(",.", ".,")   # 1,234.5 -> 1.234,5: los separadores del español


def es(x, decimales=0):
    """Número con los separadores del español: es(1234.5, 1) -> "1.234,5"."""
    return f"{x:,.{decimales}f}".translate(SEPARADORES)


pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}".rstrip("0").rstrip(".").translate(SEPARADORES))
SEMILLA = 2026   # semilla de la partición de etiquetas (sección 3.1) y de las simulaciones (sección 5): siempre da lo mismo

Por último, comparamos las versiones instaladas con las que usamos al probar el tutorial: una diferencia no siempre rompe algo, pero es lo primero que hay que revisar si aparece un error.

In [3]:
PROBADAS = {"pandas": "3.0.3", "numpy": "2.5.4", "requests": "2.34.2", "rapidfuzz": "3.14.6", "scipy": "1.18.1",
            "plotly": "7.1.0"}


def revisar_paquetes(probadas):
    """Compara la versión instalada de cada paquete con la que probamos.

    probadas: diccionario {paquete: versión probada}, con el nombre del paquete en PyPI.
    Devuelve una fila por paquete con la versión instalada ("no instalado" si falta) y si coincide.
    Una versión distinta suele funcionar; si algo falla, es lo primero que hay que revisar.
    """
    filas = []
    for paquete, probada in probadas.items():
        try:
            instalada = version(paquete)
        except PackageNotFoundError:  # el paquete no está: se reporta en lugar de detener el notebook
            instalada = "no instalado"
        filas.append({"paquete": paquete, "probada": probada, "instalada": instalada,
                      "coincide": instalada == probada})
    return pd.DataFrame(filas).set_index("paquete")


revisar_paquetes(PROBADAS)

,probada,instalada,coincide
paquete,,,
pandas,3.0.3,3.0.3,True
numpy,2.5.4,2.5.4,True
requests,2.34.2,2.34.2,True
rapidfuzz,3.14.6,3.14.6,True
scipy,1.18.1,1.18.1,True
plotly,7.1.0,7.1.0,True


### 2.2 Las fuentes y el paquete de datos

De dónde sale cada dato y cómo llega al notebook: todo viene en un paquete de datos verificado, que trae el .zip y que también está publicado en GitHub.

| Fuente | Qué se toma | Archivo del paquete |
|---|---|---|
| [XM, API de datos](https://github.com/EquipoAnaliticaXM/API_XM) y SIMEM | Demanda, aportes hídricos, embalse y precio de bolsa (mensual, 2000-2026); ENFICC y capacidad por planta; OEF de los proyectos nuevos; generación y ENFICC de las hidráulicas por período desde 2010; factores de planta y entradas solares | `xm_*.csv`, `proyectos_nuevos_oef.csv`, `entradas_sin_subasta.csv` |
| [UPME, proyección de demanda 2026-2040](https://www.upme.gov.co/simec/planeacion-energetica/proyeccion-de-demanda-v2/) (rev. julio de 2026) | Demanda mensual por escenario y por definición | `upme_demanda_mensual.csv` |
| [UPME, Catálogo Tecnológico Colombiano](https://api.upme.gov.co/catalogo_tecnologias/api/techs_year/) | Costos del gas y de la eólica | `upme_catalogo_tecnologias.json`: el JSON de su API, tal cual |
| [UPME, PIEG 2025-2039](https://docs.upme.gov.co/SIMEC/Energia%20Electrica/PIEG/2025-2039/PIEG_2025_2039_Version_final.pdf) | CAPEX solar, factores firmes, cupos, vidas útiles, tasa de descuento y retiros | `fuentes_supuestos.csv` (sección 3.2) |
| CNE (Chile), EPE (Brasil), NREL (EE. UU.) y Asocaña | Costos del diésel y del bagazo, que no trae el Catálogo; escenarios bajo y alto del CAPEX; cupo del bagazo | `fuentes_supuestos.csv` |
| UPME (costo de racionamiento y precios de los energéticos), FRED y Superintendencia Financiera | Penalidad, combustibles y moneda | `fuentes_supuestos.csv`, `metadatos.json` |
| [NOAA, índice ONI](https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt) | Fases de El Niño y La Niña | `noaa_oni.ascii.txt` (sección 2.3) |
| [DANE](https://www.dane.gov.co/), IPC | Precio de bolsa en pesos constantes | `dane_ipc_mensual.csv` |
| XM, SIMEM y registro de proyectos de la UPME | Las tres bases que se homologan y los retrasos de las subastas; los cupos del diésel y de la eólica (este, con la prensa) | `homologacion_*.csv`, `retrasos_subastas.csv`, `fuentes_supuestos.csv` |
| XM (boletín y subasta), UPME (PIEG) y CREG | Cifras externas con que se contrastan los resultados (secciones 2.5, 4.2 y 4.8) | `referencias_suficiencia.csv`: cada cifra con su documento, página y URL |
| [CREG, Resolución 071 de 2006](https://gestornormativo.creg.gov.co/gestor/entorno/docs/resolucion_creg_0071_2006.htm) | Las reglas de la ENFICC, las OEF y las subastas | Contexto |

**Por qué un paquete de datos.** Bajar la OEF y la ENFICC por planta, día a día desde 2006, y la generación por planta, hora a hora desde 2010, toma unas dos horas con la API de XM. Por eso el notebook lee un paquete de 0,8 MB que armó el pipeline completo del proyecto con las mismas fuentes: son las tablas limpias, algunos insumos intermedios y la fuente de cada supuesto, no los resultados del análisis. Aparte trae los resultados del pipeline, solo para que la sección 3 compruebe que llega a lo mismo.

**Cómo se verifica.** El notebook guarda la huella SHA-256 del manifiesto (`manifiesto.csv`), y el manifiesto, la de cada archivo. Si un archivo cambia en un solo byte, su huella cambia y el notebook se detiene en lugar de seguir con datos distintos. Si la carpeta `datos_tutorial/` ya existe al lado del notebook, solo se baja lo que falta.

In [4]:
# Dónde están los datos: la carpeta local si ya existe; si no, el repositorio público
URL_DATOS = os.environ.get("URL_DATOS_SUFICIENCIA",   # otra copia del paquete, solo para pruebas
                           "https://raw.githubusercontent.com/cterrassa/suficiencia-sin/main/datos_tutorial/")
CARPETA = Path("datos_tutorial")
HUELLA_MANIFIESTO = "de9d282b685b9d472b65cebe0ef3c8b8f25d3385a76dd45066f794034476f9a8"   # SHA-256 de manifiesto.csv


def huella(ruta):
    """Huella SHA-256 de un archivo, en hexadecimal."""
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()


def descargar(nombre, intentos=3):
    """Baja un archivo del repositorio de datos a CARPETA; reintenta si la conexión falla."""
    for k in range(1, intentos + 1):
        try:
            respuesta = requests.get(URL_DATOS + nombre, timeout=60)
            respuesta.raise_for_status()
            (CARPETA / nombre).write_bytes(respuesta.content)
            return
        except requests.RequestException as error:
            if k == intentos:
                raise RuntimeError(f"No se pudo bajar {nombre}: {error}") from error
            time.sleep(2 * k)

In [5]:
def obtener_datos():
    """Deja en CARPETA los archivos del manifiesto, cada uno verificado con su huella SHA-256.

    Baja solo lo que falta o no coincide. Si el manifiesto no tiene la huella esperada, se detiene:
    los datos no serían los que usa este tutorial. Devuelve el manifiesto.
    """
    CARPETA.mkdir(exist_ok=True)
    manifiesto = CARPETA / "manifiesto.csv"
    if not manifiesto.exists() or huella(manifiesto) != HUELLA_MANIFIESTO:
        descargar("manifiesto.csv")
    if huella(manifiesto) != HUELLA_MANIFIESTO:
        raise ValueError("manifiesto.csv no coincide con la huella esperada")
    m = pd.read_csv(manifiesto)
    for archivo, sha in zip(m["archivo"], m["sha256"]):
        ruta = CARPETA / archivo
        if not ruta.exists() or huella(ruta) != sha:
            descargar(archivo)
        if huella(ruta) != sha:
            raise ValueError(f"{archivo} no coincide con su huella SHA-256")
    return m


manifiesto = obtener_datos()
print(f"{len(manifiesto)} archivos verificados, {es(manifiesto['bytes'].sum() / 1e6, 1)} MB")
manifiesto[["archivo", "bytes"]]

20 archivos verificados, 0,8 MB


,archivo,bytes
0,candidatos_costos.csv,4017
1,dane_ipc_mensual.csv,8064
2,entradas_sin_subasta.csv,117
3,fuentes_supuestos.csv,37072
4,homologacion_bases.csv,168616
5,homologacion_etiquetas.csv,4749
6,metadatos.json,407
7,noaa_oni.ascii.txt,23972
8,pipeline_brechas_escenarios.csv,3731
9,pipeline_expansion_resumen.csv,17193


**Lectura.** Los 20 archivos pesan 0,8 MB y todos coinciden con su huella. Los más grandes son la proyección mensual de demanda, el Catálogo Tecnológico de la UPME (el JSON de su API, que la sección 3.2 usa para rastrear los costos) y las bases de la homologación; los demás son tablas de unas decenas o cientos de filas.

### 2.3 El índice de El Niño

El índice ONI de NOAA dice en qué meses hubo El Niño o La Niña. Con él se describe el precio de bolsa en la sección 2.5 y se ubica el episodio de El Niño en curso.

**El índice ONI.** Es la anomalía de la temperatura del Pacífico ecuatorial en trimestres móviles (`DJF` es diciembre-enero-febrero). Cada trimestre se asigna a su mes central. Un mes es de El Niño cuando el ONI llega a +0,5 °C o más durante al menos cinco trimestres seguidos, y de La Niña cuando baja a −0,5 °C o menos; es el criterio de NOAA. El notebook lee la copia del paquete, que llega hasta el 4 de noviembre de 2026, para que los resultados no cambien cuando NOAA publique un trimestre nuevo. La fuente original está enlazada en la tabla de la sección 2.2.

In [6]:
MES_CENTRAL = {"DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
               "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12}


def leer_oni(texto):
    """Convierte el archivo de NOAA (columnas SEAS, YR, TOTAL y ANOM) en una serie mensual de ONI."""
    d = pd.read_csv(StringIO(texto), sep=r"\s+")
    d["mes"] = pd.to_datetime(dict(year=d["YR"], month=d["SEAS"].map(MES_CENTRAL), day=1))
    return d.rename(columns={"SEAS": "trimestre", "ANOM": "oni"})[["mes", "trimestre", "oni"]]


def rachas(serie, umbral, minimo=5):
    """Marca los meses que pertenecen a una racha de al menos `minimo` valores seguidos que cumplen
    la condición: serie >= umbral si el umbral es positivo, serie <= umbral si es negativo."""
    cumple = serie >= umbral if umbral > 0 else serie <= umbral
    racha = (cumple != cumple.shift()).cumsum()            # número de cada racha
    return cumple & (cumple.groupby(racha).transform("size") >= minimo)


oni = leer_oni((CARPETA / "noaa_oni.ascii.txt").read_text(encoding="utf-8"))
oni["fase"] = np.select([rachas(oni["oni"], 0.5), rachas(oni["oni"], -0.5)], ["El Niño", "La Niña"], "Neutro")
print(f"{len(oni)} meses de ONI, de {oni['mes'].min():%Y-%m} a {oni['mes'].max():%Y-%m}: "
      f"{oni['oni'].isna().sum()} faltantes y {oni['mes'].duplicated().sum()} meses repetidos")

921 meses de ONI, de 1950-01 a 2026-09: 0 faltantes y 0 meses repetidos


In [7]:
# Un renglón por episodio de El Niño: inicio, fin, duración y ONI máximo; los cuatro más fuertes desde 1990
episodio = (oni["fase"] != oni["fase"].shift()).cumsum()
episodios = (oni[oni["fase"] == "El Niño"].groupby(episodio)
             .agg(inicio=("mes", "min"), fin=("mes", "max"), meses=("mes", "size"), oni_maximo=("oni", "max")))
desde_1990 = episodios[episodios["inicio"] >= "1990-01-01"]
print(f"Episodios de El Niño desde 1990: {len(desde_1990)}")
desde_1990.nlargest(4, "oni_maximo").reset_index(drop=True)

Episodios de El Niño desde 1990: 12


,inicio,fin,meses,oni_maximo
0,2014-10-01,2016-05-01,20,"2,59"
1,2026-05-01,2026-09-01,5,"2,46"
2,1997-05-01,1998-04-01,12,"2,37"
3,2023-06-01,2024-04-01,11,"1,99"


**Lectura.** El ONI no tiene faltantes ni meses repetidos. Desde 1990 hubo doce episodios de El Niño. Los más fuertes fueron el de 2015-2016, que según el ONI empezó a finales de 2014 (máximo de 2,59, durante 20 meses), el de 1997-1998 (2,37) y el de 2023-2024 (1,99). El que empezó en mayo de 2026 ya llegó a 2,46 en agosto-octubre, cerca del máximo de 2015-2016, y seguía en curso a la fecha de corte. La sección 2.5 usa estas fases para describir el precio de bolsa.

### 2.4 Carga, limpieza y descripción

Con las fases de El Niño listas, siguen las tablas del paquete: se cargan, se describen y se revisan antes de unirlas. `leer()` las abre y convierte las fechas, porque un CSV guarda todo como texto. La primera es la tabla mensual del sistema que publica XM, una fila por mes desde enero de 2000.

In [8]:
def leer(nombre, fechas=(), **opciones):
    """Lee un CSV del paquete de datos y convierte a fecha las columnas de `fechas`."""
    return pd.read_csv(CARPETA / nombre, parse_dates=list(fechas), **opciones)


sistema = leer("xm_sistema_mensual.csv", ["mes"])
sistema.info()

<class 'pandas.DataFrame'>
RangeIndex: 323 entries, 0 to 322
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   mes                         323 non-null    datetime64[us]
 1   periodo_cxc                 323 non-null    str           
 2   demanda_real_gwh            322 non-null    float64       
 3   demanda_sin_gwh             323 non-null    float64       
 4   aportes_gwh                 322 non-null    float64       
 5   aportes_media_hist_gwh      322 non-null    float64       
 6   volumen_util_fin_mes_gwh    322 non-null    float64       
 7   capacidad_util_fin_mes_gwh  322 non-null    float64       
 8   importaciones_gwh           304 non-null    float64       
 9   exportaciones_gwh           292 non-null    float64       
 10  enficc_sistema_gwh          240 non-null    float64       
 11  precio_bolsa_cop_kwh        322 non-null    float64       
 12  preci

**Lectura.** 323 meses, de enero de 2000 a noviembre de 2026, y 20 columnas. Las fechas quedaron como fechas (`datetime64`), los indicadores de mes completo como booleanos y lo demás como números. Algunas columnas tienen menos datos porque empiezan después: la ENFICC del sistema, el precio de escasez y el umbral de escasez, en diciembre de 2006, con el Cargo por Confiabilidad; los precios de escasez superior e inferior, en 2025 (Resolución CREG 101 066 de 2024).

In [9]:
# Faltantes por columna (solo las que tienen alguno) y meses incompletos
faltantes = sistema.isna().sum()
print(faltantes[faltantes > 0].to_string())
print("\nMeses incompletos:", ", ".join(sistema.loc[~sistema["mes_completo"], "mes"].dt.strftime("%Y-%m")))

# Limpieza: solo meses completos, y nombres cortos para describir
completo = sistema[sistema["mes_completo"]].rename(columns={
    "demanda_sin_gwh": "demanda_GWh", "aportes_gwh": "aportes_GWh",
    "volumen_util_fin_mes_gwh": "embalse_GWh", "precio_bolsa_cop_kwh": "precio_COP_kWh"})
completo[["demanda_GWh", "aportes_GWh", "embalse_GWh", "precio_COP_kWh"]].describe().round(1)

demanda_real_gwh                1
aportes_gwh                     1
aportes_media_hist_gwh          1
volumen_util_fin_mes_gwh        1
capacidad_util_fin_mes_gwh      1
importaciones_gwh              19
exportaciones_gwh              31
enficc_sistema_gwh             83
precio_bolsa_cop_kwh            1
precio_escasez_cop_kwh         84
umbral_escasez_cop_kwh         84
precio_escasez_sup_cop_kwh    302
precio_escasez_inf_cop_kwh    302
dias_enficc                    83

Meses incompletos: 2026-11


,demanda_GWh,aportes_GWh,embalse_GWh,precio_COP_kWh
count,322,322,322,322
mean,"5.160,1",4.672,"10.754,6","189,7"
std,"1.104,9","1.988,6","2.332,1","203,7"
min,"3.278,3","1.122,5","2.728,5","35,1"
25%,"4.274,7",3.246,"9.318,7",72
50%,"5.077,1","4.580,5","11.004,2","118,6"
75%,"5.985,8","5.817,2","12.494,1","199,7"
max,"7.701,5","12.536,3",15.533,"1.529,1"


**Lectura.** Solo noviembre de 2026 está incompleto: la tabla se cortó el 4 de noviembre, así que se descarta. Los demás faltantes vienen de donde empieza cada serie; las importaciones y exportaciones no se usan en el análisis. En los 322 meses completos, la demanda promedió 5.160 GWh al mes, entre 3.278 y 7.702. Los aportes hídricos varían mucho más: su desviación estándar (1.989 GWh) es el 43% de su media (4.672). Y el precio de bolsa en pesos corrientes tiene una media de 190 COP/kWh pero una mediana de 119: unos pocos meses de escasez, con picos de hasta 1.529, jalan la media hacia arriba.

Después, la proyección de demanda de la UPME: una fila por mes, definición de demanda y escenario.

In [10]:
demanda_upme = leer("upme_demanda_mensual.csv", ["mes"])
print(f"{es(len(demanda_upme))} filas, de {demanda_upme['mes'].min():%Y-%m} a {demanda_upme['mes'].max():%Y-%m}")
print("Definiciones:", ", ".join(demanda_upme["definicion"].unique()))
print("Escenarios:", ", ".join(demanda_upme["escenario"].unique()))
(demanda_upme.query("mes.dt.year == 2030")
             .pivot_table(index="definicion", columns="escenario", values="energia_gwh", aggfunc="sum")
             .round(0))

3.540 filas, de 2026-04 a 2040-12
Definiciones: SIN, SIN+GCE+ME, SIN+GCE+ME+GD_PNUMA, SIN+GCE+ME+GD_UPME
Escenarios: ic68_inf, ic68_sup, ic95_inf, ic95_sup, medio


escenario,ic68_inf,ic68_sup,ic95_inf,ic95_sup,medio
definicion,,,,,
SIN,90.274,101.050,85.638,105.733,95.638
SIN+GCE+ME,100.312,111.087,95.675,115.771,105.675
SIN+GCE+ME+GD_PNUMA,86.522,97.297,81.885,101.981,91.885
SIN+GCE+ME+GD_UPME,95.583,106.358,90.946,111.042,100.946


**Lectura.** La proyección va de abril de 2026 a diciembre de 2040, con cuatro definiciones de demanda y cinco escenarios: el medio y los bordes de las bandas de 68% y 95%. La definición importa: en 2030 el escenario medio va de 91.885 GWh (restando la generación distribuida con el supuesto del PNUMA) a 105.675 (sin restarla). El diagnóstico usa la demanda total, SIN + grandes consumidores especiales + movilidad eléctrica − generación distribuida (`SIN+GCE+ME+GD_UPME`): 100.946 GWh en 2030. La adoptamos porque, en el pipeline, fue la definición que mejor reprodujo la demanda observada de abril a octubre de 2026.

**Revisión de todas las tablas del paquete.** Antes de unir las fuentes, cada tabla pasa por la misma revisión: tamaño, tipos, celdas faltantes, llaves y filas repetidas, y valores extremos. La llave son las columnas que deben identificar cada fila una sola vez. Los valores extremos se cuentan con la regla de Tukey en las columnas numéricas continuas, fuera de tres rangos intercuartílicos: con 1,5, la regla usual, saldría marcada buena parte de un parque con muchas plantas pequeñas y pocas grandes. Las dos tablas de resultados del pipeline solo sirven para comprobar; el ONI se revisó al leerlo (sección 2.3), y el Catálogo es el JSON de la API de la UPME tal cual: la sección 3.2 lo compara con los costos.

In [11]:
def revisar_tabla(nombre, llave, fechas=(), **opciones):
    """Revisa una tabla del paquete: tamaño, tipos, faltantes, llaves y filas repetidas, y valores extremos.

    llave: columnas que deben identificar cada fila una sola vez. Los valores extremos se cuentan en las columnas
    numéricas con más de 10 valores distintos, con la regla de Tukey: fuera de [Q1 − 3·RIC, Q3 + 3·RIC].
    """
    t = leer(nombre, fechas, **opciones)
    numericas = t.select_dtypes("number").loc[:, lambda d: d.nunique() > 10]
    q1, q3 = numericas.quantile(0.25), numericas.quantile(0.75)
    extremos = ((numericas < q1 - 3 * (q3 - q1)) | (numericas > q3 + 3 * (q3 - q1))).sum()
    return {"tabla": nombre, "filas": len(t), "columnas": t.shape[1],
            "tipos": ", ".join(f"{n} {tipo}" for tipo, n in t.dtypes.astype(str).value_counts().items()),
            "celdas faltantes": int(t.isna().sum().sum()), "llave": " + ".join(llave),
            "llaves repetidas": int(t.duplicated(llave).sum()), "filas repetidas": int(t.duplicated().sum()),
            "valores extremos": ", ".join(f"{c} ({n})" for c, n in extremos[extremos > 0].items()) or "ninguno"}

In [12]:
TABLAS = [("xm_sistema_mensual.csv", ["mes"], ["mes"], {}),
          ("upme_demanda_mensual.csv", ["mes", "definicion", "escenario"], ["mes"], {}),
          ("dane_ipc_mensual.csv", ["mes"], ["mes"], {}),
          ("xm_plantas_mes_referencia.csv", ["codigo"], [], {"dtype": {"codigo": str}}),
          ("proyectos_nuevos_oef.csv", ["planta"], [], {}),
          ("entradas_sin_subasta.csv", ["grupo"], [], {}),
          ("candidatos_costos.csv", ["tecnologia", "escenario_costos"], [], {}),
          ("homologacion_bases.csv", ["fuente", "id"], ["fecha", "fecha_alta"], {"dtype": {"id": str}}),
          ("homologacion_etiquetas.csv", ["tarea", "id_izq", "id_der"], [], {"dtype": {"id_izq": str, "id_der": str}}),
          ("retrasos_subastas.csv", ["codigo_oef", "mecanismo"], ["compromiso", "fecha_entrada"], {}),
          ("xm_hidraulica_periodos.csv", ["periodo"], [], {}),
          ("referencias_suficiencia.csv", ["fuente", "documento", "periodo", "concepto"], [], {}),
          ("fuentes_supuestos.csv", ["tecnologia", "parametro", "escenario", "valor"], [], {}),
          ("xm_factores_planta_fase.csv", ["tecnologia", "fase"], [], {}),
          ("xm_solar_entradas_anuales.csv", ["anio"], [], {})]
revision = pd.DataFrame([revisar_tabla(nombre, llave, fechas, **opciones)
                         for nombre, llave, fechas, opciones in TABLAS]).set_index("tabla")
revision

,filas,columnas,tipos,celdas faltantes,llave,llaves repetidas,filas repetidas,valores extremos
tabla,,,,,,,,
xm_sistema_mensual.csv,323,20,"15 float64, 2 bool, 1 datetime64[us], 1 str, 1...",994,mes,0,0,"capacidad_util_fin_mes_gwh (5), importaciones_..."
upme_demanda_mensual.csv,3540,5,"2 str, 2 float64, 1 datetime64[us]",0,mes + definicion + escenario,0,0,ninguno
dane_ipc_mensual.csv,321,3,"1 datetime64[us], 1 float64, 1 str",0,mes,0,0,ninguno
xm_plantas_mes_referencia.csv,631,8,"5 str, 3 float64",30,codigo,0,0,"capacidad_mw (55), enficc_gwh_dia (155), oef_g..."
proyectos_nuevos_oef.csv,41,13,"6 str, 5 float64, 1 bool, 1 int64",0,planta,0,0,"oef_2029_2030 (1), oef_gwh_dia (1)"
entradas_sin_subasta.csv,2,5,"2 str, 2 float64, 1 int64",0,grupo,0,0,ninguno
candidatos_costos.csv,18,23,"15 float64, 4 str, 4 int64",0,tecnologia + escenario_costos,0,0,ninguno
homologacion_bases.csv,2600,9,"5 str, 2 float64, 2 datetime64[us]",4598,fuente + id,0,0,"capacidad_mw (347), oef_gwh_dia (1)"
homologacion_etiquetas.csv,205,4,"3 str, 1 int64",0,tarea + id_izq + id_der,0,0,ninguno


**Lectura.** Ninguna tabla tiene llaves ni filas repetidas, así que no hay duplicados que limpiar; además, cada unión del notebook usa `validate`, que se detendría si apareciera uno. Los tipos quedaron bien: fechas como fechas y códigos como texto. En `fuentes_supuestos.csv` el valor queda como texto porque el primer período no es un número; la sección 3.2 lo convierte. Los faltantes y los valores extremos sí piden decisiones:

| Tabla | Hallazgo | Decisión | Por qué |
|---|---|---|---|
| Sistema mensual de XM | Noviembre de 2026 trae solo 3 días, y por eso su ENFICC sale como valor extremo | Se descarta en los análisis mensuales | La tabla se cortó el 4 de noviembre: un mes incompleto no es comparable |
| Sistema mensual de XM | A diciembre de 2014 le falta un día de ENFICC (trae 30 de 31): es el único mes cerrado con la ENFICC incompleta | Se conserva | La ENFICC se usa como promedio diario de los días con dato, así que un día sin publicar no la sesga; descartar el mes dejaba sin el período 2014-2015 a la hidráulica por período (sección 3.3) |
| Sistema mensual de XM | 994 celdas faltantes, casi todas de series que empiezan después de 2000: la ENFICC y el precio de escasez desde diciembre de 2006, y los precios de escasez superior e inferior desde 2025 | No se imputan | Son meses en que la variable no existía; imputarlos sería inventar datos |
| Sistema mensual de XM | 18 meses con precio de bolsa extremo: El Niño de 2015-2016 y de 2023-2024, septiembre a diciembre de 2024 y julio a octubre de 2026 | Se conservan | Son episodios reales de escasez, justo el fenómeno que motiva el trabajo |
| Sistema mensual de XM | Importaciones extremas en 38 meses y capacidad útil de los embalses baja en 5 meses de 2000-2002 | Se conservan | No entran en el análisis. Las importaciones son pequeñas la mayoría de los meses (mediana de 1,8 GWh), así que cualquier mes con importaciones importantes sale como extremo |
| Plantas del mes de referencia | 30 celdas faltantes: 13 plantas registradas sin pruebas y una térmica sin capacidad, y 4 unidades fuera del listado de XM. Capacidad y ENFICC muy asimétricas | Las que no operan no suman (sección 3.3); los extremos se conservan | Sin pruebas no hay generación ni firmeza; el parque real tiene muchas plantas pequeñas y pocas grandes |
| Bases de la homologación | 4.598 celdas faltantes en capacidad, OEF, agente y fechas | Se conservan | Cada base trae atributos distintos; la revisión de atributos de la sección 3.1 da «no aplica» cuando falta el dato |
| Retrasos de las subastas | 91 faltantes en la fecha de entrada y el retraso | Se conservan | Son los proyectos que no han entrado: el faltante es la información |
| Hidráulica por período | 2012-2013 promedia 26,8 plantas con ENFICC, frente a 53 a 79 en los demás períodos; la regla de extremos no lo marca por poco, pero el salto es real | Se conserva | Generación y ENFICC se comparan para las mismas plantas, y su relación (1,22) es igual a la del período siguiente |
| Referencias externas | 10 celdas faltantes en la página y un valor extremo (la ENFICC del PIEG, 83.599 GWh) | Se conservan | La prensa y los comunicados de XM no tienen página; la columna `valor` mezcla GWh, fracciones y USD/MWh, así que un extremo no significa nada |
| Fuentes de los supuestos | 45 celdas faltantes en la referencia y la URL | Se conservan | Son los cálculos propios, las decisiones de modelación y los supuestos del equipo: no tienen un documento externo, y la columna `como` dice de dónde salen |
| Factores de planta por fase del ONI | 2 valores extremos en los meses-planta | Se conservan | Las pequeñas hidráulicas son muchas más plantas que las demás tecnologías; el factor de planta no tiene extremos |
| Entradas solares por año | 8 celdas vacías en la nota | Se conservan | La nota solo marca el año incompleto: 2026, hasta el corte |

### 2.5 Unir las fuentes

Tres uniones preparan el análisis. Cada una usa `merge` con `validate`, que falla si la llave no es única donde debe serlo, y comprueba cuántas filas se pierden.

**a) El precio de bolsa en pesos constantes y la hidrología.** El precio de XM está en pesos corrientes: para comparar 2000 con 2026 hay que llevarlo a pesos de un mismo mes (septiembre de 2026) con el IPC del DANE. Luego clasificamos la hidrología de cada mes: seca si sus aportes quedan por debajo del 80% de la media histórica, normal entre 80% y 110% y húmeda por encima. Por último, le asignamos la fase del ONI de la sección 2.3. Una tabla resume el precio por hidrología y por fase, y una figura muestra su evolución y su relación con dos indicadores de suficiencia.

In [13]:
ipc = leer("dane_ipc_mensual.csv", ["mes"])


def precio_real(sistema, ipc, base="2026-09-01"):
    """Precio de bolsa en pesos constantes de `base` y condición hidrológica de cada mes completo."""
    s = sistema[sistema["mes_completo"] & sistema["precio_bolsa_cop_kwh"].notna()]
    d = s.merge(ipc[["mes", "ipc"]], on="mes", how="inner", validate="one_to_one")
    print(f"Meses con precio: {len(s)}; con IPC: {len(d)} (el DANE publica el IPC con un mes de rezago)")
    deflactor = ipc.set_index("mes").loc[base, "ipc"] / d["ipc"]
    aportes_pct = 100 * d["aportes_gwh"] / d["aportes_media_hist_gwh"]
    return pd.DataFrame({
        "mes": d["mes"], "precio_real": d["precio_bolsa_cop_kwh"] * deflactor,
        "umbral_escasez_real": d["umbral_escasez_cop_kwh"] * deflactor,
        "condicion": pd.cut(aportes_pct, [-np.inf, 80, 110, np.inf], labels=["Seca", "Normal", "Húmeda"]).astype(str)})


precios = precio_real(sistema, ipc).merge(oni[["mes", "fase"]], on="mes", how="inner", validate="one_to_one")
precios["sobre_umbral"] = precios["precio_real"] > precios["umbral_escasez_real"]   # activa las OEF
pd.concat([precios.groupby(col).agg(meses=("precio_real", "size"), precio_medio=("precio_real", "mean"),
                                    meses_sobre_umbral=("sobre_umbral", "sum"))
           for col in ["condicion", "fase"]], keys=["Hidrología", "Fase del ONI"]).round(0)

Meses con precio: 322; con IPC: 321 (el DANE publica el IPC con un mes de rezago)


meses  precio_medio  meses_sobre_umbral
Hidrología   Húmeda      92           207                   0
             Normal     146           264                   2
             Seca        83           528                   8
Fase del ONI El Niño     81           485                   8
             La Niña     81           226                   0
             Neutro     159           276                   2

**Lectura.** De los 322 meses con precio, 321 tienen IPC: el de octubre de 2026 todavía no estaba publicado, y la unión lo deja por fuera sin duplicar ni perder nada más. En pesos de septiembre de 2026, el precio medio de los meses secos (528 COP/kWh) duplica el de los normales (264) y es 2,6 veces el de los húmedos (207). Con el ONI pasa algo parecido: 485 en El Niño frente a 226 en La Niña. El precio superó el umbral de escasez, el que activa las OEF, en 10 meses: 8 secos y 2 normales, y 8 de ellos en El Niño. Es asociación, no causalidad: la hidrología mueve a la vez los aportes, el embalse y el precio.

Los profesores recomendaron analizar el precio con gráficos de su evolución y de su relación con indicadores de suficiencia. La figura lo hace con dos: el margen de energía firme, que es la ENFICC del sistema sobre la demanda del mes, y el embalse al cierre del mes.

In [14]:
# Dos indicadores de suficiencia de cada mes: el margen de energía firme (ENFICC sobre demanda, por día) y el embalse
indicadores = sistema[sistema["enficc_sistema_gwh"].notna()]
indicadores = pd.DataFrame({
    "mes": indicadores["mes"],
    "margen": (indicadores["enficc_sistema_gwh"] / indicadores["dias_enficc"])
              / (indicadores["demanda_sin_gwh"] / indicadores["dias_con_datos"]),
    "embalse_pct": 100 * indicadores["volumen_util_fin_mes_gwh"] / indicadores["capacidad_util_fin_mes_gwh"]})
relacion = precios.merge(indicadores, on="mes", how="inner", validate="one_to_one")
rho = relacion[["precio_real", "margen", "embalse_pct"]].corr(method="spearman")["precio_real"]
print(f"Meses con precio y ENFICC: {len(relacion)}, de {relacion['mes'].min():%Y-%m} a {relacion['mes'].max():%Y-%m}")

fig = make_subplots(rows=2, cols=2, specs=[[{"colspan": 2}, None], [{}, {}]], row_heights=[0.55, 0.45],
                    vertical_spacing=0.17, horizontal_spacing=0.09,
                    subplot_titles=("Evolución mensual; meses de El Niño sombreados",
                                    f"Frente al margen firme (Spearman: {es(rho['margen'], 2)})".replace("-", "−"),
                                    f"Frente al embalse (Spearman: {es(rho['embalse_pct'], 2)})".replace("-", "−")))
fig.add_trace(go.Scatter(x=precios["mes"], y=precios["precio_real"], name="Precio de bolsa",
                         line=dict(color="#1f1f1f", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=precios["mes"], y=precios["umbral_escasez_real"], name="Umbral de escasez (activa las OEF)",
                         line=dict(color="#6250d6", width=1.5, dash="dash")), row=1, col=1)
tramo = (precios["fase"] != precios["fase"].shift()).cumsum()   # un número por cada racha de meses con la misma fase
for _, nino in precios[precios["fase"] == "El Niño"].groupby(tramo):
    fig.add_vrect(x0=nino["mes"].min(), x1=nino["mes"].max() + pd.offsets.MonthBegin(1), fillcolor="#eb6834", opacity=0.18,
                  line_width=0, layer="below", row=1, col=1)
for condicion, tono in {"Seca": "#eb6834", "Normal": "#8f8f8f", "Húmeda": "#2a78d6"}.items():
    puntos = relacion[relacion["condicion"] == condicion]
    for columna, variable in [(1, "margen"), (2, "embalse_pct")]:
        fig.add_trace(go.Scatter(x=puntos[variable], y=puntos["precio_real"], mode="markers", name=f"Hidrología {condicion.lower()}",
                                 legendgroup=condicion, showlegend=columna == 1, text=puntos["mes"].dt.strftime("%Y-%m"),
                                 marker=dict(color=tono, size=7, opacity=0.75)), row=2, col=columna)
fig.update_yaxes(type="log", tickvals=[100, 200, 500, 1000, 2000], tickformat=",.0f", title_text="COP de sep-2026 por kWh (log)")
fig.update_yaxes(title_text=None, row=2, col=2)
fig.update_xaxes(title_text="ENFICC del sistema / demanda del mes", row=2, col=1)
fig.update_xaxes(title_text="Embalse al cierre del mes (%)", row=2, col=2)
fig.update_layout(title="Precio de bolsa en pesos constantes: evolución y relación con la suficiencia",
                  height=700, separators=",.", legend=dict(orientation="h", yanchor="top", y=-0.12, x=0, title=None))
fig.show()

Meses con precio y ENFICC: 238, de 2006-12 a 2026-09


**Lectura.** Los picos del precio coinciden con El Niño: el máximo de la serie es octubre de 2015 (2.048 COP/kWh), en pleno episodio de 2015-2016, y vuelven a verse en 2023-2024 y en 2026. La excepción es octubre de 2024, el segundo mes más caro (1.712), que llegó con el embalse en 55% cuando El Niño ya había terminado. En los 238 meses con ENFICC, el precio casi no se asocia con el margen de energía firme (correlación de Spearman de −0,09) y sí con el embalse (−0,49): en el corto plazo responde al agua disponible, no a la energía firme, que cambia despacio. El margen solo bajó de 1 en cuatro meses, todos de 2026: en ellos la ENFICC del sistema fue menor que la demanda. Como en la tabla, es una asociación y no un efecto causal; por eso el precio no entra en la pregunta ni en el modelo, y queda como contexto.

**b) De meses calendario a períodos del cargo.** La UPME proyecta por mes calendario; el Cargo por Confiabilidad funciona por períodos de diciembre a noviembre. Restar una cosa de la otra sin alinearlas mezcla un año con otro. `periodo_cxc()` le asigna a cada mes su período (diciembre de 2026 abre el período 2026-2027), y la suma por período deja la demanda en la misma unidad que la energía firme. Los dos escenarios de demanda son el medio y el borde superior de la banda de 68% de la UPME.

In [15]:
PERIODOS = ["2026-2027", "2027-2028", "2028-2029", "2029-2030"]
DEFINICION = "SIN+GCE+ME+GD_UPME"   # demanda total: SIN + grandes consumidores + movilidad eléctrica − GD
ESCENARIOS_DEMANDA = {"media": "medio", "alta": "ic68_sup"}


def periodo_cxc(meses):
    """Período del Cargo por Confiabilidad (diciembre a noviembre) de cada mes: dic-2026 → "2026-2027"."""
    inicio = meses.dt.year - (meses.dt.month < 12)
    return inicio.astype(str) + "-" + (inicio + 1).astype(str)


d = demanda_upme[(demanda_upme["definicion"] == DEFINICION)
                 & demanda_upme["escenario"].isin(ESCENARIOS_DEMANDA.values())].copy()
d["periodo"], d["dias"] = periodo_cxc(d["mes"]), d["mes"].dt.days_in_month
demanda = (d[d["periodo"].isin(PERIODOS)]
           .groupby(["escenario", "periodo"], as_index=False)
           .agg(demanda_gwh=("energia_gwh", "sum"), dias=("dias", "sum"), meses=("mes", "size")))
demanda["escenario_demanda"] = demanda["escenario"].map({v: k for k, v in ESCENARIOS_DEMANDA.items()})
calendario = d[d["mes"].dt.year == 2027].groupby("escenario")["energia_gwh"].sum()["medio"]
periodo_1 = demanda.set_index(["escenario", "periodo"]).loc[("medio", "2026-2027"), "demanda_gwh"]
print(f"Escenario medio: año calendario 2027, {es(calendario)} GWh; período 2026-2027, {es(periodo_1)} GWh")

# La proyección de enero de 2025, con la que la CREG convocó la última subasta: su ejemplo de demanda para 2029-2030
# es 3% mayor que el escenario medio de esa proyección (Documento CREG 901 202 de 2025, p. 32)
referencias = leer("referencias_suficiencia.csv")
creg_medio = referencias.query("concepto == 'demanda_ejemplo_creg'")["valor"].iloc[0] * 1e3 / 1.03   # TWh -> GWh
media_2030 = demanda.set_index(["escenario", "periodo"]).loc[("medio", "2029-2030"), "demanda_gwh"]
print(f"2029-2030, escenario medio: {es(creg_medio)} GWh en la proyección de enero de 2025 y {es(media_2030)} en la "
      f"de julio de 2026 ({es(100 * (media_2030 / creg_medio - 1), 1)}% más)")
demanda.pivot(index="periodo", columns="escenario_demanda", values="demanda_gwh").round(0)

Escenario medio: año calendario 2027, 93.571 GWh; período 2026-2027, 93.508 GWh
2029-2030, escenario medio: 95.922 GWh en la proyección de enero de 2025 y 100.759 en la de julio de 2026 (5,0% más)


escenario_demanda,alta,media
periodo,,
2026-2027,96.231,93.508
2027-2028,100.007,96.284
2028-2029,103.273,98.700
2029-2030,106.107,100.759


**Lectura.** El período 2026-2027 suma 93.508 GWh en el escenario medio, y el año calendario 2027, 93.571. La diferencia es pequeña porque el período cambia un diciembre por otro; lo que importa es no mezclar los dos calendarios al restar la energía firme, que se asigna por período. La demanda media pasa de 93.508 GWh en 2026-2027 a 100.759 en 2029-2030 (+7,8%), y la alta, de 96.231 a 106.107 (+10,3%). Son las demandas del balance de la sección 3.3. Frente a la proyección de enero de 2025, con la que la CREG convocó la última subasta, la demanda media de 2029-2030 es 5,0% mayor.

**c) Lo observado frente a lo proyectado.** Para saber si el escenario medio es razonable, se compara con la demanda que XM ya midió (`DemaSIN`) en los meses en que hay proyección y dato a la vez.

In [16]:
medio = demanda_upme.query("definicion == @DEFINICION and escenario == 'medio'")[["mes", "energia_gwh"]]
observado = sistema.loc[sistema["mes_completo"], ["mes", "demanda_sin_gwh"]]
control = observado.merge(medio, on="mes", how="inner", validate="one_to_one")
control["desvio_%"] = 100 * (control["demanda_sin_gwh"] / control["energia_gwh"] - 1)
MESES_CORTOS = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
control.set_index(control["mes"].map(lambda m: f"{MESES_CORTOS[m.month - 1]}-{m.year}")).drop(columns="mes").round(1)

,demanda_sin_gwh,energia_gwh,desvio_%
mes,,,
abr-2026,"7.217,2","7.169,1","0,7"
may-2026,"7.664,6",7.725,"-0,8"
jun-2026,7.298,"7.425,3","-1,7"
jul-2026,"7.701,5",7.837,"-1,7"
ago-2026,"7.696,6","7.874,8","-2,3"
sep-2026,"7.531,9","7.766,5",-3
oct-2026,"7.529,5","8.010,8",-6


**Lectura.** De abril a septiembre de 2026, la demanda observada quedó entre 0,7% por encima y 3,0% por debajo del escenario medio de la demanda total; y en octubre, 6,0% por debajo. Es decir, el escenario medio no subestima la demanda; si acaso, la exagera, porque la UPME suma grandes consumidores nuevos y movilidad eléctrica que la serie observada todavía no muestra. Por eso tratamos la demanda alta como un escenario de estrés y no como la base.

**Lo que sigue.** Los datos quedaron revisados y listos: la demanda de la UPME por período del cargo, el parque y los proyectos de XM, los costos y las bases que hay que emparejar. La sección 3 los usa para construir el método, paso a paso.

<a id="metodologia" name="metodologia"></a>
## 3. Metodología y desarrollo

Con los datos de la sección 2, cuatro pasos responden la pregunta. Cada uno explica primero la idea y después la programa:

1. **Homologación (3.1):** emparejar plantas y proyectos entre bases que no comparten código. Con ella se sabe qué proyectos con OEF ya operan, y con ella el pipeline midió cuánto se retrasaron los de las subastas.
2. **Costos (3.2):** convertir la inversión y la operación de cada tecnología en un costo anual por MWh de energía firme, con la fuente de cada parámetro.
3. **Balance de energía firme (3.3):** cuánta energía firme falta, y desde cuándo, en tres escenarios y con tres hidrologías.
4. **Expansión de mínimo costo (3.4 y 3.5):** con qué bloques se cierra esa brecha al menor costo, y qué cambia si cambian los supuestos.

Al final de 3.3, 3.4 y 3.5, `comprobar()` compara los resultados con los del pipeline completo del proyecto, que los calculó con su propio código. Si alguna cifra no coincide, el notebook se detiene.

In [17]:
def comprobar(calculado, esperado, que, tolerancia=0.5, contra="el pipeline"):
    """Compara dos tablas (o series) numéricas alineadas por su índice.

    Se detiene si falta alguna fila o si alguna diferencia absoluta pasa de `tolerancia`
    (en las unidades de la tabla); si no, informa la diferencia máxima. `contra` dice con qué se compara.
    """
    calculado, esperado = calculado.align(esperado, join="outer")
    diferencia = (calculado - esperado).abs()
    maxima = float(np.nanmax(diferencia.to_numpy()))
    if diferencia.isna().to_numpy().any() or maxima > tolerancia:
        raise AssertionError(f"{que}: no coincide con {contra} (diferencia máxima {maxima:,.3f})")
    print(f"✓ {que}: coincide con {contra} (diferencia máxima {f'{maxima:.1e}'.replace('.', ',')})")

### 3.1 Homologación de plantas y proyectos con RapidFuzz

La OEF, la ENFICC y la capacidad de las plantas que ya operan traen el código de XM y se unen al listado de forma exacta. Pero hay tres bases sin un código común:

| Base | Qué se sabe de cada registro |
|---|---|
| **OEF** (bitácora de SIMEM): proyectos `PROG#####` con obligaciones asignadas | Solo la razón social de la empresa, la OEF y las fechas |
| **UPME** (registro de proyectos de generación, septiembre de 2026) | Nombre, tecnología, MW, municipio y fecha prevista |
| **XM** (listado de recursos de generación) | Nombre, tecnología, agente, estado y fecha de entrada |

Sin emparejarlas no se puede saber si un proyecto con OEF ya entró en operación, ni cuándo. El método tiene cuatro pasos:
1. **Normalizar** los nombres, porque "PARQUE SOLAR LA LOMA" y "La Loma Solar" son la misma planta escrita distinto.
2. **Puntuar** cada par con dos medidas de RapidFuzz, solo dentro del mismo bloque de tecnología, y quedarse con los tres mejores candidatos de cada registro.
3. **Revisar atributos:** etapa, capacidad, energía firme posible y fechas. Un par con un atributo incompatible se descarta por alto que sea su puntaje.
4. **Escoger el umbral** con 205 pares etiquetados: el que da la mejor combinación de precisión y exhaustividad. Después lo comprobamos fuera de muestra: escogemos con una mitad de las etiquetas y evaluamos con la otra.

**Normalizar.** `normalizar()` pasa a mayúsculas, quita tildes, razones sociales (S.A.S., E.S.P.), conectores, palabras genéricas (PARQUE, SOLAR, PCH...) y capacidades ("200 MW"), y convierte los números romanos en arábigos.

In [18]:
FORMAS_JURIDICAS = {"SAS", "SA", "ESP", "SCA", "CIA", "LTDA", "ZOMAC", "SC", "BIC", "EU"}
CONECTORES = {"DE", "DEL", "LA", "EL", "LOS", "LAS", "Y", "A", "AL", "EN", "SU"}
GENERICAS = {"PARQUE", "PLANTA", "CENTRAL", "PROYECTO", "SOLAR", "FOTOVOLTAICO", "FOTOVOLTAICA", "PHOTOVOLTAIC",
             "FV", "PV", "EOLICO", "EOLICA", "PCH", "PEQUENA", "HIDROELECTRICA", "HIDROELECTRICO", "TERMICA",
             "TERMICO", "TERMOELECTRICA", "GENERACION", "GENERADORA", "ENERGIA", "ENERGIAS", "RENOVABLES",
             "AUTOG", "AGPE", "MW"}
DESCARTAR = FORMAS_JURIDICAS | CONECTORES | GENERICAS
UNIDADES = ["", "I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX"]
ROMANOS = {"X" * (n // 10) + UNIDADES[n % 10]: n for n in range(1, 40)}   # "XIV" → 14


def sin_tildes(texto):
    """Quita tildes y diacríticos y pasa a mayúsculas: "Pequeña Central" → "PEQUENA CENTRAL"."""
    return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().upper()

In [19]:
def normalizar(nombre):
    """Deja solo las palabras que identifican la planta: "Parque Solar Colombia XIV S.A.S." → "COLOMBIA 14"."""
    if not isinstance(nombre, str) or not nombre.strip():
        return ""
    s = re.sub(r"\bY SU CONEXION\b.*$", " ", sin_tildes(nombre))        # "... Y SU CONEXIÓN A LA S/E ..."
    s = re.sub(r"\d+(?:[.,]\d+)?\s*(?:MWP?|MWAC|KWP?|KV)\b", " ", s)    # capacidad o tensión
    s = re.sub(r"[^A-Z0-9]+", " ", s.replace(".", ""))                  # S.A.S. → SAS; signos → espacio
    fichas = []
    for f in s.split():
        if f in DESCARTAR:
            continue
        if f.startswith("TERMO") and len(f) > 7:                         # TERMOCAIMAN → CAIMAN
            f = f[5:]
        fichas.append(str(ROMANOS[f]) if f in ROMANOS else f)
    return " ".join(fichas) or re.sub(r"[^A-Z0-9]+", " ", sin_tildes(nombre)).strip()


def numeros(nombre_norm):
    """Números sueltos de un nombre normalizado: etapas ("LLANOS 8") o unidades ("RUSA 2")."""
    return frozenset(re.findall(r"\b\d+\b", nombre_norm))

In [20]:
ejemplos = ["PARQUE DE GENERACIÓN FOTOVOLTAICO LA PRIMAVERA DE 57 MW", "Parque Solar Colombia XIV S.A.S. - E.S.P.",
            "BOSQUES SOLARES DE LOS LLANOS 8 Y SU CONEXIÓN A LA SUBESTACIÓN OCOA 115 KV",
            "PEQUEÑA CENTRAL HIDROELÉCTRICA AKUA 1", "PLANTA TÉRMICA EL TESORITO II 200 MW"]
pd.DataFrame({"nombre": ejemplos, "normalizado": [normalizar(n) for n in ejemplos],
              "números": [sorted(numeros(normalizar(n))) for n in ejemplos]})

,nombre,normalizado,números
0,PARQUE DE GENERACIÓN FOTOVOLTAICO LA PRIMAVERA...,PRIMAVERA,[]
1,Parque Solar Colombia XIV S.A.S. - E.S.P.,COLOMBIA 14,[14]
2,BOSQUES SOLARES DE LOS LLANOS 8 Y SU CONEXIÓN ...,BOSQUES SOLARES LLANOS 8,[8]
3,PEQUEÑA CENTRAL HIDROELÉCTRICA AKUA 1,AKUA 1,[1]
4,PLANTA TÉRMICA EL TESORITO II 200 MW,TESORITO 2,[2]


**Lectura.** De cada nombre queda solo lo que lo identifica: "PRIMAVERA", "COLOMBIA 14", "BOSQUES SOLARES LLANOS 8", "AKUA 1" y "TESORITO 2". "SOLARES" se conserva a propósito: es parte de la marca "Bosques Solares de los Llanos". Los números que quedan (14, 8, 1 y 2) distinguen etapas y unidades.

**Puntuar.** RapidFuzz da un puntaje de 0 a 100 a cada par de nombres. Se usan dos medidas:
- `token_set_ratio` compara los conjuntos de palabras: da 100 si las palabras de un nombre están contenidas en el otro, sin importar el orden. Ayuda ("LOMA" frente a "LOMA") y engaña: "LOMA 2", una segunda etapa, también saca 100.
- `WRatio` combina varias medidas (cadena completa, parcial y por palabras) y se queda con la mejor, ponderada.

In [21]:
def candidatos(izq, der, top_k=3):
    """Los `top_k` mejores candidatos de `der` para cada registro de `izq`, dentro del mismo bloque de tecnología.

    Ordena por token_set_ratio y desempata con WRatio. Devuelve una fila por par, con las columnas de
    cada lado y los sufijos _izq y _der.
    """
    tsr = process.cdist(izq["nombre_norm"], der["nombre_norm"], scorer=fuzz.token_set_ratio, workers=-1)
    wr = process.cdist(izq["nombre_norm"], der["nombre_norm"], scorer=fuzz.WRatio, workers=-1)
    ti, td = izq["tecnologia"].to_numpy(), der["tecnologia"].to_numpy()
    bloque = (ti[:, None] == td[None, :]) | (ti == "desconocida")[:, None] | (td == "desconocida")[None, :]
    clave = np.where(bloque, tsr * 1000 + wr, -1.0)                  # fuera del bloque, nunca es candidato
    filas = [(i, j, tsr[i, j], wr[i, j]) for i in range(len(izq))
             for j in np.argsort(-clave[i], kind="stable")[:top_k] if clave[i, j] >= 0]
    pares = pd.DataFrame(filas, columns=["i", "j", "token_set_ratio", "wratio"])
    a = izq.add_suffix("_izq").iloc[pares["i"]].reset_index(drop=True)
    b = der.add_suffix("_der").iloc[pares["j"]].reset_index(drop=True)
    return pd.concat([pares.drop(columns=["i", "j"]), a, b], axis=1)

**Revisar atributos.** Cada par pasa cinco pruebas. Cada una da verdadero, falso o "no aplica" (faltante) cuando no hay con qué comparar:

| Prueba | Regla | Ejemplo que descarta |
|---|---|---|
| `numeros_ok` | Las etapas coinciden; un número de un solo lado, distinto de 1, es otra etapa | "TESORITO 2" frente a "TESORITO" |
| `capacidad_ok` | Capacidades iguales con ±10% o ±2 MW | Un proyecto de 150 MW frente a un parque de 2 MW |
| `factor_ok` | OEF / capacidad dentro de un rango posible para la tecnología | 0,4 GWh-día de OEF en un autogenerador de 100 kW |
| `fecha_ok` | La planta de XM no operaba más de un año antes de que apareciera el proyecto | Una planta que opera desde 2015 no es un proyecto de 2019 |
| `codigo_ok` | La planta no tenía ya OEF con su propio código cuando se le asignó al proyecto | Plantas con OEF propia desde antes |

In [22]:
FACTOR_ADMISIBLE = {"solar": (0.01, 0.35), "eolica": (0.02, 0.60), "hidraulica": (0.02, 1.0),
                    "termica": (0.20, 1.05), "biomasa": (0.05, 1.0), "desconocida": (0.01, 1.05)}


def numeros_ok(a, b):
    """True si los números coinciden, False si son otra etapa y NaN si no hay evidencia (solo un "1" o nada)."""
    if a and b:
        return a == b
    solo = a or b
    return np.nan if not solo or solo == {"1"} else False

In [23]:
def atributos(c):
    """Añade a cada par las cinco pruebas (True, False o NaN si no aplica) y si tiene algún conflicto."""
    c = c.copy()
    c["numeros_ok"] = [numeros_ok(numeros(a), numeros(b)) for a, b in zip(c["nombre_norm_izq"], c["nombre_norm_der"])]
    ci, cd = c["capacidad_mw_izq"], c["capacidad_mw_der"]
    tolerancia = np.maximum(0.10 * np.maximum(ci, cd), 2.0)
    c["capacidad_ok"] = np.where(ci.notna() & cd.notna(), (ci - cd).abs() <= tolerancia, np.nan)
    oef = c["oef_gwh_dia_izq"].fillna(c["oef_gwh_dia_der"])                 # OEF del lado que la tiene
    factor = oef * 1e3 / 24 / np.where(c["oef_gwh_dia_izq"].notna(), cd, ci)  # MW medios / capacidad del otro
    tec = np.where(c["tecnologia_izq"] == "desconocida", c["tecnologia_der"], c["tecnologia_izq"])
    limites = np.array([FACTOR_ADMISIBLE[t] for t in tec])
    c["factor_ok"] = np.where(factor.notna(), (factor >= limites[:, 0]) & (factor <= limites[:, 1]), np.nan)
    c["fecha_ok"] = c["codigo_ok"] = np.nan
    if (c["fuente_der"] == "xm").all():
        alta = c["fecha_alta_izq"]                                           # cuándo apareció el proyecto
        c["fecha_ok"] = np.where(c["fecha_der"].notna() & alta.notna(), c["fecha_der"] >= alta - pd.Timedelta(days=365), np.nan)
        if (c["fuente_izq"] == "oef").all():
            c["codigo_ok"] = np.where(c["fecha_alta_der"].notna() & alta.notna(), c["fecha_alta_der"] > alta, np.nan)
    pruebas = ["numeros_ok", "capacidad_ok", "factor_ok", "fecha_ok", "codigo_ok"]
    c["conflicto"] = (c[pruebas] == False).any(axis=1)  # noqa: E712  (NaN no es conflicto)
    return c

Con las tres funciones se arman los candidatos de las tres tareas: OEF→XM (¿el proyecto ya está en XM?), OEF→UPME (¿está en el registro?) y UPME→XM (¿el proyecto del registro ya opera?). En OEF→XM solo entran los códigos `PROG`: un elemento con código de planta no puede ser otra planta.

In [24]:
bases = leer("homologacion_bases.csv", ["fecha", "fecha_alta"], dtype={"id": str})
bases["nombre_norm"] = bases["nombre"].map(normalizar)
print(bases["fuente"].value_counts().to_string())

TAREAS = {"oef_xm": ("oef", "xm"), "oef_upme": ("oef", "upme"), "upme_xm": ("upme", "xm")}
partes = []
for tarea, (a, b) in TAREAS.items():
    izq = bases[bases["fuente"] == a]
    if tarea == "oef_xm":
        izq = izq[izq["id"].str.startswith("PROG")]
    der = bases[bases["fuente"] == b]
    partes.append(atributos(candidatos(izq.reset_index(drop=True), der.reset_index(drop=True))).assign(tarea=tarea))
cand = pd.concat(partes, ignore_index=True)
print(f"\n{len(cand)} pares candidatos; {cand['conflicto'].sum()} con algún conflicto de atributos")
cand.groupby("tarea").agg(pares=("conflicto", "size"), con_conflicto=("conflicto", "sum"))

fuente
xm      2499
upme      52
oef       49



429 pares candidatos; 319 con algún conflicto de atributos


,pares,con_conflicto
tarea,,
oef_upme,147,57
oef_xm,126,110
upme_xm,156,152


**Lectura.** Las bases tienen 49 registros con OEF, 52 proyectos de la UPME y 2.499 plantas de XM. Con los tres mejores candidatos de cada registro quedan 429 pares, y 319 tienen algún conflicto de atributos: la mayoría de los candidatos por nombre no son la misma planta. En UPME→XM chocan 152 de 156: los atributos descartan casi todos los candidatos que propone el nombre.

**Escoger el umbral.** Un par se acepta si es el mejor candidato de su registro y su puntaje alcanza el umbral. Para escoger el umbral hace falta saber cuáles pares son de verdad la misma planta: el paquete trae 205 pares etiquetados con su evidencia (agente, capacidad, etapa y fechas), 17 de ellos parejas verdaderas. Las etiquetas las propuso una IA y otro agente de IA las revisó par por par, con evidencia. Después revisamos los 24 pares que deciden las métricas (las 17 parejas verdaderas y los que la IA había dejado con confianza media o baja) y los verificamos en fuentes públicas, con una IA para la búsqueda: ninguna etiqueta cambió (declaración de uso de IA). Para cada umbral se mide:
- **Precisión:** de los pares aceptados, cuántos son verdaderos.
- **Exhaustividad:** de las parejas verdaderas, cuántas se encuentran.
- **F1:** la media armónica de las dos.

In [25]:
def curva(c, etiquetas, medidas=("token_set_ratio", "wratio"), umbrales=range(50, 101, 2)):
    """Precisión, exhaustividad y F1 de cada medida y umbral contra los pares revisados, sin y con atributos."""
    llave = ["tarea", "id_izq", "id_der"]
    filas = []
    for medida in medidas:
        for regla, ok in [("solo puntaje", pd.Series(True, index=c.index)), ("con atributos", ~c["conflicto"])]:
            s = c[medida].where(ok)
            mejor = s == s.groupby([c["tarea"], c["id_izq"]]).transform("max")
            base = c[llave].assign(s=s, mejor=mejor).merge(etiquetas, on=llave, how="right", validate="one_to_one")
            for u in umbrales:
                pred = base["mejor"].eq(True) & (base["s"] >= u)
                vp = int((pred & (base["es_misma"] == 1)).sum())
                fp = int((pred & (base["es_misma"] == 0)).sum())
                fn = int((~pred & (base["es_misma"] == 1)).sum())
                prec = vp / (vp + fp) if vp + fp else np.nan
                exh = vp / (vp + fn) if vp + fn else np.nan
                f1 = 2 * prec * exh / (prec + exh) if prec and exh else 0.0
                filas.append((medida, regla, u, vp, fp, fn, prec, exh, f1))
    return pd.DataFrame(filas, columns=["medida", "regla", "umbral", "vp", "fp", "fn", "precision", "exhaustividad", "f1"])


def elegir_umbral(cur, precision_min=0.9):
    """Regla con atributos: entre los umbrales con precisión >= precision_min, el de mayor F1. Si hay empate,
    la medida que lo sostiene en más umbrales y, dentro de ella, el umbral más bajo."""
    ok = cur[(cur["regla"] == "con atributos") & (cur["precision"] >= precision_min)]
    top = ok[np.isclose(ok["f1"], ok["f1"].max())]
    estable = top.groupby("medida")["umbral"].count().sort_values(ascending=False).index[0]
    return top[top["medida"] == estable].sort_values("umbral").iloc[0]

In [26]:
etiquetas = leer("homologacion_etiquetas.csv", dtype={"id_izq": str, "id_der": str})
print(f"{len(etiquetas)} pares revisados, {int(etiquetas['es_misma'].sum())} parejas verdaderas")
cur = curva(cand, etiquetas)
elegido = elegir_umbral(cur)
print(f"Umbral elegido: {elegido['medida']} >= {elegido['umbral']:.0f} (precisión {es(elegido['precision'], 2)}, "
      f"exhaustividad {es(elegido['exhaustividad'], 2)}, F1 {es(elegido['f1'], 2)})")
assert (elegido["medida"], elegido["umbral"]) == ("token_set_ratio", 86), "no es el umbral que usa el pipeline"
cur[cur["umbral"].isin([70, 86, 100])].pivot_table(index=["medida", "regla"], columns="umbral",
                                                   values=["precision", "exhaustividad"]).round(2)

205 pares revisados, 17 parejas verdaderas


Umbral elegido: token_set_ratio >= 86 (precisión 0,92, exhaustividad 0,71, F1 0,80)


exhaustividad           precision          
umbral                                  70   86   100       70   86   100
medida          regla                                                    
token_set_ratio con atributos          0,71 0,71 0,71      0,63 0,92 0,92
                solo puntaje           0,71 0,71 0,71      0,14 0,19 0,23
wratio          con atributos          0,71 0,71 0,53       0,6 0,92    1
                solo puntaje           0,71 0,71 0,53      0,13 0,19 0,75

La figura muestra las curvas de la medida elegida, sin y con la revisión de atributos.

In [27]:
d = cur[cur["medida"] == elegido["medida"]].melt(id_vars=["regla", "umbral"], value_vars=["precision", "exhaustividad", "f1"],
                                                   var_name="métrica", value_name="valor")
fig = px.line(d, x="umbral", y="valor", color="métrica", facet_col="regla", markers=True,
              category_orders={"regla": ["solo puntaje", "con atributos"]},
              labels={"umbral": "Umbral de puntaje (0-100)", "valor": "Proporción", "métrica": "Métrica"},
              title=f"Homologación: precisión, exhaustividad y F1 según el umbral ({elegido['medida']})")
fig.add_vline(x=elegido["umbral"], line_dash="dash", line_color="gray", col=2,
              annotation_text=f"umbral elegido: {elegido['umbral']:.0f}")
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("regla=", "Regla: ")))
fig.update_layout(height=420, legend=dict(orientation="h", y=-0.25))
fig.show()

**Lectura.** 
- **Solo con el puntaje, la precisión es baja.** Con `token_set_ratio` no pasa de 0,23; con `WRatio` llega a 0,75 en el umbral 100, pero encuentra apenas 9 de las 17 parejas (exhaustividad de 0,53). La causa son los cientos de autogeneradores pequeños de XM, cuyos nombres comparten palabras con las sociedades de los proyectos.
- **Con los atributos, la precisión sube a 0,92 desde el umbral 86.** La exhaustividad se queda en 0,71 en todos los umbrales, porque cinco parejas verdaderas no se parecen por nombre. En el pipeline, una regla adicional (la única planta del mismo agente) encuentra tres de ellas.
- **El umbral elegido es `token_set_ratio` ≥ 86,** con F1 de 0,80. Entre 70 y 86 queda la zona gris, que revisa una persona: son los dos umbrales de Fellegi y Sunter (sección 1).

**Fuera de muestra.** Esas métricas se midieron con las mismas etiquetas con que escogimos el umbral, así que pueden ser optimistas. Para comprobarlo partimos los 205 pares al azar en dos mitades, con la misma proporción de parejas verdaderas, escogemos la regla con una mitad y la evaluamos con la otra, en los dos sentidos.

In [28]:
# Fuera de muestra: se escoge la regla con una mitad de las etiquetas y se evalúa con la otra, en los dos sentidos
mitad_a = etiquetas.groupby("es_misma").sample(frac=0.5, random_state=SEMILLA)   # la misma proporción de parejas verdaderas
mitad_b = etiquetas.drop(mitad_a.index)
filas = []
for mitad, escoge, evalua in [("A", mitad_a, mitad_b), ("B", mitad_b, mitad_a)]:
    elegida = elegir_umbral(curva(cand, escoge))
    fuera = curva(cand, evalua, medidas=(elegida["medida"],), umbrales=[elegida["umbral"]])
    fuera = fuera[fuera["regla"] == "con atributos"].iloc[0]
    filas.append({"mitad que escoge": mitad, "pares": len(escoge), "parejas verdaderas": int(escoge["es_misma"].sum()),
                  "regla escogida": f"{elegida['medida']} >= {elegida['umbral']:.0f}",
                  "precisión en la otra mitad": fuera["precision"], "exhaustividad en la otra mitad": fuera["exhaustividad"],
                  "F1 en la otra mitad": fuera["f1"]})
pd.DataFrame(filas).set_index("mitad que escoge").round(2)

,pares,parejas verdaderas,regla escogida,precisión en la otra mitad,exhaustividad en la otra mitad,F1 en la otra mitad
mitad que escoge,,,,,,
A,102,8,wratio >= 96,1,"0,56","0,71"
B,103,9,token_set_ratio >= 86,"0,83","0,62","0,71"


**Lectura.** Las dos mitades no escogen la misma regla. La B escoge `token_set_ratio` ≥ 86, la misma que sale con todas las etiquetas; la A, `WRatio` ≥ 96. Lo decide un solo par, Andrómeda, el único que el método acepta y la etiqueta rechaza (sección 4.9): cayó en la mitad A, y con él ningún umbral de `token_set_ratio` llega a 0,90 de precisión, el mínimo que pide `elegir_umbral()`. Evaluadas con la mitad que no vieron, las reglas dan una exhaustividad de 0,56 y de 0,62: el 0,71 medido con todas las etiquetas era optimista. La precisión queda a lado y lado de 0,92 (1,00 y 0,83). Con 17 parejas verdaderas, las dos cifras dependen de la partición. Para lo que sigue conservamos `token_set_ratio` ≥ 86, la regla escogida con todas las etiquetas: como muestra la figura, de 86 a 100 las métricas no cambian, y cualquier umbral en ese rango acepta las mismas parejas.

**De la homologación a la oferta y a los retrasos.** Las parejas aceptadas dicen qué proyectos con OEF ya están en el listado de XM, y la sección 3.3 las usa para no contar dos veces su energía firme. Los retrasos de las subastas vienen en `retrasos_subastas.csv`, que el pipeline armó con su tabla maestra de la homologación y la bitácora de SIMEM: el notebook usa esa tabla, no la rehace. La subasta de 2019 es la única con plantas nuevas cuyo compromiso ya venció para todas: es la evidencia para modelar los retrasos de los proyectos que faltan. Ocho de sus nueve plantas ya figuran en el listado de XM con su código, y la novena, Windpeshi, como proyecto con OEF.

In [29]:
ret = leer("retrasos_subastas.csv", ["compromiso", "fecha_entrada"])
s2019 = ret[ret["mecanismo"].str.contains("TERCERA SUBASTA") & ret["mecanismo"].str.contains("Nuevas")].copy()
entro = s2019["estado_entrada"].isin(["en operación", "en pruebas"])
s2019["grupo"] = np.select([entro & (s2019["retraso_meses"] > 1), entro], ["entró tarde", "entró a tiempo"], "no ha entrado")
resumen_2019 = s2019.groupby("grupo").agg(plantas=("nombre", "size"), oef_gwh_dia=("oef_registrada_gwh_dia", "sum"))
resumen_2019["fraccion_oef"] = resumen_2019["oef_gwh_dia"] / resumen_2019["oef_gwh_dia"].sum()
tarde = s2019.loc[s2019["grupo"] == "entró tarde", "retraso_meses"]

# El modelo de retrasos del caso base: fracciones de OEF redondeadas a dos decimales y el retraso en períodos
MODELO_RETRASOS = {"a_tiempo": round(float(resumen_2019.loc["entró a tiempo", "fraccion_oef"]), 2),
                   "tarde": round(float(resumen_2019.loc["entró tarde", "fraccion_oef"]), 2),
                   "periodos_retraso": int(np.ceil(tarde.mean() / 12))}
print(f"Retrasos de las que entraron tarde: {', '.join(es(m, 1) for m in tarde)} meses (media de {es(tarde.mean(), 1)})")
print(f"Modelo de retrasos: {es(100 * MODELO_RETRASOS['a_tiempo'])}% de la OEF a tiempo y {es(100 * MODELO_RETRASOS['tarde'])}% "
      f"con {MODELO_RETRASOS['periodos_retraso']} períodos de retraso; el resto no entra")
display(s2019[["nombre", "compromiso", "fecha_entrada", "estado_entrada", "retraso_meses", "oef_registrada_gwh_dia"]])
resumen_2019.round(3)

Retrasos de las que entraron tarde: 27,6, 18,8, 15,7 meses (media de 20,7)
Modelo de retrasos: 53% de la OEF a tiempo y 19% con 2 períodos de retraso; el resto no entra


,nombre,compromiso,fecha_entrada,estado_entrada,retraso_meses,oef_registrada_gwh_dia
0,TERMOCARIBE III 1,2021-12-01,2024-03-21,en operación,"27,6","0,88947"
1,LATAM SOLAR LA LOMA,2022-12-01,2024-06-24,en operación,"18,8","0,525902"
2,TESORITO,2022-12-01,2022-09-13,en operación,"-2,6","4,572488"
3,EL PASO,2022-12-01,2024-03-23,en operación,"15,7","0,237644"
55,ENEL COLOMBIA SA ESP,2021-12-01,NaT,no ha entrado (vencido),"59,1","0,781183"
56,FUTURA - PARQUE ALPHA,2022-12-01,NaT,no ha entrado (vencido),"47,1","0,153058"
57,FUTURA - APOTOLORRU,2022-12-01,NaT,no ha entrado (vencido),"47,1","0,446234"
58,FUTURA - PARQUE BETA,2022-12-01,NaT,no ha entrado (vencido),"47,1","0,202152"
59,FUTURA - CASA ELECTRICA,2022-12-01,NaT,no ha entrado (vencido),"47,1","0,890679"


,plantas,oef_gwh_dia,fraccion_oef
grupo,,,
entró a tiempo,1,"4,572","0,526"
entró tarde,3,"1,653","0,19"
no ha entrado,5,"2,473","0,284"


**Lectura.** De las nueve plantas nuevas de la subasta de 2019, solo Tesorito entró a tiempo, y de hecho unos tres meses antes. Tres entraron tarde, entre 16 y 28 meses: El Paso, La Loma y Termocaribe III. Cinco no han entrado, con 47 meses de retraso o más: Windpeshi (de Enel) y los cuatro parques eólicos de La Guajira (Alpha, Beta, Apotolorru y Casa Eléctrica). Medido por OEF: 53% a tiempo, 19% tarde y 28% sin entrar. Las que entraron tarde se demoraron 20,7 meses en promedio, que ocupan dos períodos del cargo. Ese es el modelo de retrasos del caso base; reemplaza el "20% de proyectos retrasados" que suponía la propuesta sin fuente.

### 3.2 Costos y bloques candidatos

En la sección 3.1 vimos los retrasos de los proyectos con OEF. Antes de medir la brecha (3.3), aquí preparamos las seis tecnologías con las que el modelo de expansión (3.4) podrá cerrarla, con su costo anual por MWh firme y la fuente de cada parámetro.

El modelo de expansión escoge entre **bloques**: una tecnología con un tamaño fijo (100 MW de ciclo abierto, por ejemplo), que puede entrar desde un período mínimo y hasta un cupo. Para compararlos hace falta llevar la inversión a un costo anual. El **factor de recuperación de capital** (CRF) convierte una inversión en una cuota anual con la tasa de descuento $r$ y la vida útil $n$:

$$\text{CRF}(r, n) = \frac{r(1+r)^n}{(1+r)^n - 1}, \qquad C_b = \text{MW}_b \big[\text{CAPEX}_b \cdot \text{CRF}(r, n_b) + \text{O\&M fijo}_b\big] + \text{costo variable}_b$$

El costo variable es el de la energía que se espera que la planta genere (combustible y O&M variable). La **energía firme** de un bloque es $\text{MW}_b \times 8.760 \text{ h} \times f_b$, con el factor firme $f_b$ (el del PIEG, Tabla 11, o el observado en XM donde el PIEG no lo trae). Dividir el costo anual por la energía firme da el **costo anual por MWh firme**: lo que cuesta cubrir 1 MWh de brecha durante un año con esa tecnología.

**De dónde salen los costos.** Cada parámetro viene de la fuente de mayor jerarquía que lo trae. Primero, el Catálogo Tecnológico Colombiano de la UPME: el CAPEX, el O&M y la eficiencia del gas y de la eólica. Después, el PIEG 2025-2039: el CAPEX de la solar a gran escala, que el Catálogo no trae, las vidas útiles, los factores firmes, los cupos y la tasa de descuento. Para los motores diésel y la cogeneración con bagazo, que no traen ni el Catálogo ni el PIEG, CNE (Chile) y EPE (Brasil). Los factores de planta se miden en XM. El ATB 2025 de NREL da los escenarios bajo y alto del CAPEX, y todo se lleva a USD de 2025 con el CPI de EE. UU. Como tasa de descuento usamos el 10% anual del PIEG (p. 131). El PIEG no dice si es real o nominal; la usamos como real porque los costos están en USD constantes, y con 8% y 12% (pipeline) la energía sin cubrir cambia en 20 GWh o menos. `fuentes_supuestos.csv` trae, para cada parámetro, su fuente, la tabla o la página, la URL y la conversión.

In [30]:
fuentes = leer("fuentes_supuestos.csv")
fuentes["numero"] = pd.to_numeric(fuentes["valor"], errors="coerce")   # el primer período es texto, no número


def supuesto(tecnologia, parametro, escenario="central"):
    """Valor numérico de un supuesto de fuentes_supuestos.csv."""
    fila = fuentes.query("tecnologia == @tecnologia and parametro == @parametro and escenario == @escenario")
    return float(fila["numero"].iloc[0])


TASA = supuesto("todas", "tasa_descuento")   # 10% anual del PIEG 2025-2039 (p. 131), usada como tasa real
# MW que pueden entrar en el horizonte por grupo; el gas es un cupo compartido por ciclo abierto y combinado
CUPOS = {"solar_fv": supuesto("solar_fv", "cupo"), "eolica_terrestre": supuesto("eolica_terrestre", "cupo"),
         "gas": supuesto("gas_ciclo_abierto", "cupo"), "diesel_motores": supuesto("diesel_motores", "cupo"),
         "biomasa_bagazo": supuesto("biomasa_bagazo", "cupo")}
NOMBRES_TECNOLOGIA = {"solar_fv": "Solar FV", "eolica_terrestre": "Eólica", "biomasa_bagazo": "Bagazo",
                      "gas_ciclo_abierto": "Gas, ciclo abierto", "gas_ciclo_combinado": "Gas, ciclo combinado",
                      "diesel_motores": "Motores diésel"}
candidatos_costos = leer("candidatos_costos.csv")


def crf(r, n):
    """Factor de recuperación de capital: r(1+r)^n / ((1+r)^n − 1). Anualiza una inversión."""
    return r * (1 + r) ** n / ((1 + r) ** n - 1)

In [31]:
def bloques(escenario_costos="central", r=TASA, cupos=None, retraso_periodos=0, objetivo="costo_total"):
    """Un bloque por tecnología candidata, con su costo anual (MUSD), su factor firme y su primer período.

    escenario_costos: "bajo", "central" o "alto". r: tasa de descuento. cupos: cambios a CUPOS. retraso_periodos: corre el primer período factible. objetivo: "costo_total"
    (fijo + variable) o "solo_fijo" (la lógica del cargo, que paga la firmeza y no la energía).
    """
    c = candidatos_costos[candidatos_costos["escenario_costos"] == escenario_costos].copy()
    c["costo_fijo_musd_ano"] = c["tamano_mw"] * (c["capex_usd_kw"] * crf(r, c["vida_util"]) + c["om_fijo_usd_kw_ano"]) / 1e3
    c["costo_anual_musd"] = c["costo_fijo_musd_ano"] + (c["costo_variable_musd_ano"] if objetivo == "costo_total" else 0.0)
    c["k_min"] = c["primer_periodo"].map(PERIODOS.index) + retraso_periodos
    c["cupo_mw"] = c["grupo_cupo"].map({**CUPOS, **(cupos or {})}).astype(float)
    c["costo_firme_usd_mwh"] = c["costo_anual_musd"] / (c["tamano_mw"] * 8.76 * c["factor_firme"] / 1e3)
    return c.reset_index(drop=True)


COLUMNAS_BLOQUES = {"tamano_mw": "tamaño (MW)", "primer_periodo": "primer período", "cupo_mw": "cupo (MW)",
                    "factor_firme": "factor firme", "costo_fijo_musd_ano": "costo fijo (MUSD/año)",
                    "costo_variable_musd_ano": "costo variable (MUSD/año)", "costo_firme_usd_mwh": "USD por MWh firme"}
(bloques().set_index("tecnologia")[list(COLUMNAS_BLOQUES)].sort_values("costo_firme_usd_mwh").round(2)
          .rename(columns=COLUMNAS_BLOQUES, index=NOMBRES_TECNOLOGIA))

,tamaño (MW),primer período,cupo (MW),factor firme,costo fijo (MUSD/año),costo variable (MUSD/año),USD por MWh firme
tecnologia,,,,,,,
Bagazo,20,2028-2029,40,"0,54","1,94",0,"20,54"
"Gas, ciclo combinado",300,2029-2030,1.400,"0,9","45,09","46,84","38,87"
Motores diésel,20,2027-2028,0,"0,88","1,33","5,96","47,24"
"Gas, ciclo abierto",100,2028-2029,1.400,"0,9","13,27","27,14","51,26"
Solar FV,100,2027-2028,7.918,"0,13","10,84",0,"95,19"
Eólica,100,2029-2030,464,"0,14","17,84",0,"145,43"


**Lectura.** 
- **Costo por MWh firme:** bagazo 21 USD, ciclo combinado 39, motores diésel 47, ciclo abierto 51, solar 95 y eólica 145. La solar y la eólica son caras por MWh firme porque el PIEG les reconoce poca energía firme: un bloque solar de 100 MW aporta unos 114 GWh firmes al año, frente a 788 de un ciclo abierto del mismo tamaño.
- **El calendario pesa más que el costo:** en 2027-2028 solo podrían llegar la solar y el diésel, que no tiene cupo; el ciclo abierto y el bagazo, desde 2028-2029, y el ciclo combinado y la eólica, en 2029-2030.
- **Cuidado con esta métrica:** ignora el valor de la energía que producen las renovables, que desplaza generación térmica. La sección 3.5 prueba el objetivo de solo costo fijo.

**La fuente de cada parámetro.** La tabla de abajo dice de dónde sale cada parámetro de cada bloque; el detalle (tabla o página, URL y conversión) está en `fuentes_supuestos.csv`. Después, dos comprobaciones: que los costos del gas y de la eólica salen del JSON del Catálogo, y que los costos de los bloques salen de esos parámetros.

In [32]:
PARAMETROS = {"capex": "CAPEX", "om_fijo": "O&M fijo", "vida_util": "Vida útil", "construccion": "Plazo de obra",
              "primer_periodo": "Primer período", "factor_planta": "Factor de planta", "precio_combustible": "Combustible",
              "factor_firme": "Factor firme", "cupo": "Cupo", "tamano_bloque": "Tamaño del bloque"}
(fuentes.query("escenario == 'central' and parametro in @PARAMETROS and tecnologia in @NOMBRES_TECNOLOGIA")
        .pivot(index="parametro", columns="tecnologia", values="fuente_corta")
        .reindex(index=list(PARAMETROS), columns=list(NOMBRES_TECNOLOGIA)).fillna("—")
        .rename(index=PARAMETROS, columns=NOMBRES_TECNOLOGIA))

tecnologia,Solar FV,Eólica,Bagazo,"Gas, ciclo abierto","Gas, ciclo combinado",Motores diésel
parametro,,,,,,
CAPEX,"PIEG, Gráfica 8",Catálogo UPME,"EPE, p. 10",Catálogo UPME,Catálogo UPME,"CNE, §5.4.3"
O&M fijo,CNE y EPE,Catálogo UPME,"EPE, p. 10",Catálogo UPME,Catálogo UPME,"CNE, Tabla 14"
Vida útil,"PIEG, p. 131","PIEG, p. 131","PIEG, p. 131","PIEG, p. 131","PIEG, p. 131","PIEG, p. 131"
Plazo de obra,"CNE, Tabla 15",Catálogo UPME,"EPE, p. 10",Catálogo UPME,Catálogo UPME,"CNE, Tabla 15"
Primer período,Cálculo propio,Cálculo propio,Cálculo propio,Cálculo propio,Cálculo propio,Cálculo propio
Factor de planta,"XM, observado",Catálogo UPME,"XM, observado","XM, observado","XM, observado","XM, observado"
Combustible,—,—,—,"UPME, precios","UPME, precios","UPME, precios"
Factor firme,"PIEG, Tabla 11","PIEG, Tabla 11","XM, observado","PIEG, Tabla 11","PIEG, Tabla 11","XM, observado"
Cupo,"PIEG, Tabla 23",UPME y prensa,Asocaña,"PIEG, Tabla 23",—,"UPME, registro"


In [33]:
# 1. El CAPEX, el O&M fijo y el consumo térmico del gas y de la eólica salen del JSON de la API del Catálogo
catalogo = pd.DataFrame(json.loads((CARPETA / "upme_catalogo_tecnologias.json").read_text(encoding="utf-8")))
CATALOGO = {"gas_ciclo_abierto": "Turbina de gas ciclo simple", "gas_ciclo_combinado": "Ciclo combinado gas",
            "eolica_terrestre": "Eólica terrrestre gran esc."}          # "terrrestre": así viene en la API


def del_catalogo(tecnologia, variable):
    """Valor de una variable del Catálogo en 2027: interpolación lineal entre 2024 y 2030."""
    fila = catalogo[(catalogo["technology"] == CATALOGO[tecnologia]) & (catalogo["variable"] == variable)].iloc[0]
    return fila["data_2024"] + (fila["data_2030"] - fila["data_2024"]) * (2027 - 2024) / 6


f24 = supuesto("todas", "factor_usd_2024_2025")          # USD de 2024 -> USD de 2025, con el CPI de EE. UU.
filas = []
for tec in CATALOGO:
    filas += [(tec, "capex", del_catalogo(tec, "Inversión específica - sistema total") * 1e3 * f24),   # M USD/MW -> USD/kW
              (tec, "om_fijo", del_catalogo(tec, "O&M fijas") / 1e3 * f24)]                          # USD/MW -> USD/kW
    if tec != "eolica_terrestre":   # eficiencia en poder calorífico inferior -> MMBtu/MWh en poder calorífico superior
        filas.append((tec, "heat_rate", 3.412 / (del_catalogo(tec, "Eficiencia eléctrica neta media anual") / 100) * 1.108))
desde_json = pd.DataFrame(filas, columns=["tecnologia", "parametro", "valor"]).set_index(["tecnologia", "parametro"])["valor"]
en_tabla = fuentes.query("escenario == 'central'").set_index(["tecnologia", "parametro"])["numero"].reindex(desde_json.index)
comprobar(desde_json, en_tabla, "Costos del gas y de la eólica desde el JSON del Catálogo", 1e-4, "fuentes_supuestos.csv")


✓ Costos del gas y de la eólica desde el JSON del Catálogo: coincide con fuentes_supuestos.csv (diferencia máxima 5,6e-07)


In [34]:
# 2. El costo fijo y el variable de los seis bloques salen de los parámetros de la tabla de fuentes
par = (fuentes.query("escenario == 'central'").pivot(index="tecnologia", columns="parametro", values="numero")
              .reindex(list(NOMBRES_TECNOLOGIA)))
costo_unitario = par["om_variable"].fillna(0) + par["heat_rate"].fillna(0) * par["precio_combustible"].fillna(0)  # USD/MWh
desde_fuentes = pd.DataFrame({
    "fijo": par["tamano_bloque"] * (par["capex"] * crf(TASA, par["vida_util"]) + par["om_fijo"]) / 1e3,
    "variable": par["tamano_bloque"] * 8.76 * par["factor_planta"] * costo_unitario / 1e3})
centrales = candidatos_costos.query("escenario_costos == 'central'").set_index("tecnologia")
comprobar(desde_fuentes, centrales[["costo_fijo_musd_ano", "costo_variable_musd_ano"]].set_axis(["fijo", "variable"], axis=1),
          "Costo fijo y variable de los 6 bloques (MUSD/año) desde sus parámetros", 1e-4, "candidatos_costos.csv")

✓ Costo fijo y variable de los 6 bloques (MUSD/año) desde sus parámetros: coincide con candidatos_costos.csv (diferencia máxima 1,1e-08)


**Lectura.** Cada parámetro tiene una fuente identificada, y las dos comprobaciones pasan: el JSON del Catálogo reproduce los costos del gas y de la eólica, y los costos de los seis bloques salen exactamente de sus parámetros.
- **El Catálogo de la UPME** da el CAPEX, el O&M y la eficiencia del gas y de la eólica; **el PIEG**, el CAPEX de la solar, la vida útil de todos, los factores firmes de la solar, la eólica y el gas, y los cupos de la solar y del gas. **CNE y EPE** dan el diésel y el bagazo: una sola fuente cada uno.
- **XM** da los factores de planta (salvo el eólico, del Catálogo) y los factores firmes del diésel y del bagazo, que el PIEG no trae. El ciclo combinado no tiene cupo propio: comparte el del gas con el ciclo abierto. En las térmicas, el factor de planta mide el despacho: sirve para el costo variable, no para la energía firme.
- **No todo es un dato.** El tamaño de los bloques es una decisión de modelación. El primer período sale del plazo de obra de la fuente, contado desde el corte: así resultan los del gas y el bagazo. La solar y el diésel entran un período antes de lo que da su plazo, y la eólica uno después, por el calendario de sus proyectos; el detalle está en la columna `como` de la tabla.

### 3.3 Balance de energía firme

Con la demanda de la sección 2.5, el parque de XM y los proyectos con sus retrasos (3.1), en esta sección medimos cuánta energía firme falta en cada período, con la hidrología crítica de la ENFICC y con un año seco y uno normal observados.

La brecha de cada período $t$ es la demanda menos la oferta firme:

$$\text{Brecha}_t = \text{Demanda}_t - \Big(\underbrace{\text{ENFICC}_t}_{\text{plantas que operan}} + \underbrace{\textstyle\sum_m \text{MW}_m \cdot 24 \cdot f_m / 1000}_{\text{menores sin ENFICC}} + S_t + \sum_j \text{OEF}_{j,t} \cdot e_{j,t}\Big) \times \text{días}_t$$

Los términos del paréntesis están en GWh por día, y su suma se multiplica por los días del período. Cada uno tiene una regla:
- **Plantas con ENFICC:** cuenta su ENFICC de octubre de 2026 (el mes de referencia) si están en operación o en pruebas. Las registradas sin pruebas no generaron en doce meses y no suman.
- **Menores sin ENFICC:** las plantas pequeñas no despachadas centralmente no tienen ENFICC propia. Se cuentan con su capacidad × 24 h × el factor firme del PIEG de su tecnología ($f_m$), salvo las que probablemente ya son generación distribuida, que la UPME resta de la demanda: nombre que empieza por "GD", "MINIGRANJA" o solar de 1 MW o menos (la generación distribuida es de menos de 1 MW y la autogeneración a pequeña escala, de hasta 1 MW: Resolución CREG 174 de 2021, art. 3).
- **Retiros:** Termoyopal 2 sale el 1 de diciembre de 2027 (PIEG, Tabla 6, p. 26); es la única planta del cronograma de salida sin OEF para 2029-2030.
- **Entradas sin subasta ($S_t$):** las PCH y la solar de 1 a 20 MW entran cada año sin OEF, al ritmo observado en XM (`entradas_sin_subasta.csv`): 12,5 MW de PCH por año, de noviembre de 2019 a octubre de 2026, y 163 MW de solar por año, de noviembre de 2023 a octubre de 2026. Cuenta un año de entradas en 2027-2028, dos en 2028-2029 y tres en 2029-2030.
- **Proyectos nuevos:** su OEF de cada período por $e_{j,t}$, la fracción que se espera que esté operando. Sin retrasos, $e = 1$ desde su período de entrada. Con retrasos, el modelo de 3.1: el 53% entra a tiempo, el 19% dos períodos tarde y el resto no entra; un proyecto con el compromiso ya vencido no entra.

| Escenario | Demanda de la UPME | Proyectos nuevos |
|---|---|---|
| Base | Media | Con retrasos |
| Estrés | Alta (borde superior de la banda de 68%) | Con retrasos |
| Optimista | Media | Sin retrasos |

In [35]:
# Factores firmes de las menores (PIEG, Tabla 11; biomasa: XM) y tope de la generación distribuida (CREG 174 de 2021)
FACTORES_MENORES = fuentes.query("parametro == 'factor_firme_menores'").set_index("tecnologia")["numero"].to_dict()
GD_MAX_MW = supuesto("todas", "gd_max_mw")
RETIROS = {"TYP2": "2027-12-01"}   # Termoyopal 2 sale el 1-dic-2027 (PIEG 2025-2039, Tabla 6, p. 26)
plantas = leer("xm_plantas_mes_referencia.csv", dtype={"codigo": str})


def parque_firme(plantas):
    """Energía firme diaria (GWh-día) de cada planta existente, con el motivo si no suma."""
    p = plantas.copy()
    con_enficc = p["enficc_gwh_dia"] > 0
    opera = p["situacion"].isin(["en operación", "en pruebas"])
    nombre = p["nombre"].fillna("").str.upper()
    gd = ~con_enficc & (nombre.str.startswith("GD ") | nombre.str.contains("MINIGRANJA")
                        | ((p["tecnologia"] == "solar") & (p["capacidad_mw"] <= GD_MAX_MW)))
    firme_menor = (p["capacidad_mw"] * 24 * p["tecnologia"].map(FACTORES_MENORES) / 1e3).fillna(0)
    p["fuente"] = np.where(con_enficc, "ENFICC", "menor (factor firme)")
    p["motivo"] = np.select([con_enficc & ~opera, gd], ["no opera", "probable generación distribuida"], "suma")
    p["firme_gwh_dia"] = np.select([con_enficc & opera, ~con_enficc & ~gd], [p["enficc_gwh_dia"], firme_menor], 0.0)
    return p


parque = parque_firme(plantas)
suma = parque["motivo"] == "suma"
print(f"Capacidad efectiva de las plantas que suman: {es(parque.loc[suma, 'capacidad_mw'].sum() / 1e3, 1)} GW, de ellos "
      f"{es(parque.loc[suma & (parque['tecnologia'] == 'hidraulica'), 'capacidad_mw'].sum() / 1e3, 1)} hidráulicos")
parque.groupby(["fuente", "motivo"]).agg(plantas=("codigo", "size"), firme_gwh_dia=("firme_gwh_dia", "sum"),
                                         enficc_gwh_dia=("enficc_gwh_dia", "sum")).round(2)

Capacidad efectiva de las plantas que suman: 22,8 GW, de ellos 13,0 hidráulicos


plantas  firme_gwh_dia  enficc_gwh_dia
fuente               motivo                                                                 
ENFICC               no opera                              17              0            9,99
                     suma                                 138         237,69          237,69
menor (factor firme) probable generación distribuida      233              0               0
                     suma                                 243          13,65               0

**Lectura.** El parque que opera aporta 237,7 GWh-día de ENFICC en 138 plantas. Otras 17 unidades, con 10,0 GWh-día de ENFICC, no suman: están registradas sin pruebas o fuera del listado de XM, y ninguna generó en doce meses. Las menores suman 13,65 GWh-día en 243 plantas; otras 233, de 1 MW o menos o con nombre de generación distribuida, quedan por fuera para no contarlas dos veces. En total, el parque existente garantiza unos 251,3 GWh-día, con 22,8 GW de capacidad efectiva, 13,0 de ellos hidráulicos.

Ahora los proyectos nuevos. `fraccion_entrada()` aplica el modelo de retrasos de la sección 3.1 a un proyecto, y la suma sobre todos da la OEF nueva que se espera en cada período.

In [36]:
nuevos = leer("proyectos_nuevos_oef.csv")
COLUMNAS_OEF = [f"oef_{p.replace('-', '_')}" for p in PERIODOS]


def fraccion_entrada(periodo_entrada, vencido, retrasos, modelo=MODELO_RETRASOS):
    """Fracción esperada de la OEF de un proyecto que está operando en cada período del horizonte."""
    k, t = PERIODOS.index(periodo_entrada), np.arange(len(PERIODOS))
    if not retrasos:
        return (t >= k).astype(float)
    if vencido:                                   # ya pasó su compromiso sin entrar: no entra
        return np.zeros(len(PERIODOS))
    return modelo["a_tiempo"] * (t >= k) + modelo["tarde"] * (t >= k + modelo["periodos_retraso"])


def oferta_nuevos(nuevos, retrasos):
    """OEF esperada de los proyectos nuevos en cada período (GWh-día)."""
    e = np.array([fraccion_entrada(p, v, retrasos) for p, v in zip(nuevos["periodo_entrada"], nuevos["vencido"])])
    return pd.Series((nuevos[COLUMNAS_OEF].to_numpy() * e).sum(axis=0), index=PERIODOS)


def solar_comprometida(nuevos, retrasos):
    """MW solares con OEF que se espera que entren en cada período (para el tope de construcción solar)."""
    s = nuevos[nuevos["tecnologia"] == "solar"]
    e = np.array([fraccion_entrada(p, v, retrasos) for p, v in zip(s["periodo_entrada"], s["vencido"])])
    entra = np.diff(e, axis=1, prepend=0)          # lo que entra en cada período, no lo acumulado
    return pd.Series((entra * s[["mw"]].to_numpy()).sum(axis=0), index=PERIODOS)


print(f"{len(nuevos)} proyectos con OEF que no operan; por período de entrada:")
pd.DataFrame({"con retrasos": oferta_nuevos(nuevos, True), "sin retrasos": oferta_nuevos(nuevos, False)}).round(2)

41 proyectos con OEF que no operan; por período de entrada:


,con retrasos,sin retrasos
2026-2027,0,0
2027-2028,"4,03","7,61"
2028-2029,"4,34","8,18"
2029-2030,"11,02","17,97"


**Lectura.** Los 41 proyectos con OEF que no operan aportarían 18,0 GWh-día en 2029-2030 si todos llegaran a tiempo, pero solo 11,0 con los retrasos de la subasta de 2019: un 39% menos. En 2027-2028 la diferencia es aún mayor (4,0 frente a 7,6), porque en su primer período solo cuenta el 53% de cada proyecto.

**Las parejas de la homologación en el balance.** La sección 3.1 aceptó parejas entre proyectos con OEF y plantas de XM. La celda siguiente mira en qué estado está cada planta: si ya opera, su energía firme está en el parque con su ENFICC y no puede sumarse otra vez como proyecto.

In [37]:
# Las parejas OEF→XM que aceptó la homologación (sección 3.1): ¿el proyecto ya opera o sigue pendiente?
puntaje = cand["token_set_ratio"].where(~cand["conflicto"])
aceptada = (puntaje == puntaje.groupby([cand["tarea"], cand["id_izq"]]).transform("max")) & (puntaje >= elegido["umbral"])
parejas = (cand.loc[aceptada & (cand["tarea"] == "oef_xm"), ["id_izq", "nombre_izq", "oef_gwh_dia_izq", "id_der", "nombre_der"]]
           .merge(plantas[["codigo", "situacion", "enficc_gwh_dia"]], left_on="id_der", right_on="codigo", how="left",
                  validate="one_to_one").drop(columns="codigo"))
opera = parejas["situacion"].isin(["en operación", "en pruebas"])
parejas["en el balance"] = np.select([opera, parejas["id_der"].isin(nuevos["planta"])],
                                     ["parque, con su ENFICC", "proyecto nuevo, con su OEF"], "sin clasificar")
assert not (opera & parejas["id_der"].isin(nuevos["planta"])).any(), "un proyecto quedaría contado dos veces"
assert (parejas["en el balance"] != "sin clasificar").all()
print(f"{len(parejas)} proyectos con OEF encontrados en XM: {opera.sum()} ya operan, con "
      f"{es(parejas.loc[opera, 'enficc_gwh_dia'].sum(), 2)} GWh-día de ENFICC en el parque y "
      f"{es(parejas.loc[opera, 'oef_gwh_dia_izq'].sum(), 2)} de OEF como proyectos; {(~opera).sum()} siguen como proyectos")
parejas

8 proyectos con OEF encontrados en XM: 4 ya operan, con 3,37 GWh-día de ENFICC en el parque y 1,17 de OEF como proyectos; 4 siguen como proyectos


,id_izq,nombre_izq,oef_gwh_dia_izq,id_der,nombre_der,situacion,enficc_gwh_dia,en el balance
0,PROG00525,ATLÁNTICO PHOTOVOLTAIC S.A.S ESP,"0,554337",ATLP,ATLANTICO,en operación,"1,013954","parque, con su ENFICC"
1,PROG00575,PARQUE SOLAR PUERTA DE ORO S.A.S. E.S.P.,NaN,PORO,PARQUE SOLAR PUERTA DE ORO,en operación,"1,432653","parque, con su ENFICC"
2,PROG00616,OPERADORA SHANGRI-LA S.A.S. E.S.P.,"0,543469",4YCF,SHANGRI LA,en operación,"0,855213","parque, con su ENFICC"
3,PROG01395,PARQUE SOLAR ANDRÓMEDA I SAS ESP,"0,479654",ANDR,ANDROMEDA,"registrada en XM, sin pruebas","0,530257","proyecto nuevo, con su OEF"
4,PROG01472,EMPRESA URRA S.A. E.S.P.,"0,070561",PSUA,PARQUE SOLAR URRA,en pruebas,"0,070561","parque, con su ENFICC"
5,PROG04645,FOTOVOLTAICO TOLIMA NORTE S.A.S E.S.P,"0,097213",TNOR,TOLIMA NORTE,"registrada en XM, sin pruebas","0,102958","proyecto nuevo, con su OEF"
6,PROG04976,Solar Villavieja S.A.S E.S.P,"0,405969",VEJA,VILLAVIEJA,"registrada en XM, sin pruebas","0,3918","proyecto nuevo, con su OEF"
7,PROG08586,TAMARINO COLOMBIA S.A.S. E.S.P.,"0,416426",TNOS,TAMARINO,"registrada en XM, sin pruebas","0,415288","proyecto nuevo, con su OEF"


**Lectura.** La homologación encontró 8 proyectos con OEF en el listado de XM. Cuatro ya operan: Atlántico, Puerta de Oro, Shangri-La y el parque solar de Urrá, con 3,37 GWh-día de ENFICC en el parque. Sin la homologación, su OEF de proyecto (1,17 GWh-día; la de Puerta de Oro no figura en la base) se habría sumado otra vez como proyecto nuevo: unos 430 GWh por período contados dos veces. Los otros cuatro (Andrómeda, Tamarino, Villavieja y Tolima Norte) están registrados en XM sin pruebas y siguen entre los 41 proyectos nuevos, con su código de XM.

Con todas las piezas, `oferta_firme()` arma la oferta de cada período y `balance()` la resta de la demanda de la sección 2.5.

In [38]:
ESCENARIOS = {"base": ("media", True), "estres": ("alta", True), "optimista": ("media", False)}
NOMBRES_CORTOS = {"base": "Base", "estres": "Estrés", "optimista": "Optimista"}
dias = demanda.drop_duplicates("periodo").set_index("periodo")["dias"].reindex(PERIODOS)
entradas = leer("entradas_sin_subasta.csv")
sin_subasta_dia = (entradas["mw_ano"] * entradas["tecnologia"].map(FACTORES_MENORES) * 24 / 1e3).sum()  # por año
retiros_dia = pd.Series([sum(parque.loc[parque["codigo"] == c, "firme_gwh_dia"].sum()
                             for c, fecha in RETIROS.items() if f"{p[:4]}-12-01" >= fecha) for p in PERIODOS], index=PERIODOS)


def oferta_firme(retrasos):
    """Energía firme de cada período (GWh): parque con ENFICC sin retiros, menores, entradas sin subasta y nuevos."""
    base = parque.groupby("fuente")["firme_gwh_dia"].sum()
    return pd.DataFrame({"enficc_gwh": (base["ENFICC"] - retiros_dia) * dias,
                         "menores_gwh": base["menor (factor firme)"] * dias,
                         "sin_subasta_gwh": sin_subasta_dia * np.arange(len(PERIODOS)) * dias,
                         "nuevas_gwh": oferta_nuevos(nuevos, retrasos) * dias})


def balance(escenario):
    """Demanda, oferta firme por componente y brecha de cada período de un escenario."""
    demanda_esc, retrasos = ESCENARIOS[escenario]
    t = oferta_firme(retrasos)
    t["oferta_firme_gwh"] = t.sum(axis=1)
    t["demanda_gwh"] = demanda.query("escenario_demanda == @demanda_esc").set_index("periodo")["demanda_gwh"]
    t["brecha_gwh"] = t["demanda_gwh"] - t["oferta_firme_gwh"]
    t["brecha_pct"] = 100 * t["brecha_gwh"] / t["demanda_gwh"]
    return t.assign(escenario=escenario, dias=dias).rename_axis("periodo").reset_index()

In [39]:
balances = pd.concat([balance(e) for e in ESCENARIOS], ignore_index=True)
pipeline = leer("pipeline_brechas_escenarios.csv").query("definicion == @DEFINICION")
comprobar(balances.set_index(["escenario", "periodo"])["brecha_gwh"],
          pipeline.set_index(["escenario", "periodo"])["brecha_firme_gwh"], "Brecha firme (GWh), 3 escenarios × 4 períodos")
(balances.pivot(index="periodo", columns="escenario", values=["brecha_gwh", "brecha_pct"]).round(1)
         .rename(columns={"brecha_gwh": "brecha (GWh)", "brecha_pct": "brecha (% de la demanda)", **NOMBRES_CORTOS}))

✓ Brecha firme (GWh), 3 escenarios × 4 períodos: coincide con el pipeline (diferencia máxima 1,8e-06)


brecha (GWh)                   brecha (% de la demanda)                 
escenario         Base  Estrés Optimista                     Base Estrés Optimista
periodo                                                                           
2026-2027        1.770 4.493,1     1.770                      1,9    4,7       1,9
2027-2028      2.808,6 6.531,8   1.499,3                      2,9    6,5       1,6
2028-2029      5.167,8 9.740,3   3.764,1                      5,2    9,4       3,8
2029-2030      4.584,9 9.933,4   2.048,8                      4,6    9,4         2

**Lectura.** La brecha coincide con la del pipeline en los doce casos; la diferencia máxima, menos de dos millonésimas de GWh, es redondeo.
- **Base (demanda media, con retrasos):** falta energía firme desde 2026-2027: 1.770 GWh (1,9% de la demanda), sin proyectos nuevos ese período. El máximo llega en 2028-2029: 5.168 GWh (5,2%), porque solo dos proyectos con OEF tienen su compromiso en ese período (sección 4.3). En 2029-2030 quedan 4.585 GWh (4,6%).
- **Optimista (sin retrasos):** baja a 1.499 GWh en 2027-2028 y a 2.049 en 2029-2030, pero no desaparece en ningún período.
- **Estrés (demanda alta, con retrasos):** falta desde el primer período y crece hasta 9.933 GWh en 2029-2030 (9,4%).

**Un año seco y un año normal observados.** La ENFICC es la energía que el parque garantiza con la hidrología crítica de la serie histórica, así que el balance de arriba aplica el criterio del Cargo por Confiabilidad, más severo que un año seco cualquiera. Los profesores nos recomendaron calcular un escenario de El Niño y uno normal, y los construimos con la misma vara: para cada período del cargo desde 2010-2011 (la generación por planta empieza en 2010) medimos cuánto generaron las hidráulicas con ENFICC frente a su propia ENFICC, y clasificamos el período por sus aportes con los umbrales de la sección 2.5: seco por debajo del 80% de la media histórica, normal entre 80% y 110% y húmedo por encima. Los tres períodos secos (2015-2016, 2019-2020 y 2023-2024) empezaron con El Niño en curso (columna `meses_nino`, con las fases de la sección 2.3): el año seco observado es El Niño medido con datos, y la ENFICC, El Niño que exige la regulación.

En el año seco observado, la hidráulica con ENFICC aporta su ENFICC multiplicada por la mediana de los períodos secos, y en el año normal, por la de los normales; lo demás no cambia. El año normal subestima la holgura: en un año de lluvias normales, lo que limita la generación hidráulica es la demanda y no el agua, de modo que el agua que sobra no aparece en la generación.

In [40]:
hidro = leer("xm_hidraulica_periodos.csv")
hidro["aportes_pct"] = 100 * hidro["aportes_gwh"] / hidro["aportes_media_hist_gwh"]
hidro["condicion"] = pd.cut(hidro["aportes_pct"], [-np.inf, 80, 110, np.inf], labels=["Seca", "Normal", "Húmeda"])
hidro["generacion_sobre_enficc"] = hidro["generacion_hidraulica_gwh"] / hidro["dias"] / hidro["enficc_hidraulica_gwh_dia"]
# Meses de cada período con El Niño, según las fases del ONI de la sección 2.3
hidro["meses_nino"] = (hidro["periodo"].map(oni[oni["fase"] == "El Niño"].groupby(periodo_cxc(oni["mes"])).size())
                       .fillna(0).astype(int))
FACTOR_ANO_SECO = hidro.loc[hidro["condicion"] == "Seca", "generacion_sobre_enficc"].median()
FACTOR_ANO_NORMAL = hidro.loc[hidro["condicion"] == "Normal", "generacion_sobre_enficc"].median()
print(f"{len(hidro)} períodos completos desde 2010-2011; la hidráulica generó {es(FACTOR_ANO_SECO, 2)} veces su ENFICC "
      f"en los secos y {es(FACTOR_ANO_NORMAL, 2)} en los normales (medianas)")
# Qué tanto depende la mediana de los normales de un solo período: se quita uno a la vez
normales = hidro[hidro["condicion"] == "Normal"].set_index("periodo")["generacion_sobre_enficc"]
sin_uno = [normales.drop(p).median() for p in normales.index]
print(f"Mediana de los normales si se deja un período por fuera: de {es(min(sin_uno), 2)} a {es(max(sin_uno), 2)}")
display(hidro.set_index("periodo")[["aportes_pct", "condicion", "meses_nino", "generacion_sobre_enficc"]].round(2))
hidro.groupby("condicion", observed=True)["generacion_sobre_enficc"].agg(["count", "min", "median", "max"]).round(2)

15 períodos completos desde 2010-2011; la hidráulica generó 1,05 veces su ENFICC en los secos y 1,22 en los normales (medianas)
Mediana de los normales si se deja un período por fuera: de 1,22 a 1,26


,aportes_pct,condicion,meses_nino,generacion_sobre_enficc
periodo,,,,
2010-2011,"133,18",Húmeda,0,"1,48"
2011-2012,"111,83",Húmeda,0,"1,39"
2012-2013,"88,38",Normal,0,"1,22"
2013-2014,"93,19",Normal,2,"1,22"
2014-2015,"81,94",Normal,12,"1,08"
2015-2016,"79,17",Seca,6,"1,05"
2016-2017,"98,56",Normal,0,"1,32"
2017-2018,"103,94",Normal,3,"1,3"
2018-2019,"86,75",Normal,9,"1,06"


,count,min,median,max
condicion,,,,
Seca,3,"1,04","1,05","1,2"
Normal,7,"1,06","1,22","1,35"
Húmeda,5,"1,28","1,4","1,48"


In [41]:
ENFICC_HIDRAULICA = parque.loc[(parque["fuente"] == "ENFICC") & (parque["tecnologia"] == "hidraulica"), "firme_gwh_dia"].sum()


def balance_hidrologia(escenario, factor):
    """Brecha (GWh por período) si la hidráulica con ENFICC aporta `factor` veces su ENFICC y lo demás queda como en el
    balance firme."""
    t = balance(escenario).set_index("periodo")
    return t["brecha_gwh"] - ENFICC_HIDRAULICA * (factor - 1) * t["dias"]


print(f"ENFICC hidráulica del parque: {es(ENFICC_HIDRAULICA, 1)} GWh-día; aporta {es(ENFICC_HIDRAULICA * (FACTOR_ANO_SECO - 1), 1)} "
      f"GWh-día más en el año seco observado y {es(ENFICC_HIDRAULICA * (FACTOR_ANO_NORMAL - 1), 1)} en el año normal")
# Con qué factor la brecha del estrés sería cero: brecha = ENFICC hidráulica × (factor − 1) × días
equilibrio = 1 + balance("estres").set_index("periodo")["brecha_gwh"] / (ENFICC_HIDRAULICA * dias)
print(f"La brecha del estrés es cero con un factor de {es(equilibrio[PERIODOS[2]], 2)} en {PERIODOS[2]} y de "
      f"{es(equilibrio[PERIODOS[3]], 2)} en {PERIODOS[3]}")
HIDROLOGIAS = {"Hidrología crítica (ENFICC)": 1.0, "Año seco observado": FACTOR_ANO_SECO, "Año normal": FACTOR_ANO_NORMAL}
hidrologias = pd.concat({nombre: pd.DataFrame({NOMBRES_CORTOS[e]: balance_hidrologia(e, f) for e in ["base", "estres"]})
                         for nombre, f in HIDROLOGIAS.items()}, axis=1)
hidrologias.round(0)

ENFICC hidráulica del parque: 113,4 GWh-día; aporta 6,0 GWh-día más en el año seco observado y 25,4 en el año normal
La brecha del estrés es cero con un factor de 1,24 en 2028-2029 y de 1,24 en 2029-2030


Hidrología crítica (ENFICC)        Año seco observado        Año normal       
                                 Base Estrés               Base Estrés       Base Estrés
periodo                                                                                 
2026-2027                       1.770  4.493               -428  2.295     -7.518 -4.795
2027-2028                       2.809  6.532                605  4.328     -6.505 -2.782
2028-2029                       5.168  9.740              2.970  7.543     -4.120    452
2029-2030                       4.585  9.933              2.387  7.736     -4.703    645

**Lectura.** La hidráulica generó más que su ENFICC en todos los períodos observados: en los tres secos, entre 1,04 y 1,20 veces (mediana de 1,05); en los siete normales, entre 1,06 y 1,35 (mediana de 1,22), y en los húmedos, 1,40. La ENFICC resulta más severa que los años secos de 2010 a 2025. En un año seco como esos, las hidráulicas del parque (113,4 GWh-día de ENFICC) aportan 6,0 GWh-día más: unos 2.200 GWh por período, más que toda la brecha base de 2026-2027.

Cuánto falta, y desde cuándo, depende de la hidrología:
- **Hidrología crítica (ENFICC):** en el base falta desde 2026-2027 (1.770 GWh) y llega a 5.168 en 2028-2029.
- **Año seco observado:** en el base sobran 428 GWh en 2026-2027; falta desde 2027-2028 (605) y llega a 2.970 en 2028-2029. Con la demanda alta falta en todos los períodos, de 2.295 a 7.736 GWh.
- **Año normal:** en el base no falta en ningún período: sobran entre 4.120 y 7.518 GWh. Con la demanda alta sobra hasta 2027-2028, y faltan 452 GWh en 2028-2029 y 645 en 2029-2030, menos del 1% de la demanda.

Dimensionamos la expansión con la ENFICC porque es la regla del Cargo por Confiabilidad y porque no podemos descartar un año más seco que los observados desde 2010. Las dos medianas son frágiles. Con solo tres períodos secos, la de 1,05 depende de cuáles sean: 2023-2024 llegó a 1,20. La de los normales va de 1,22 a 1,26 según el período que se deje por fuera, y la brecha del estrés de 2028-2030 es cero con 1,24: con la demanda alta, el año normal queda en el margen. Además, la clasificación es por aportes y no por El Niño: 2014-2015 y 2018-2019 tuvieron El Niño casi todo el período y quedan entre los normales, con la menor generación frente a su ENFICC (1,08 y 1,06).

### 3.4 Expansión de mínimo costo (MILP)

Con la brecha de la sección 3.3 y los bloques de la 3.2, un modelo de optimización con variables enteras escoge qué construir, cuándo y a qué costo.

Un **MILP** (*mixed-integer linear program*, programa lineal entero mixto) es un problema de optimización con objetivo y restricciones lineales en el que algunas variables solo pueden tomar valores enteros. Aquí las enteras son los bloques de cada tecnología que entran en cada período (no se construye media planta), y la continua es la energía que ningún bloque alcanza a cubrir a tiempo.

$$\min_{y,\,u}\; \sum_{t} \delta_t \Big[ \sum_{b} C_b\, x_{b,t} + \pi\, u_t \Big] \quad \text{sujeto a} \quad
\begin{aligned}
x_{b,t} &= \textstyle\sum_{\tau \le t} y_{b,\tau} && \text{bloques en operación}\\
\textstyle\sum_b F_{b,t}\, x_{b,t} + u_t &\ge G_t && \text{brecha firme (sección 3.3)}\\
y_{b,t} &= 0 \ \text{ si } t < t^{\min}_b && \text{primer período factible}\\
\textstyle\sum_{b \in g} \text{MW}_b\, x_{b,T} &\le \text{cupo}_g && \text{cupo por grupo}\\
\text{MW}_s\, y_{s,t} &\le \text{tope}_t && \text{tope de construcción solar}\\
y_{b,t} \in \mathbb{Z}_{\ge 0},\ u_t &\ge 0
\end{aligned}$$

| Símbolo | Qué es | De dónde sale |
|---|---|---|
| $y_{b,t}$ | Bloques de la tecnología $b$ que entran en el período $t$ (entero) | Variable |
| $u_t$ | Energía que ningún bloque cubre (GWh) | Variable |
| $C_b$ | Costo anual del bloque (MUSD) | Sección 3.2 |
| $F_{b,t}$ | Energía firme del bloque en el período: MW × horas × factor firme | Sección 3.2 |
| $G_t$ | Brecha firme | Sección 3.3 |
| $\delta_t$ | Descuento $(1+r)^{-t}$, con $t = 0$ en 2026-2027 y $r$ = 10% | PIEG |
| $\pi$ | Penalidad de la energía no cubierta: el costo de racionamiento de la UPME, primer tramo | UPME (octubre de 2026) |
| $\text{tope}_t$ | 1.400 MW de solar por período, el récord de entradas solares en un año, menos los MW solares con OEF que se espera que entren en $t$ | XM (`fuentes_supuestos.csv`); sección 3.3 |

Dos detalles de implementación. Primero, $x$ no es una variable: la reemplazamos por la suma acumulada de $y$, así que quedan 24 variables enteras ($6$ tecnologías × $4$ períodos) y 4 continuas. Segundo, como los costos son anuales, cada bloque paga una anualidad por cada período que opera dentro del horizonte, así que uno que entra en 2029-2030 paga solo una. Lo que cuesta y lo que aporta después de 2030 queda por fuera, y es una limitación (sección 4.9).

**La penalidad.** El costo de racionamiento de la UPME viene en pesos por kWh. Se lleva a dólares con la TRM del mes de referencia y a USD de 2025 con el CPI de EE. UU.:

In [42]:
meta = json.loads((CARPETA / "metadatos.json").read_text(encoding="utf-8"))
m = meta["moneda"]
PENALIDADES = {k: v["cop_kwh"] * 1000 / m["trm"] * m["factor_cpi"] for k, v in meta["penalidades"].items()}
for criterio, usd in PENALIDADES.items():
    print(f"{criterio}: {es(meta['penalidades'][criterio]['cop_kwh'], 1)} COP/kWh → {es(usd)} USD de 2025/MWh")
TOPE_SOLAR = supuesto("solar_fv", "tope_anual")   # el récord de entradas solares en un año (XM)
solares = leer("xm_solar_entradas_anuales.csv").set_index("anio")
print(f"Récord de entradas solares: {es(solares['mw_mas_de_1mw'].max())} MW en {solares['mw_mas_de_1mw'].idxmax()} "
      f"(plantas de más de 1 MW) y {es(solares['mw_todas'].max())} con todas; tope de {es(TOPE_SOLAR)} MW por período")
SOLAR_COMPROMETIDA = {r: solar_comprometida(nuevos, r) for r in (True, False)}
pd.DataFrame({"solar con OEF, con retrasos (MW)": SOLAR_COMPROMETIDA[True],
              "lo que el tope deja, con retrasos (MW)": (TOPE_SOLAR - SOLAR_COMPROMETIDA[True]).clip(lower=0)}).round(0)

costo_racionamiento_upme: 2.572,5 COP/kWh → 770 USD de 2025/MWh
precio_escasez_inferior: 327,2 COP/kWh → 98 USD de 2025/MWh
Récord de entradas solares: 1.365 MW en 2024 (plantas de más de 1 MW) y 1.418 con todas; tope de 1.400 MW por período


,"solar con OEF, con retrasos (MW)","lo que el tope deja, con retrasos (MW)"
2026-2027,0,1.400
2027-2028,1.486,0
2028-2029,238,1.162
2029-2030,1.295,105


**Lectura.** La penalidad queda en 770 USD de 2025 por MWh, lo que cuesta racionar el primer 1,5% de la demanda: el tramo más barato del costo de racionamiento de la UPME. El precio de escasez inferior (98 USD/MWh), la referencia regulatoria más baja, queda como sensibilidad (sección 3.5). El tope solar de 1.400 MW por período está entre los dos récords de entradas solares en un año, ambos de 2024 (1.365 MW en plantas de más de 1 MW y 1.418 con todas). Ese tope no deja espacio en 2027-2028, porque los proyectos solares con OEF que esperamos ese período ya suman unos 1.486 MW con retrasos; en 2028-2029 quedan libres unos 1.162 MW, y en 2029-2030, apenas 105.

El modelo se arma en dos funciones: `restricciones()` escribe cada fila de la matriz $A$ con sus límites, y `armar_milp()` agrega el vector de costos, las cotas y la integralidad.

In [43]:
def restricciones(bl, F, mw, brecha, tope_solar):
    """Filas de A con sus límites lb y ub: brecha firme de cada período, cupo de cada grupo y tope solar."""
    nb, nt = F.shape
    n = nb * nt + nt                                  # variables: y[b, t] en la posición b·nt + t, y después u[t]
    A, lb, ub, filas = [], [], [], []
    for t in range(nt):                               # energía firme nueva + energía no cubierta ≥ brecha
        a = np.zeros(n)
        for k in range(nb):
            a[k * nt: k * nt + t + 1] = F[k, t]       # un bloque que entró en τ ≤ t está operando en t
        a[nb * nt + t] = 1.0
        A.append(a), lb.append(brecha[t]), ub.append(np.inf), filas.append(f"firme {PERIODOS[t]}")
    for grupo, sub in bl.groupby("grupo_cupo", sort=False):   # MW totales del grupo ≤ cupo
        a = np.zeros(n)
        for k in sub.index:
            a[k * nt:(k + 1) * nt] = mw[k]
        A.append(a), lb.append(-np.inf), ub.append(float(sub["cupo_mw"].iloc[0])), filas.append(f"cupo {grupo}")
    k = int(np.flatnonzero(bl["tecnologia"] == "solar_fv")[0])
    for t in range(nt):                               # MW solares que entran en t ≤ lo que deja el tope
        a = np.zeros(n)
        a[k * nt + t] = mw[k]
        A.append(a), lb.append(-np.inf), ub.append(float(tope_solar.iloc[t])), filas.append(f"tope solar {PERIODOS[t]}")
    return np.array(A), np.array(lb), np.array(ub), filas

In [44]:
def armar_milp(bl, brecha, dias, r, pi, tope_solar):
    """Arma el MILP: costos c, restricciones lb ≤ A·v ≤ ub, cotas e integralidad, con v = (y[b, t], u[t]).

    bl: bloques de la sección 3.2. brecha y dias: por período. r: tasa de descuento. pi: penalidad (USD/MWh).
    tope_solar: MW solares nuevos permitidos en cada período.
    """
    nb, nt = len(bl), len(PERIODOS)
    delta = (1 + r) ** -np.arange(nt, dtype=float)
    mw = bl["tamano_mw"].to_numpy(float)
    F = np.outer(mw * bl["factor_firme"].to_numpy(float), np.asarray(dias, float) * 24) / 1e3   # GWh firmes
    C = bl["costo_anual_musd"].to_numpy(float)
    c = np.r_[(C[:, None] * (np.triu(np.ones((nt, nt))) @ delta)[None, :]).ravel(), delta * pi / 1e3]
    A, lb, ub, filas = restricciones(bl, F, mw, np.asarray(brecha, float), tope_solar)
    alto = np.r_[np.repeat(np.floor(bl["cupo_mw"].to_numpy() / mw), nt), np.full(nt, np.inf)]
    alto[:nb * nt] *= (np.arange(nt)[None, :] >= bl["k_min"].to_numpy()[:, None]).ravel()   # antes de t_min: 0
    return dict(c=c, A=A, lb=lb, ub=ub, cotas=Bounds(0, alto), enteras=np.r_[np.ones(nb * nt), np.zeros(nt)],
                F=F, C=C, delta=delta, filas=filas, pi=pi)

`resolver()` llama a `scipy.optimize.milp`, que usa el solver HiGHS.

In [45]:
def resolver(pr, relajado=False):
    """Resuelve el MILP con scipy.optimize.milp (HiGHS); con `relajado`, sin integralidad. Se detiene si falla."""
    res = milp(pr["c"], constraints=LinearConstraint(pr["A"], pr["lb"], pr["ub"]), bounds=pr["cotas"],
               integrality=np.zeros_like(pr["enteras"]) if relajado else pr["enteras"],
               options={"mip_rel_gap": 1e-9})
    if not res.success:
        raise RuntimeError(f"El MILP no se resolvió: {res.message}")
    return res


def brecha_de(escenario):
    """Brecha firme (GWh) y días de cada período de un escenario, de la sección 3.3."""
    return balances.query("escenario == @escenario").set_index("periodo")[["brecha_gwh", "dias"]].reindex(PERIODOS)

**Un caso paso a paso: el estrés** (demanda alta, con retrasos), el escenario más exigente. Primero se arma el problema y se mira la matriz: cada fila es una restricción y cada columna una variable. Se muestran las columnas del ciclo abierto, la solar y la energía no cubierta; las demás tecnologías tienen la misma estructura.

In [46]:
bl = bloques()
b = brecha_de("estres")
pr = armar_milp(bl, b["brecha_gwh"], b["dias"], TASA, PENALIDADES["costo_racionamiento_upme"],
                (TOPE_SOLAR - SOLAR_COMPROMETIDA[True]).clip(lower=0))
corto = [p[2:4] + "-" + p[7:] for p in PERIODOS]                       # "2026-2027" → "26-27"
columnas = [f"y {t} {p}" for t in bl["tecnologia"] for p in corto] + [f"u {p}" for p in corto]
print(f"{pr['A'].shape[0]} restricciones y {pr['A'].shape[1]} variables ({int(pr['enteras'].sum())} enteras)")
pd.DataFrame(pr["A"], index=pr["filas"], columns=columnas).filter(regex=r"^(y gas_ciclo_abierto|y solar_fv|u) ").round(0)

13 restricciones y 28 variables (24 enteras)


,y gas_ciclo_abierto 26-27,y gas_ciclo_abierto 27-28,y gas_ciclo_abierto 28-29,y gas_ciclo_abierto 29-30,y solar_fv 26-27,y solar_fv 27-28,y solar_fv 28-29,y solar_fv 29-30,u 26-27,u 27-28,u 28-29,u 29-30
firme 2026-2027,788,0,0,0,114,0,0,0,1,0,0,0
firme 2027-2028,791,791,0,0,114,114,0,0,0,1,0,0
firme 2028-2029,788,788,788,0,114,114,114,0,0,0,1,0
firme 2029-2030,788,788,788,788,114,114,114,114,0,0,0,1
cupo biomasa_bagazo,0,0,0,0,0,0,0,0,0,0,0,0
cupo diesel_motores,0,0,0,0,0,0,0,0,0,0,0,0
cupo eolica_terrestre,0,0,0,0,0,0,0,0,0,0,0,0
cupo gas,100,100,100,100,0,0,0,0,0,0,0,0
cupo solar_fv,0,0,0,0,100,100,100,100,0,0,0,0
tope solar 2026-2027,0,0,0,0,100,0,0,0,0,0,0,0


In [47]:
res = resolver(pr)
nb, nt = len(bl), len(PERIODOS)
y = np.round(res.x[:nb * nt]).reshape(nb, nt)      # bloques que entran en cada período
x, u = y.cumsum(axis=1), res.x[nb * nt:]            # bloques en operación y energía no cubierta
print(f"HiGHS: {'solución óptima' if res.status == 0 else res.message}. Costo en valor presente: {es(res.fun, 1)} MUSD de 2025")
mw = pd.DataFrame(x * bl["tamano_mw"].to_numpy()[:, None], index=bl["tecnologia"], columns=PERIODOS)
display(mw[mw.sum(axis=1) > 0].rename(index=NOMBRES_TECNOLOGIA).rename_axis("MW en operación"))
pd.DataFrame({"brecha firme (GWh)": b["brecha_gwh"], "energía firme nueva (GWh)": (x * pr["F"]).sum(axis=0),
              "energía no cubierta (GWh)": u}).round(0)

HiGHS: solución óptima. Costo en valor presente: 8.841,5 MUSD de 2025


,2026-2027,2027-2028,2028-2029,2029-2030
MW en operación,,,,
Bagazo,0,0,40,40
Eólica,0,0,0,100
"Gas, ciclo abierto",0,0,1.200,1.200
Solar FV,0,0,100,200


,brecha firme (GWh),energía firme nueva (GWh),energía no cubierta (GWh)
periodo,,,
2026-2027,4.493,0,4.493
2027-2028,6.532,0,6.532
2028-2029,9.740,9.764,0
2029-2030,9.933,10.000,0


**Lectura.** 
- **2026-2027 y 2027-2028:** no llega ninguna tecnología, así que la brecha queda sin cubrir: 4.493 y 6.532 GWh. En 2027-2028 solo podría entrar la solar, pero los proyectos con OEF ya llenan el tope, y el diésel no tiene cupo.
- **2028-2029:** entran 12 bloques de ciclo abierto (1.200 MW), 40 MW de bagazo, que llega a su cupo, y 100 MW solares en el espacio que deja el tope: 9.764 GWh firmes para una brecha de 9.740.
- **2029-2030:** la brecha sube a 9.933 y lo construido cubre 9.764: faltan 169 GWh. Entran 100 MW eólicos y 100 solares; otro bloque de ciclo abierto traería 788 GWh, casi cinco veces lo que falta, y costaría más.
- **Costo:** 8.842 MUSD en valor presente, casi todo penalidad de la energía que no alcanza a cubrirse a tiempo. Los bloques cuestan 810.

**Los tres escenarios.** `correr()` repite lo anterior para cualquier escenario y caso de sensibilidad, y resume la solución: el costo de los bloques en valor presente, el costo total con la penalidad, la energía no cubierta y los MW de cada tecnología al final del horizonte. También resuelve la relajación lineal, para validar.

In [48]:
CASOS = {"Central": {}}   # la sección 3.5 agrega los casos de sensibilidad


def correr(escenario, caso="Central", **cambios):
    """Resuelve el MILP de un escenario con los cambios de un caso de sensibilidad y resume la solución.

    Devuelve (resumen, detalle): el resumen tiene los costos en valor presente (MUSD de 2025), la energía
    no cubierta (GWh) y los MW de cada tecnología en 2029-2030; el detalle, el problema y la solución.
    """
    cambios = CASOS.get(caso, {}) | cambios
    bl = bloques(cambios.get("escenario_costos", "central"), cupos=cambios.get("cupo"),
                 retraso_periodos=cambios.get("retraso_periodos", 0))
    pi = PENALIDADES[cambios.get("criterio_penalidad", "costo_racionamiento_upme")]
    b = brecha_de(escenario)
    tope = (cambios.get("tope_solar", TOPE_SOLAR) - SOLAR_COMPROMETIDA[ESCENARIOS[escenario][1]]).clip(lower=0)
    pr = armar_milp(bl, b["brecha_gwh"] + cambios.get("brecha_extra", 0), b["dias"], TASA, pi, tope)
    res, lp = resolver(pr), resolver(pr, relajado=True)
    nb, nt = len(bl), len(PERIODOS)
    x = np.round(res.x[:nb * nt]).reshape(nb, nt).cumsum(axis=1)
    u = res.x[nb * nt:]
    resumen = {"escenario": escenario, "caso": caso, "costo_bloques_vp_musd": float(pr["delta"] @ (pr["C"] @ x)),
               "costo_total_vp_musd": res.fun, "objetivo_lp_musd": lp.fun, "no_cubierta_gwh": u.sum()}
    resumen |= {f"mw_{t}": x[k, -1] * bl.at[k, "tamano_mw"] for k, t in enumerate(bl["tecnologia"])}
    return resumen, {"pr": pr, "x": x, "u": u, "bl": bl}

In [49]:
corridas = {e: correr(e) for e in ESCENARIOS}
central = pd.DataFrame([resumen for resumen, _ in corridas.values()]).set_index("escenario")
pipeline_exp = leer("pipeline_expansion_resumen.csv").query("definicion == @DEFINICION").set_index(["escenario", "caso"])
cols = ["costo_bloques_vp_musd", "costo_total_vp_musd", "no_cubierta_gwh"] + [c for c in central if c.startswith("mw_")]
comprobar(central[cols], pipeline_exp.xs("Central", level="caso")[cols], "MILP, caso central de los 3 escenarios")
brecha_lp = 100 * (central["costo_total_vp_musd"] - central["objetivo_lp_musd"]) / central["costo_total_vp_musd"]
print("Relajación lineal: su costo es menor que el del MILP en",
      ", ".join(f"{NOMBRES_CORTOS[e].lower()}, {es(v, 1)}%" for e, v in brecha_lp.items()))
COLUMNAS_RESUMEN = {"costo_bloques_vp_musd": "bloques (MUSD, VP)", "costo_total_vp_musd": "total (MUSD, VP)",
                    "no_cubierta_gwh": "sin cubrir (GWh)"} | {f"mw_{k}": f"{v} (MW)" for k, v in NOMBRES_TECNOLOGIA.items()}
central[cols].round(1).rename(columns=COLUMNAS_RESUMEN, index=NOMBRES_CORTOS)

✓ MILP, caso central de los 3 escenarios: coincide con el pipeline (diferencia máxima 2,9e-06)
Relajación lineal: su costo es menor que el del MILP en base, 0,7%, estrés, 0,3%, optimista, 0,9%


,"bloques (MUSD, VP)","total (MUSD, VP)",sin cubrir (GWh),Bagazo (MW),Motores diésel (MW),Eólica (MW),"Gas, ciclo abierto (MW)","Gas, ciclo combinado (MW)",Solar FV (MW)
escenario,,,,,,,,,
Base,"422,9","3.764,7",4.599,40,0,0,600,0,200
Estrés,"809,9","8.841,5","11.024,9",40,0,100,1.200,0,200
Optimista,"318,8","2.731,1","3.269,3",0,0,0,500,0,0


**Lectura.** Los tres escenarios coinciden con el pipeline.
- **Base:** 600 MW de ciclo abierto, 200 MW solares y 40 MW de bagazo, desde 2028-2029. Los bloques cuestan 423 MUSD en valor presente. Quedan sin cubrir 4.599 GWh, casi todos en 2026-2028, cuando nada alcanza a llegar.
- **Estrés:** 1.200 MW de ciclo abierto, 200 solares, 100 eólicos y 40 de bagazo, por 810 MUSD; 11.025 GWh sin cubrir.
- **Optimista:** 500 MW de ciclo abierto, por 319 MUSD; 3.269 GWh sin cubrir.
- **Validación:** la relajación lineal cuesta entre 0,3% y 0,9% menos que el MILP, como debe ser: nunca puede costar más. La diferencia es pequeña porque las brechas son grandes frente al tamaño de un bloque.

### 3.5 Sensibilidades

Cada caso cambia un supuesto de las secciones 3.2 a 3.4 y vuelve a resolver los tres escenarios: así se ve cuál mueve la respuesta.

Además del caso central, corremos siete casos de sensibilidad (ocho en total, 24 corridas con los tres escenarios): los cinco que mueven la energía sin cubrir (plazos un período más largos, los dos topes solares del PIEG, un cupo de diésel de 200 MW y la penalidad más baja) y los costos bajos y altos de las tecnologías. Sus valores y fuentes están en `fuentes_supuestos.csv`.

In [50]:
CASOS.update({   # los que mueven la energía sin cubrir y los costos bajos y altos; el pipeline corre 16
    "Costos bajos": {"escenario_costos": "bajo"}, "Costos altos": {"escenario_costos": "alto"},
    "Plazos un período más largos": {"retraso_periodos": 1},
    "Tope solar del PIEG, referencia (2.315 MW/año)": {"tope_solar": 2315},
    "Tope solar del PIEG, máximo (4.325 MW/año)": {"tope_solar": 4325},
    "Diésel con cupo de 200 MW (supuesto)": {"cupo": {"diesel_motores": 200}},
    "Penalidad: precio de escasez inferior": {"criterio_penalidad": "precio_escasez_inferior"}})

sens = pd.DataFrame([correr(e, caso)[0] for e in ESCENARIOS for caso in CASOS]).set_index(["escenario", "caso"])
comprobar(sens[cols], pipeline_exp.loc[sens.index, cols], f"MILP, {len(sens)} corridas (3 escenarios × {len(CASOS)} casos)")
mw_costos = sens.filter(like="mw_").sum(axis=1).unstack("caso")[["Costos bajos", "Central", "Costos altos"]]
print("MW nuevos con costos bajos, centrales y altos:",
      "; ".join(f"{NOMBRES_CORTOS[e].lower()}, {es(b)}, {es(c)} y {es(a)}" for e, (b, c, a) in mw_costos.iterrows()))
sens.xs("estres", level="escenario")[["costo_bloques_vp_musd", "costo_total_vp_musd", "no_cubierta_gwh"]].round(0).rename(
    columns=COLUMNAS_RESUMEN)

✓ MILP, 24 corridas (3 escenarios × 8 casos): coincide con el pipeline (diferencia máxima 3,5e-06)
MW nuevos con costos bajos, centrales y altos: base, 840, 840 y 840; estrés, 1.540, 1.540 y 1.540; optimista, 500, 500 y 840


,"bloques (MUSD, VP)","total (MUSD, VP)",sin cubrir (GWh)
caso,,,
Central,810,8.842,11.025
Costos bajos,765,8.797,11.025
Costos altos,1.245,9.277,11.025
Plazos un período más largos,524,13.956,19.513
"Tope solar del PIEG, referencia (2.315 MW/año)",939,8.332,10.111
"Tope solar del PIEG, máximo (4.325 MW/año)",1.304,7.098,7.828
Diésel con cupo de 200 MW (supuesto),866,7.815,9.479
Penalidad: precio de escasez inferior,779,1.821,11.285


**Lectura.** Las 24 corridas coinciden con el pipeline. En el estrés:
- **Los plazos son lo que más pesa.** Si cada tecnología llega un período después, quedan 19.513 GWh sin cubrir y el costo total sube 58%, a 13.956 MUSD.
- **Antes de 2028 solo hay dos palancas.** Un cupo de diésel de 200 MW, un caso hipotético porque no hay proyectos en curso, deja 9.479 GWh sin cubrir; con el tope solar más alto del PIEG (4.325 MW/año) quedan 7.828.
- **Los costos** mueven el costo de los bloques entre 765 y 1.245 MUSD, sin cambiar los MW ni la energía sin cubrir. En el base tampoco cambian los MW; en el optimista, con costos altos, pasan de 500 a 840.
- **La penalidad** cambia sobre todo el costo total: con el precio de escasez inferior, el portafolio casi no cambia (salen la eólica y un bloque solar, y quedan 260 GWh más sin cubrir).

Los otros nueve casos que corre el pipeline (la tasa de descuento de 8% y de 12%, los factores firmes observados en XM, sin eólica, sin bagazo o con 70 MW, el precio de escasez superior, la penalidad multiplicada por 10 y el solo costo fijo) cambian la energía sin cubrir en 20 GWh o menos en los tres escenarios, y por eso salieron del tutorial.

**Lo que sigue.** Con esto ya tenemos las cifras del método: la brecha de cada escenario y período, con la hidrología crítica y con un año seco y uno normal observados, el portafolio de mínimo costo y su sensibilidad a cada supuesto. En la sección 4 las mostramos en figuras y respondemos la pregunta.

<a id="resultados" name="resultados"></a>
## 4. Resultados y visualización

Aquí mostramos lo que calculamos en la sección 3 en siete figuras interactivas, de la demanda a la expansión, y cerramos con la respuesta a la pregunta y sus limitaciones. Al pasar el cursor sobre una figura se ven sus valores.

In [51]:
# Nombres y colores que comparten las figuras de esta sección
NOMBRES_ESCENARIO = {"base": "Base: demanda media, con retrasos", "estres": "Estrés: demanda alta, con retrasos",
                     "optimista": "Optimista: demanda media, sin retrasos"}
COLORES_ESCENARIO = {"base": "#2a78d6", "estres": "#eb6834", "optimista": "#1baf7a"}
COLORES_TECNOLOGIA = {"Solar FV": "#eda100", "Eólica": "#1baf7a", "Bagazo": "#008300", "Gas, ciclo abierto": "#eb6834",
                      "Gas, ciclo combinado": "#6250d6", "Motores diésel": "#e87ba4"}
LEYENDA = dict(orientation="h", yanchor="top", y=-0.22, x=0, title=None)

### 4.1 La demanda: observada y proyectada

La demanda observada de XM desde 2018 y la proyección de la UPME para la demanda total, con su banda de 68%. El borde superior de la banda es la demanda del escenario de estrés.

In [52]:
obs = sistema[sistema["mes_completo"] & (sistema["mes"] >= "2018-01-01")]
upme = (demanda_upme.query("definicion == @DEFINICION and mes <= '2030-12-01'")
                    .pivot(index="mes", columns="escenario", values="energia_gwh"))
fig = go.Figure()
fig.add_trace(go.Scatter(x=upme.index, y=upme["ic68_sup"], line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=upme.index, y=upme["ic68_inf"], fill="tonexty", fillcolor="rgba(42,120,214,0.18)",
                         line=dict(width=0), name="UPME, banda de 68%"))
fig.add_trace(go.Scatter(x=upme.index, y=upme["medio"], name="UPME, escenario medio", line=dict(color="#2a78d6", width=2)))
fig.add_trace(go.Scatter(x=obs["mes"], y=obs["demanda_sin_gwh"], name="Observada (XM, DemaSIN)",
                         line=dict(color="#1f1f1f", width=2)))
fig.add_vrect(x0="2026-12-01", x1="2030-11-30", fillcolor="#e9e7e2", opacity=0.5, layer="below", line_width=0,
              annotation_text="Horizonte: dic-2026 a nov-2030", annotation_position="top left")
fig.update_layout(title="Demanda mensual de energía del SIN: observada y proyectada (demanda total de la UPME)",
                  xaxis_title="Mes", yaxis_title="GWh por mes", height=440, legend=LEYENDA, hovermode="x unified")
fig.show()

**Lectura.** La demanda observada creció de 69.127 GWh en 2018 a 84.049 en 2025, un 22% (2,8% al año), con una sola caída: la de 2020, por la pandemia (−2,1% frente a 2019). La proyección media de la demanda total sube más rápido: 100.946 GWh en 2030, un 20% más que en 2025 (3,7% al año), porque suma grandes consumidores especiales y movilidad eléctrica que todavía no aparecen en la serie observada. En 2030 la banda de 68% va de 95.583 a 106.358 GWh, cerca de ±5% alrededor del escenario medio; su borde superior es la demanda del escenario de estrés (sección 2.5).

### 4.2 La brecha de energía firme

La brecha de cada período en los tres escenarios de la sección 3.3, con la hidrología crítica. Los marcadores vacíos son la brecha del año seco observado (triángulos) y del año normal (círculos y cuadrados), y los rombos, las cifras del Boletín 364 de XM, tomadas de la prensa.

In [53]:
ref = leer("referencias_suficiencia.csv").query("concepto == 'brecha_firme' and fuente == 'XM' and periodo in @PERIODOS")
fig = px.bar(balances.assign(escenario=balances["escenario"].map(NOMBRES_ESCENARIO)), x="periodo", y="brecha_gwh",
             color="escenario", barmode="group", color_discrete_map={NOMBRES_ESCENARIO[k]: v for k, v in COLORES_ESCENARIO.items()},
             custom_data=["brecha_pct"], labels={"periodo": "Período del Cargo por Confiabilidad (diciembre a noviembre)",
                                                "brecha_gwh": "GWh por período (positivo = falta)", "escenario": "Escenario"},
             title="Brecha de energía firme por período: demanda menos oferta firme")
fig.update_traces(hovertemplate="%{y:,.0f} GWh (%{customdata[0]:.1f}% de la demanda)<extra></extra>")
fig.add_scatter(x=ref["periodo"], y=ref["valor"], mode="markers", name="XM, Boletín 364 (prensa)",
                marker=dict(symbol="diamond", size=12, color="#1f1f1f"))
for hidrologia, simbolos in [("Año seco observado", ("triangle-up-open", "triangle-down-open")),
                             ("Año normal", ("circle-open", "square-open"))]:
    for e, simbolo in zip(["base", "estres"], simbolos):
        fig.add_scatter(x=PERIODOS, y=hidrologias[(hidrologia, NOMBRES_CORTOS[e])], mode="markers",
                        name=f"{hidrologia}, {NOMBRES_CORTOS[e].lower()}",
                        marker=dict(symbol=simbolo, size=11, color=COLORES_ESCENARIO[e], line=dict(width=2)))
fig.update_layout(height=480, legend=LEYENDA, separators=",.")
fig.show()

# Cuánto pesan las plantas menores en la comparación con el Boletín de XM
menores_2027 = parque.loc[parque["fuente"] == "menor (factor firme)", "firme_gwh_dia"].sum() * dias["2026-2027"]
brecha_2027 = brecha_de("base").at["2026-2027", "brecha_gwh"]
boletin_2027 = ref.set_index("periodo").at["2026-2027", "valor"]
print(f"2026-2027: brecha base {es(brecha_2027)} GWh; sin las plantas menores ({es(menores_2027)} GWh), {es(brecha_2027 + menores_2027)}; "
      f"Boletín 364 de XM, {es(boletin_2027)}")

2026-2027: brecha base 1.770 GWh; sin las plantas menores (4.982 GWh), 6.752; Boletín 364 de XM, 7.669


**Lectura.** Con la hidrología crítica, la brecha crece hasta 2028-2029 en el base y hasta 2029-2030 en el estrés, y no se cierra en ningún escenario: ni siquiera el optimista, sin retrasos, alcanza la demanda media. Con un año seco observado es unos 2.200 GWh menor, y en el base de 2026-2027 desaparece; con un año normal sobra energía firme en todos los períodos del base, y con la demanda alta sobra hasta 2027-2028 y faltan 452 y 645 GWh en 2028-2029 y 2029-2030.

El Boletín 364 de XM estima 7.669 GWh para 2026-2027, más que el propio estrés (4.493). Las plantas menores explican buena parte de la distancia: aquí aportan 4.982 GWh en 2026-2027, y sin ellas la brecha base de ese período sería de 6.752 GWh. El Boletín solo se conoce por la prensa, que no explica su método, así que la comparación es indicativa.

### 4.3 Por qué crece la brecha

Cuánto cambia cada componente de la oferta firme del caso base frente a 2026-2027, y cuánto crece la demanda. El parque existente, que pone unos 91.700 GWh por período, casi no cambia: lo que cuenta es la diferencia.

In [54]:
base = balance("base").set_index("periodo").select_dtypes("number")
cambio = base.loc[PERIODOS[1:]] - base.loc[PERIODOS[0]]                    # frente a 2026-2027
demandas = demanda.pivot(index="periodo", columns="escenario_demanda", values="demanda_gwh").reindex(PERIODOS)
cambio_demanda = demandas.loc[PERIODOS[1:]] - demandas.loc[PERIODOS[0]]
componentes = {"nuevas_gwh": "Proyectos con OEF (con retrasos)", "sin_subasta_gwh": "Entradas sin subasta",
               "menores_gwh": "Menores (días del período)", "enficc_gwh": "Plantas con ENFICC (retiro de Termoyopal 2 y días)"}
colores = ["#eda100", "#008300", "#9cc3ec", "#2a78d6"]
fig = go.Figure()
for (col, nombre), color in zip(componentes.items(), colores):
    fig.add_trace(go.Bar(x=PERIODOS[1:], y=cambio[col], name=nombre, marker_color=color,
                         hovertemplate=f"{nombre}: %{{y:,.0f}} GWh<extra></extra>"))
for escenario, estilo in [("media", "solid"), ("alta", "dash")]:
    fig.add_trace(go.Scatter(x=PERIODOS[1:], y=cambio_demanda[escenario], name=f"Aumento de la demanda {escenario}",
                             mode="lines+markers", line=dict(color="#1f1f1f", dash=estilo, width=2),
                             hovertemplate="%{y:,.0f} GWh<extra></extra>"))
fig.update_layout(barmode="relative", title="Cuánto crecen la oferta firme del caso base y la demanda frente a 2026-2027",
                  xaxis_title="Período del Cargo por Confiabilidad", yaxis_title="GWh por período, frente a 2026-2027",
                  height=460, legend=LEYENDA, separators=",.")
fig.show()
print("Frente a 2026-2027 (GWh): " + "; ".join(
    f"{p}: oferta +{es(cambio.at[p, 'oferta_firme_gwh'])}, demanda media +{es(cambio_demanda.at[p, 'media'])} "
    f"y alta +{es(cambio_demanda.at[p, 'alta'])}" for p in PERIODOS[1:]))

Frente a 2026-2027 (GWh): 2027-2028: oferta +1.737, demanda media +2.776 y alta +3.776; 2028-2029: oferta +1.795, demanda media +5.192 y alta +7.042; 2029-2030: oferta +4.436, demanda media +7.251 y alta +9.876


**Lectura.** Frente a 2026-2027, la oferta firme del caso base crece 4.436 GWh hasta 2029-2030: 4.022 de los proyectos con OEF, con retrasos, y 607 de las entradas sin subasta, menos 193 por el retiro de Termoyopal 2. La demanda media crece 7.251 GWh y la alta, 9.876: la oferta nueva cubre unos tres quintos del crecimiento de la demanda media, y el resto es la brecha que se abre. En 2028-2029 la oferta apenas sube frente a 2027-2028 (de 1.476 a 1.583 GWh de proyectos con OEF, porque solo dos tienen su compromiso en ese período), mientras la demanda sigue creciendo: por eso ahí la brecha llega a su máximo.

### 4.4 Cuánto cuesta un MWh firme

El costo anual por MWh firme de cada bloque candidato (sección 3.2), en el escenario central de costos, con el rango entre el bajo y el alto. Las líneas son los precios de cierre de la subasta del cargo 2029-2030, que pagan solo la firmeza.

In [55]:
costos = pd.concat([bloques(e).assign(escenario_costos=e) for e in ["bajo", "central", "alto"]])
c = costos.pivot(index="tecnologia", columns="escenario_costos", values="costo_firme_usd_mwh").sort_values("central")
nombres = c.index.map(NOMBRES_TECNOLOGIA)
fig = go.Figure(go.Bar(x=c["central"], y=nombres, orientation="h", marker_color="#2a78d6", name="Escenario central",
                       error_x=dict(type="data", symmetric=False, array=c["alto"] - c["central"],
                                    arrayminus=c["central"] - c["bajo"], color="#555"),
                       hovertemplate="%{y}: %{x:.0f} USD/MWh firme<extra></extra>"))
cierres = leer("referencias_suficiencia.csv").set_index("concepto")["valor"]
for concepto, nombre, estilo in [("precio_cierre_cxc_renovables", "Cierre del cargo 2029-2030, renovables", "dash"),
                                 ("precio_cierre_cxc_gas_liquidos", "Cierre del cargo 2029-2030, gas y líquidos", "dot")]:
    fig.add_trace(go.Scatter(x=[cierres[concepto]] * 2, y=[nombres[0], nombres[-1]], mode="lines",
                             name=f"{nombre}: {es(cierres[concepto], 1)}", line=dict(color="#eb6834", dash=estilo)))
fig.update_layout(title="Costo anual por MWh de energía firme de cada bloque candidato (USD de 2025)",
                  xaxis_title="USD por MWh firme (costo fijo + costo variable esperado)", yaxis_title="Bloque candidato",
                  height=420, legend=LEYENDA, separators=",.")
fig.show()
fijo = bloques(objetivo="solo_fijo").set_index("tecnologia")["costo_firme_usd_mwh"]
print(f"Solo con el costo fijo: ciclo abierto {es(fijo['gas_ciclo_abierto'], 1)} y bagazo {es(fijo['biomasa_bagazo'], 1)} "
      "USD/MWh firme")

Solo con el costo fijo: ciclo abierto 16,8 y bagazo 20,5 USD/MWh firme


**Lectura.** Ordenados por costo por MWh firme: bagazo (21 USD), ciclo combinado (39), motores diésel (47), ciclo abierto (51), solar (95) y eólica (145). Las líneas no se comparan directamente con las barras: el cierre de la subasta del cargo (22,0 USD/MWh para renovables y 16,4 para gas y líquidos) paga solo la firmeza, porque la planta vende aparte su energía, mientras que las barras cubren todo el costo de la planta. Con solo el costo fijo, el ciclo abierto cuesta 16,8 USD por MWh firme, casi el cierre para gas, y el bagazo 20,5, cerca del cierre para renovables. Los costos del modelo son del orden de lo que pagó la subasta. Es una señal de orden de magnitud, no una validación: el cierre de una subasta refleja las estrategias de oferta y los demás ingresos de las plantas, no solo su costo.

### 4.5 El portafolio de mínimo costo

Los MW nuevos en operación por tecnología y período que escoge el MILP en cada escenario, en el caso central (sección 3.4).

In [56]:
filas = []
for escenario, (_, det) in corridas.items():
    for k, tec in enumerate(det["bl"]["tecnologia"]):
        for t, periodo in enumerate(PERIODOS):
            filas.append({"escenario": NOMBRES_ESCENARIO[escenario], "periodo": periodo, "tecnologia": NOMBRES_TECNOLOGIA[tec],
                          "mw": det["x"][k, t] * det["bl"].at[k, "tamano_mw"], "firme_gwh": det["x"][k, t] * det["pr"]["F"][k, t]})
portafolio = pd.DataFrame(filas)
fig = px.bar(portafolio[portafolio["mw"] > 0], x="periodo", y="mw", color="tecnologia", facet_col="escenario",
             color_discrete_map=COLORES_TECNOLOGIA, category_orders={"periodo": PERIODOS, "escenario": list(NOMBRES_ESCENARIO.values())},
             labels={"periodo": "Período del cargo", "mw": "MW nuevos en operación", "tecnologia": "Tecnología"},
             title="Expansión de mínimo costo: MW nuevos en operación por escenario (caso central)")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(categoryorder="array", categoryarray=PERIODOS)
fig.update_layout(height=460, legend=LEYENDA, separators=",.")
fig.show()

**Lectura.** Nada nuevo entra antes de 2028-2029 en ningún escenario: en 2026-2027 no llega ninguna tecnología, y en 2027-2028 la solar con OEF ya llena el tope. Desde 2028-2029 el modelo escoge sobre todo gas de ciclo abierto: 600 MW en el base, 1.200 en el estrés y 500 en el optimista, que ponen 4.730, 9.461 y 3.942 GWh firmes. En el base quedan 20 GWh sin cubrir en 2028-2029: al MILP le sale más barato pagar la penalidad por ellos que construir otro bloque. En el base y el estrés agrega bagazo hasta su cupo (40 MW) y solar (200 MW en total). En 2029-2030 solo el estrés construye más, 100 MW eólicos y 100 solares, porque en el base y el optimista lo construido para 2028-2029 ya sobra cuando la brecha baja. El ciclo combinado no entra en ningún escenario: llega en 2029-2030, cuando lo ya construido casi cubre la brecha. En el estrés, el gas usa 1.200 de sus 1.400 MW de cupo, y el modelo supone que quema GNL importado, el combustible de su costo variable (sección 3.2).

### 4.6 Con qué se cierra la brecha del estrés

La energía firme que aportan los bloques nuevos frente a la brecha del estrés, y la energía que ningún bloque alcanza a cubrir a tiempo.

In [57]:
_, det = corridas["estres"]
cobertura = portafolio[(portafolio["escenario"] == NOMBRES_ESCENARIO["estres"]) & (portafolio["firme_gwh"] > 0)]
fig = px.bar(cobertura, x="periodo", y="firme_gwh", color="tecnologia", color_discrete_map=COLORES_TECNOLOGIA,
             category_orders={"periodo": PERIODOS},
             labels={"periodo": "Período del Cargo por Confiabilidad", "firme_gwh": "GWh por período", "tecnologia": "Tecnología"},
             title="¿Con qué se cierra la brecha? Estrés: demanda alta, con retrasos (caso central)")
fig.add_bar(x=PERIODOS, y=det["u"], name="Energía no cubierta (ningún bloque llega a tiempo)",
            marker=dict(color="#c9c7bf", pattern_shape="/"))
fig.add_scatter(x=PERIODOS, y=brecha_de("estres")["brecha_gwh"], name="Brecha firme (sección 3.3)",
                mode="lines+markers", line=dict(color="#1f1f1f", width=2))
fig.update_xaxes(categoryorder="array", categoryarray=PERIODOS)
fig.update_layout(barmode="stack", height=460, legend=LEYENDA, separators=",.")
fig.show()

**Lectura.** En el estrés, toda la brecha de 2026-2027 y 2027-2028 (4.493 y 6.532 GWh) queda sin cubrir: ningún bloque alcanza a llegar. Desde 2028-2029 los bloques nuevos la cubren, con un pequeño exceso porque son indivisibles: 9.764 GWh firmes para una brecha de 9.740, y 10.000 para una de 9.933 en 2029-2030. La energía no cubierta no es un costo de construir: es la exposición al racionamiento que deja el calendario.

### 4.7 Qué cambia con los supuestos

Los MW de cada tecnología en 2029-2030 y el costo en valor presente de cada caso de sensibilidad (sección 3.5), en el estrés. El costo se separa en el de los bloques y el de la penalidad de la energía no cubierta.

In [58]:
e = sens.xs("estres", level="escenario").iloc[::-1]          # el caso central arriba
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.03, column_widths=[0.55, 0.45],
                    subplot_titles=("MW nuevos en operación en 2029-2030", "Costo en valor presente (MUSD de 2025)"))
for tec, nombre in NOMBRES_TECNOLOGIA.items():
    if e[f"mw_{tec}"].max() > 0:
        fig.add_trace(go.Bar(y=e.index, x=e[f"mw_{tec}"], orientation="h", name=nombre,
                             marker_color=COLORES_TECNOLOGIA[nombre]), row=1, col=1)
fig.add_trace(go.Bar(y=e.index, x=e["costo_bloques_vp_musd"], orientation="h", name="Costo de los bloques",
                     marker_color="#555"), row=1, col=2)
fig.add_trace(go.Bar(y=e.index, x=e["costo_total_vp_musd"] - e["costo_bloques_vp_musd"], orientation="h",
                     name="Penalidad de la energía no cubierta", marker=dict(color="#c9c7bf", pattern_shape="/")), row=1, col=2)
fig.update_layout(barmode="stack", title="Sensibilidades del MILP en el estrés: capacidad y costo de cada caso",
                  height=480, legend=dict(LEYENDA, y=-0.1), margin=dict(l=280), separators=",.")
fig.update_xaxes(title_text="MW", row=1, col=1)
fig.update_xaxes(title_text="MUSD de 2025", row=1, col=2)
fig.update_yaxes(title_text="Caso de sensibilidad", row=1, col=1)
fig.show()

**Lectura.** El gas de ciclo abierto es estable: pone entre 1.000 y 1.200 MW en todos los casos salvo uno, el tope solar máximo del PIEG, con el que baja a 800 porque entran 3.100 MW solares, que llegan antes. La solar, en cambio, va de 100 a 3.100 MW según el caso: llena el espacio que dejan el tope y los plazos. Lo que cambia la energía sin cubrir es el calendario: con plazos un período más largos sube a 19.513 GWh y la penalidad domina el costo; con el tope solar del PIEG baja a 10.111 GWh (referencia) o 7.828 (máximo), y con 200 MW de diésel, a 9.479. Los costos bajos y altos no mueven los MW, solo el costo de los bloques. Con la penalidad más baja salen la eólica y un bloque solar.

Para mover estos supuestos sin correr el notebook, el proyecto tiene una [app interactiva](https://cterrassa.github.io/suficiencia-sin/app/) que corre en el navegador.

### 4.8 Respuesta a la pregunta

Aquí juntamos las piezas de las secciones 3 y 4 para responder la pregunta de la sección 1, en sus tres pasos: cuánta energía firme falta, qué se construye y cuánto cuesta.

**La respuesta.** En el caso base, con la demanda media de la UPME y los retrasos observados en la subasta de 2019, la expansión de mínimo costo es de 840 MW que entran en 2028-2029: 600 MW de gas de ciclo abierto, 200 MW solares y 40 MW de bagazo, por 423 MUSD en valor presente. Esa cifra es el valor presente de sus costos anuales dentro del horizonte, no la inversión, que ronda los 803 MUSD. Desde 2028-2029 el plan cubre la brecha, salvo 20 GWh que al MILP le resulta más barato no cubrir; antes no hay con qué, y en 2026-2028 quedan 4.579 GWh sin cubrir. Paso por paso:

**1. Cuánta energía firme falta y cuándo.** Con la hidrología crítica de la ENFICC, el criterio del Cargo por Confiabilidad, falta desde 2026-2027: 1.770 GWh, el 1,9% de la demanda. La brecha sube a 2.809 GWh en 2027-2028, llega a su máximo en 2028-2029 (5.168 GWh, el 5,2%) y queda en 4.585 (4,6%) en 2029-2030. Con la demanda alta llega a 9.933 GWh (9,4%); sin retrasos baja, pero no se cierra en ningún período. Con un año seco como los observados desde 2010, los tres con El Niño, la brecha es unos 2.200 GWh menor: en el base aparece en 2027-2028 (605 GWh) y llega a 2.970 en 2028-2029, y con la demanda alta falta en todos los períodos. En un año normal no falta en el base; con la demanda alta faltan 452 GWh en 2028-2029 y 645 en 2029-2030, un resultado en el margen (sección 3.3).

**2. Qué se construye y cuándo.** Nada nuevo llega antes de diciembre de 2028: en 2027-2028 solo podría entrar la solar, y los proyectos solares con OEF ya ocupan el tope de construcción. Desde 2028-2029, la expansión es sobre todo gas de ciclo abierto: de lo que puede llegar ese período, es lo más barato por MWh firme después del bagazo, que solo puede sumar 40 MW, y el ciclo combinado, más barato aún, no llega antes de 2029-2030. En el estrés, el portafolio sube a 1.200 MW de ciclo abierto, 200 solares, 100 eólicos y 40 de bagazo.

**3. Cuánto cuesta y qué queda sin cubrir.** Los bloques cuestan 423 MUSD en valor presente en el base y 810 en el estrés. La energía que no alcanza a cubrirse es, casi toda, la de 2026-2028, cuando nada nuevo puede llegar: 4.599 GWh en todo el horizonte en el base y 11.025 en el estrés.

**Qué cambia la respuesta.** Lo que más la mueve es el plazo: si cada tecnología llega un período más tarde, la energía sin cubrir del estrés pasa de 11.025 a 19.513 GWh. Le sigue la hidrología: con un año seco observado, la brecha de cada período es unos 2.200 GWh menor. Los costos, en cambio, mueven poco el portafolio: no cambian los MW del base ni del estrés, y en el optimista, con costos altos, pasan de 500 a 840 (sección 3.5).

**El riesgo.** El problema no es de costos: el costo fijo del ciclo abierto por MWh firme (16,8 USD) es prácticamente el cierre de la subasta del cargo para gas (16,4). El riesgo está en los plazos (de la OEF nueva de la subasta de 2019, el 47% entró tarde o no ha entrado) y en el combustible que respalda la ENFICC térmica, que el balance da por hecho.

**Con la brecha como distribución.** Esta respuesta usa el valor esperado de la brecha. La sección 5 la trata como una distribución: el plan de 840 MW cubre 2029-2030 en la mitad de los casos (53%), y cubrirlo con 80% de confianza pide 1.340 MW y 755 MUSD.

### 4.9 Limitaciones

La respuesta depende de supuestos que conviene tener presentes al leer las cifras. Agrupamos sus limitaciones por tipo.

**Supuestos.**
- **Energía, no potencia.** El balance es de energía firme por período; no hay un balance hora a hora, y por eso las baterías no entran al modelo.
- **El combustible se da por hecho.** El balance supone que las térmicas tienen el gas que respalda su ENFICC, aunque el PIEG advierte que abastecerlo exige nuevos yacimientos y más infraestructura para importar GNL (p. 93). El modelo limita los MW de gas, no el combustible.
- **Un bloque por tecnología,** sin economías de escala ni ubicación. La red solo entra como cupo, y el único tope de construcción es el solar, tomado del récord de entradas en XM y no de un estudio de la cadena de suministro.
- **El horizonte corta los costos.** Cada bloque paga, en valor presente, solo las anualidades de los períodos que opera hasta 2029-2030, aunque dure de 25 a 35 años. El método supone que después de 2030 cada anualidad se paga con lo que aporta la planta, algo que el modelo no comprueba.
- **Fuentes de distinto peso.** El costo del gas y de la eólica viene del Catálogo de la UPME; el del diésel y el del bagazo, de una sola fuente extranjera cada uno (CNE y EPE). El primer período de la solar y del diésel es más temprano que el que da su plazo de obra (sección 3.2).
- **La tasa de descuento.** El PIEG trabaja con un 10% anual (p. 131), pero no dice si es real o nominal. La usamos como real porque los costos están en USD constantes; con 8% y 12%, la energía sin cubrir cambia en 20 GWh o menos (pipeline).
- **La conversión de moneda es aproximada.** Los precios de los combustibles (UPME, en USD de diciembre de 2024) y los costos de EPE (en reales de diciembre de 2024) se llevan a USD de 2025 con el CPI promedio de 2024, y los de CNE (en USD de febrero de 2026), con el promedio de 2026. Con el CPI del mes de origen (factores de 1,0137 y 0,9832 en vez de 1,0263 y 0,9702), los primeros bajan 1,2% y los de CNE suben 1,3%. En una corrida de comprobación del mismo MILP con esos factores, el portafolio no cambia en ningún escenario y el costo de los bloques baja 0,7%: de 423 a 420 MUSD en el base y de 810 a 804 en el estrés.
- **Los años seco y normal son aproximaciones.** Salen de la generación de las hidráulicas con ENFICC en tres períodos secos y siete normales desde 2010, así que sus medianas son frágiles: la de los normales va de 1,22 a 1,26 según el período que se deje por fuera, y eso basta para que, con la demanda alta, el año normal pase de faltar a sobrar. La clasificación es por aportes y no por El Niño: 2014-2015 y 2018-2019 tuvieron El Niño casi todo el período y quedan entre los normales. El año normal subestima la holgura, porque en él la generación hidráulica la limita la demanda y no el agua. Las plantas menores mantienen su factor firme en los dos.

**Sesgos.**
- **La muestra de retrasos es pequeña y no se parece a los proyectos que faltan.** Sale de una sola subasta, con nueve plantas nuevas, y el 53% que entró a tiempo es casi todo de una sola planta, Tesorito: la muestra equivale a unas tres plantas iguales (ejercicio 3 del notebook de ejercicios). Además, la cohorte de 2019 era sobre todo eólica en La Guajira, con problemas de transmisión y de consultas previas. Si los proyectos que faltan, en su mayoría solares, se retrasan menos, la brecha estaría sobreestimada.
- **La homologación se evalúa con sus propias etiquetas.** Las etiquetas de los 205 pares las propuso una IA y las revisó otro agente de IA; nosotros revisamos 24 y los verificamos en fuentes públicas (resultados de las subastas, listado y energía firme por planta de XM, bitácora de SIMEM, informes y registro de la UPME, CREG y ANLA). Con esas mismas etiquetas se escogió el umbral, así que la precisión de 0,92 y la exhaustividad de 0,71 pueden ser optimistas. Al escoger con una mitad de las etiquetas y evaluar con la otra, la exhaustividad baja a 0,56 y a 0,62, y la precisión queda en 1,00 y en 0,83 (sección 3.1). Con solo 17 parejas verdaderas, esas cifras dependen de la partición. Dos pares no tienen una respuesta limpia. Andrómeda es el único que el método acepta y la etiqueta rechaza. La evidencia apunta a que es la misma planta: el registro de proyectos de XM tiene un solo Andrómeda, con el código de la OEF. Pero el listado de XM la tiene a nombre de otro agente y ningún documento lo aclara, así que dejamos la etiqueta como distinta, la opción prudente: si se confirma, la precisión sería 1,00 y no 0,92. En la oferta la contamos como la misma, una sola vez. La repotenciación de Proeléctrica, una térmica de Cartagena con OEF de planta existente con obras, es la misma central, pero su obra aún no figura en el listado de XM, y por eso no cuenta como pareja.
- **Una cifra es una inferencia nuestra:** la energía firme que suma la obra de Ituango en 2029-2030 (2,17 GWh-día). El ejercicio 2 del notebook de ejercicios mide cuánto cambia la respuesta si no llega.

**Alcance de la inferencia.**
- **El cálculo de las secciones 3 y 4 es determinista:** valores esperados para tres escenarios. La sección 5 muestra que la incertidumbre de la brecha es mayor que la brecha, y sus percentiles dependen de los supuestos de la simulación: no son intervalos de confianza de un modelo estimado.
- **Algunos insumos vienen procesados.** Las plantas del mes de referencia, los proyectos con OEF, los retrasos de las subastas, las entradas sin subasta, la hidráulica por período, las bases de la homologación y los costos de los bloques los armó el pipeline del proyecto con descargas que tardan horas. Este notebook rehace el método a partir de ellos, comprueba que llega a las mismas cifras y documenta la fuente de cada parámetro.

**Causalidad.**
- **La relación entre la hidrología, El Niño y el precio de bolsa (sección 2.5) es una asociación,** no un efecto causal: la hidrología mueve a la vez los aportes, el embalse y el precio.
- **El MILP es normativo:** dice qué convendría construir al menor costo, no predice qué se construirá ni qué decidirán los inversionistas o el regulador.

**Validez externa.**
- **Las cifras valen para el SIN, con los datos al 4 de noviembre de 2026 y las reglas vigentes de la CREG.** Una nueva subasta, un cambio de reglas o una nueva proyección de la UPME cambian la respuesta. El método se puede llevar a otro sistema que use energía firme; sus números, no.

**Lo que sigue.** La respuesta usa el valor esperado de los retrasos y tres escenarios fijos. En la sección 5 le preguntamos a una IA cómo mejorar el método e implementamos dos mejoras que tratan la brecha como incierta.

<a id="mas-alla" name="mas-alla"></a>
## 5. Uso de IA para ir más allá de la metodología

Le preguntamos a una IA cómo mejorar la metodología de las secciones 2 a 4. Aquí reproducimos la consulta y la respuesta completas, señalamos qué va más allá de lo que recomendaron los profesores e implementamos dos mejoras, con su efecto sobre la respuesta.

### 5.1 La consulta

La hicimos el 18 de noviembre de 2026, hacia las 9:50 a. m., en un chat nuevo de Claude (modelo Fable 5.2 Max, de Anthropic), en modo incógnito y sin búsqueda web ni conectores, para que la IA no conociera el proyecto.

El prompt tiene ocho partes: contexto, pregunta, metodología actual, resultados, lo que ya recomendaron los profesores, lo que le pedimos, restricciones y formato de la respuesta. No incluye nuestras limitaciones (sección 4.9), para ver si la IA encontraba sola las debilidades del método. Este es el prompt, tal como se envió:

```text
Contexto
Somos un equipo de tres estudiantes de la Maestría en Economía Aplicada de la Universidad de los Andes (Colombia). Para el trabajo final del Taller de Programación en Python preparamos un tutorial en un Jupyter Notebook que analiza la suficiencia de energía firme del sistema eléctrico colombiano. Queremos su concepto crítico sobre cómo mejorar la metodología.

El Sistema Interconectado Nacional (SIN) depende sobre todo de la generación hidráulica. Para garantizar el suministro en los años secos, como los de El Niño, la CREG creó en 2006 el Cargo por Confiabilidad. Cada planta tiene una energía firme (ENFICC): la que puede garantizar con la hidrología crítica, si es hidráulica, o con su disponibilidad y su combustible, si es térmica. En subastas se asignan obligaciones de energía firme (OEF) por períodos de diciembre a noviembre; los proyectos nuevos que ganan OEF se comprometen a entrar en operación en una fecha.

La pregunta
¿Qué expansión de mínimo costo necesita el SIN para que su energía firme cubra la demanda proyectada por la UPME en cada período del Cargo por Confiabilidad entre 2026-2027 y 2029-2030? El horizonte va de diciembre de 2026 a noviembre de 2030, con datos al 4 de noviembre de 2026. El análisis es de energía firme (año seco); no incluye potencia hora a hora, red de transmisión ni precio de bolsa.

La metodología actual
1. Datos. XM, el operador del sistema (API y SIMEM): ENFICC, OEF y capacidad por planta, y series mensuales de demanda, aportes hídricos y precio desde 2000. UPME: proyección mensual de demanda 2026-2040 (revisión de julio de 2026), Catálogo Tecnológico de costos, Plan Indicativo de Expansión de la Generación (PIEG 2025-2039) y registro de proyectos. DANE: IPC. NOAA: índice ONI. Todo se agrega a períodos de diciembre a noviembre. La demanda es la total: SIN + grandes consumidores especiales + movilidad eléctrica − generación distribuida.
2. Homologación. Las bases de proyectos con OEF, de proyectos registrados en la UPME y de plantas de XM no comparten un código. Las emparejamos con RapidFuzz: normalizamos los nombres, puntuamos los pares dentro de la misma tecnología, descartamos los que tienen atributos incompatibles (capacidad, etapa, fechas) y escogemos el umbral (token_set_ratio >= 86) con 205 pares revisados a mano: precisión de 0,92 y exhaustividad de 0,71.
3. Retrasos. Con la homologación medimos qué pasó con las nueve plantas nuevas de la subasta de 2019, la única cuyos compromisos ya vencieron. Medido por OEF, el 53% entró a tiempo, el 19% entró tarde (20,7 meses en promedio, unos dos períodos) y el 28% no ha entrado. A cada uno de los 41 proyectos con OEF que aún no operan le aplicamos esas fracciones como valor esperado: cuenta el 53% de su OEF desde su período comprometido y el 72% desde dos períodos después; el resto nunca. Si su compromiso ya venció sin entrar, no cuenta.
4. Balance de energía firme. Oferta de cada período = ENFICC de las plantas que operan (237,7 GWh-día en octubre de 2026) + plantas menores sin ENFICC, con su capacidad por el factor firme del PIEG de su tecnología (13,65 GWh-día) + entradas sin subasta al ritmo observado (pequeñas centrales hidráulicas y solares de 1 a 20 MW) + OEF esperada de los proyectos nuevos − retiros. Brecha = demanda − oferta. Tres escenarios deterministas: base (demanda media, con retrasos), estrés (borde superior de la banda de 68% de la UPME, con retrasos) y optimista (demanda media, sin retrasos).
5. Expansión de mínimo costo. Un programa lineal entero mixto (scipy.optimize.milp, con el solver HiGHS). Variables enteras: los bloques que entran por tecnología y período (gas de ciclo abierto, gas de ciclo combinado, solar, eólica, bagazo y motores diésel). Variable continua: la energía firme que queda sin cubrir. Objetivo: minimizar el valor presente de los costos anualizados (inversión con factor de recuperación de capital al 10% real, O&M y costo variable) más una penalidad de 770 USD/MWh por la energía no cubierta (primer tramo del costo de racionamiento de la UPME). Restricciones: la energía firme de los bloques más la no cubierta cubre la brecha de cada período; cada tecnología entra desde su primer período factible; cupos de bagazo (40 MW), diésel (0 MW) y eólica (464 MW), y un tope de construcción solar de 1.400 MW por año, el récord observado. La energía firme de un bloque usa el factor firme del PIEG de su tecnología. Costo anual por MWh firme: bagazo 21 USD, ciclo combinado 39, diésel 47, ciclo abierto 51, solar 95 y eólica 145.
6. Sensibilidades. 17 casos por escenario: costos alto y bajo, tasa de descuento, cupos, factores firmes observados en XM en lugar de los del PIEG, plazos un período más largos, el tope solar del PIEG, 200 MW de diésel, la penalidad igual a los precios de escasez y un objetivo de solo costo fijo.

Resultados principales
- Brecha del caso base: 1.770, 2.809, 5.168 y 4.585 GWh en los cuatro períodos (entre 1,9% y 5,2% de la demanda). En el estrés llega a 9.933 GWh en 2029-2030.
- Antes de diciembre de 2028 no alcanza a llegar nada nuevo, salvo la solar dentro del tope, que ya llenan los proyectos con OEF.
- Expansión del caso base: 600 MW de gas de ciclo abierto, 200 MW solares y 40 MW de bagazo desde 2028-2029, por 423 millones de USD en valor presente; quedan 4.599 GWh sin cubrir, casi todos en 2026-2028. En el estrés: 1.200 MW de ciclo abierto, 200 solares, 100 eólicos y 40 de bagazo, por 810 millones de USD.
- La sensibilidad que más pesa son los plazos: un período más de demora sube la energía sin cubrir del estrés de 11.025 a 19.513 GWh.

Lo que ya nos recomendaron los profesores (por favor no lo repita)
- No usar el precio de bolsa como resultado del modelo: dejarlo como contexto descriptivo.
- Documentar la fuente de los costos de cada tecnología y justificar los supuestos de cada una.
- Delimitar el horizonte.
- Crear una llave común entre las bases y alinear el año calendario con los períodos de diciembre a noviembre.
- Darle más peso a la homologación con RapidFuzz.
- Plantear la pregunta como cuánta energía falta y para cuándo, con un escenario de hidrología normal y uno de El Niño.
- Reducir el alcance del trabajo.

Lo que le pedimos
¿Cómo mejoraría esta metodología para responder mejor la misma pregunta? Sea crítico: señale también las debilidades que vea, aunque no tengan una solución sencilla.

Restricciones
- La pregunta y el horizonte no cambian.
- Solo Python, con los paquetes que ya usamos (pandas, NumPy, SciPy, RapidFuzz, Plotly y requests); sin solvers comerciales.
- Solo datos públicos como los de la lista. No tenemos datos horarios de demanda ni de generación para el horizonte, ni datos de la red de transmisión.
- El notebook debe correr completo en pocos minutos en un computador portátil y seguir siendo un tutorial que entienda un estudiante de maestría con conocimientos básicos de Python.
- Tenemos pocos días de trabajo.

Formato de la respuesta
Entre cinco y siete sugerencias, de la más a la menos importante. Para cada una:
a) qué cambiaría;
b) qué debilidad de la metodología corrige;
c) cómo implementarla en Python con los datos disponibles;
d) qué efecto esperaría sobre los resultados;
e) qué tan difícil es (baja, media o alta).
Si una sugerencia necesita datos que no están en la lista, dígalo. Al final, diga cuál implementaría primero y por qué.
```

Antes de que la IA respondiera, enviamos un segundo mensaje («Activa el pensamiento extendido para dar respuesta al prompt anterior»), porque esa opción no aparecía en la interfaz. Pedirlo por mensaje no cambia la configuración del modelo; lo dejamos registrado por transparencia. La IA respondió una sola vez, a los dos mensajes, y su respuesta se reproduce completa a continuación.

### 5.2 La respuesta completa

La respuesta, sin cambios. La IA la escribió en Markdown, así que se reproduce con su formato.

> El punto débil no es el optimizador. Con una penalidad de 770 USD/MWh frente a costos de 21 a 145, el MILP solo llena la brecha en orden de costo hasta donde lo dejan los topes y los plazos. La respuesta la fijan entonces tres insumos: los retrasos, las fechas de entrada y el menú de tecnologías. Además, la brecha es la resta de dos números grandes: un error de 1% en la demanda o en la ENFICC existente (unos 900 GWh) mueve la brecha de 2026-2027 cerca de 50%.
> 1. Simular los retrasos y la demanda en lugar de usar fracciones esperadas y tres escenarios
>
> * a) En cada simulación, cada proyecto con OEF recibe un desenlace (a tiempo, tarde con un retraso en meses, o no entra) y la demanda se sortea dentro de la banda de la UPME. Las tres fracciones también se sortean, porque vienen de nueve plantas. La brecha pasa a ser una distribución y el MILP actual se corre sobre sus percentiles 50, 80 y 95.
> * b) Ningún proyecto entrega el 53% de su OEF, y como la energía sin cubrir es max(0, brecha), promediar antes de optimizar la subestima. Con nueve plantas, el 53% tiene un margen de unos ±30 puntos. Las reglas además contradicen sus propios datos: un proyecto vencido cuenta cero, aunque 19/(19+28) = 40% de la OEF que no llegó a tiempo terminó entrando. Y uno con compromiso en diciembre de 2026 que el 4 de noviembre aún no operaba cuenta 53%. El «estrés» es solo el percentil 84 de la demanda con los retrasos del caso base.
> * c) Con NumPy: `rng.dirichlet(conteos + 1, size=S)` da las fracciones de cada simulación, compartidas por todos los proyectos, lo que correlaciona sus retrasos. `rng.random` asigna el desenlace y `rng.choice` un retraso de los observados. Los sorteos que contradicen lo que ya se sabe al 4 de noviembre se descartan y se repiten: eso es condicionar. Trabajen por mes y agreguen al final, para no redondear 20,7 meses a dos períodos. Demanda: media + z × (borde superior − media), con el mismo z en los cuatro períodos. Corre en segundos con datos que ya tienen.
> * d) No sé hacia dónde se mueve la mediana, porque las dos correcciones de las reglas van en sentidos opuestos. Sí espero un rango muy ancho: solo por demanda, la brecha de 2029-2030 es 4.585 ± 5.350 GWh, con cerca de 20% de probabilidad de que no haya brecha. El plan base, que hoy deja unos 20 GWh sin cubrir en 2028-2030, dejaría del orden de 1.900 GWh esperados solo en 2029-2030. La respuesta pasa de «600 MW de gas» a una curva de MW contra nivel de confianza.
> * e) Media.
>
> 2. Medir los bloques del modelo con la misma vara que los proyectos con OEF
>
> * a) Aplicar a los bloques candidatos el mismo riesgo de retraso y de no entrada, y fijar su primer período factible con evidencia. Reportar dos cifras: los MW efectivos que hacen falta y los MW que habría que contratar.
> * b) Por lo que describen, un proyecto con OEF y garantías cuenta 53%, mientras un bloque que aún no existe llega completo y puntual. Eso hace ver la expansión más pequeña y barata de lo que es, justo en el parámetro que su sensibilidad señala como el más importante. Las dos últimas subastas dejaron casi cuatro años entre la adjudicación y el inicio de la obligación (lo cito de memoria), frente a los 25 meses que supone entrar en diciembre de 2028. Revisen también si el tope solar se llena con los MW nominales de proyectos que en la oferta cuentan 53% a 72%.
> * c) Versión corta: multiplicar el coeficiente de energía firme de cada bloque por la fracción esperada de su período. Versión completa: reutilizar los sorteos de la sugerencia 1 y darle a la energía no cubierta un índice de escenario. Son 24 variables enteras más 4 continuas por escenario, que HiGHS resuelve en segundos con 200 a 500 escenarios (matriz con `scipy.sparse`).
> * d) Si los bloques fallan como la cohorte de 2019, cubrir 2028-2029 exige contratar hasta 1/0,53 ≈ 1,9 veces la energía firme. El plan base (840 MW) se acerca al del estrés (1.540 MW) y su costo casi se duplica.
> * e) Baja la versión corta, media la completa.
>
> 3. Revisar el menú: gas sin límite de combustible y ninguna opción de corto plazo
>
> * a) Añadir una restricción de combustible a los bloques de gas. Dejar entrar opciones rápidas (generación de emergencia o diésel, demanda desconectable voluntaria) con costo alto y cupo explícito, en vez de un cupo de 0 MW.
> * b) El resultado depende de que el ciclo abierto sea la única tecnología barata sin cupo que llega a tiempo. Sus 600 MW necesitan del orden de 120 a 140 GBTUD de gas en firme, y los 1.200 MW del estrés el doble. La demanda nacional ronda 1.000 GBTUD y ya se cubre en parte con GNL importado (cifras de memoria). Para 2026-2028 el modelo no tiene variables de decisión: los 4.579 GWh sin cubrir son la brecha misma.
> * c) Una fila más en el MILP: bloques de gas acumulados × MW × factor firme × 24 h × consumo específico de calor ≤ gas disponible. El gas disponible está en el balance de gas natural de la UPME, que no está en su lista. Sin datos nuevos, barran el cupo de gas (0 a 1.200 MW) y el del diésel (0 a 1.000 MW) y grafiquen MW, costo y energía sin cubrir. El costo de la generación de emergencia y el potencial de demanda desconectable tampoco están en su lista.
> * d) Con gas limitado, el modelo pasa a solar, eólica y energía sin cubrir, y el costo marginal por MWh firme sube de 51 a entre 95 y 145 USD. Con opciones rápidas, la energía sin cubrir de 2026-2028 deja de ser un dato y aparece su precio.
> * e) Media (baja si se quedan en los barridos).
>
> 4. Separar el costo de tener la planta del costo de usarla
>
> * a) Dividir el costo en fijo (anualidad y O&M fijo) y variable (combustible), y cargar el variable por la generación esperada, no por la energía firme. Reportar por separado la inversión comprometida y el costo dentro del horizonte.
> * b) La energía firme es una capacidad para el año seco. Una térmica de respaldo genera mucho menos que eso en un año normal, y una solar o eólica genera más, y ese excedente hoy vale cero. El «costo por MWh firme» no dice cuántas horas de combustible supone. Que el diésel (47) quede por debajo del ciclo abierto (51) sugiere que no son las mismas para todas. Los 423 millones no son la inversión: por lo que reconstruyo, son dos anualidades de unos 270 millones, descontadas. Y la penalidad trata como racionamiento seguro un déficit que solo se materializa en un período seco.
> * c) Dos columnas en la tabla de costos. La probabilidad de año seco sale del ONI. Los factores de uso por tecnología salen de la generación por planta de la API de XM en meses secos y normales (`groupby`). La penalidad se usa como simple holgura y se reportan GWh, o se multiplica por esa probabilidad.
> * d) Los MW casi no cambian, porque el gas sigue siendo la firmeza más barata. Si hoy cargan el combustible por toda la energía firme, el costo cae hacia su sensibilidad de solo costo fijo. Con la penalidad ponderada, la eólica a 145 USD puede dejar de justificarse.
> * e) Baja.
>
> 5. Contrastar la contabilidad de energía firme con los años secos
>
> * a) Comparar, por tecnología, lo que el método cuenta como firme con lo generado en los meses críticos de 2015-2016 y 2023-2024. Recalcular así el factor firme de las plantas menores y dejar de suponer constante la ENFICC existente.
> * b) El balance nunca se compara con un resultado observado. Las plantas menores aportan 13,65 GWh-día, unos 4.980 GWh al año, casi tanto como la mayor brecha. Buena parte debe de ser hidráulica a filo de agua, que cae justo en el año seco. Los 237,7 GWh-día se mantienen cuatro años, aunque la ENFICC térmica depende de contratos de combustible que se renuevan.
> * c) Calculen el factor de planta mensual por tecnología y su mínimo móvil de 12 meses (`rolling(12).mean().min()`). Ya tienen los factores observados como sensibilidad; falta definirlos sobre la ventana seca. Dos comprobaciones baratas:
>    * 2027-2028 tiene 366 días, y un día de oferta son 251 GWh, casi 9% de esa brecha.
>    * La generación distribuida que resta la UPME no debe solaparse con las solares de 1 a 20 MW que ustedes suman.
> * d) Espero un factor menor para las pequeñas hidráulicas. Cada 10% de error en ese renglón son unos 500 GWh al año, 28% de la brecha de 2026-2027.
> * e) Media.
>
> 6. Medir el error de la homologación en GWh, no en pares
>
> * a) Evaluar ponderando por OEF y sobre pares no usados para elegir el umbral. Verificar a mano los 50 proyectos con OEF (9 + 41).
> * b) Según entiendo, la precisión de 0,92 y la exhaustividad de 0,71 se midieron sobre los mismos 205 pares con que se escogió el umbral, así que son optimistas. También pesan igual una planta de 5 MW y una de 500 MW. Una planta que ya opera y no se empareja queda como pendiente: se cuenta dos veces o infla el 28% de «no ha entrado».
> * c) Partir los 205 pares al azar, elegir el umbral con una mitad y reportar con la otra. Usar `np.average(acierto, weights=oef)` y una tabla CSV de correcciones manuales que prevalezca sobre RapidFuzz.
> * d) Poco efecto en la brecha central si los proyectos grandes ya están bien. Evita un doble conteo silencioso y da una barra de error al paso 3.
> * e) Baja.
>
> Debilidades que quedan aunque hagan todo lo anterior
>
> * Horizonte: cuatro años es menos que el plazo de casi cualquier tecnología. Para 2026-2028 no hay decisiones que optimizar, y una planta de 25 años se elige con dos años de costos.
> * Muestra de retrasos: nueve plantas de una sola subasta, afectadas por causas que el modelo excluye (transmisión, licencias, consultas). La simulación muestra esa ignorancia, no la reduce. Creo recordar que las subastas de 2008 y 2011-2012 también tienen plantas nuevas con compromisos vencidos (Ituango, Gecelca 3, Termocol, Porce IV). Si aparecen en sus datos, ampliarían la muestra.
> * Energía firme: es una cifra declarada y anual. Cubrirla no garantiza el suministro, y no cubrirla no equivale a racionar.
>
> Cuál implementaría primero
> La 1, en su versión mínima: sorteos, percentiles y el MILP sin tocar. La incertidumbre de la brecha es mayor que la brecha, así que una cifra única es lo que más debilita la respuesta. Son unas decenas de líneas de NumPy que un estudiante puede seguir, y permite retirar buena parte de las sensibilidades, lo que ayuda con el alcance. Enseguida haría la versión corta de la 2, que reutiliza esos sorteos y es la que más cambia los MW.

### 5.3 Qué va más allá de lo que recomendaron los profesores

Cada sugerencia frente a las recomendaciones que hicieron los profesores al evaluar la propuesta, y lo que hicimos con ella. Implementamos tres (las dos mejoras de 5.4 a 5.6 y el contraste con los años secos observados) y otras tres en parte.

| Sugerencia de la IA | ¿La habían recomendado los profesores? | Qué hicimos |
|---|---|---|
| 1. Simular los retrasos y la demanda | No: la evaluación de la propuesta no habla de incertidumbre | **Implementada** (5.4 y 5.5): la respuesta pasa de una cifra a un nivel de confianza, con datos que ya teníamos |
| 2. Medir los bloques nuevos con la misma vara que los proyectos con OEF | No | **Implementada en su versión corta** (5.6): corrige una incoherencia del modelo |
| 3. Límite de combustible para el gas y opciones de corto plazo | No | **En parte** (5.7): barrimos los cupos de gas y de diésel, la versión sin datos nuevos que propuso la IA. El límite de combustible pide el balance de gas de la UPME, que no está en los datos: queda como limitación (4.9) |
| 4. Separar el costo fijo del variable | En parte: pidieron documentar y justificar los costos | **En parte**: el costo variable ya se carga por la generación esperada (5.7), y corregimos cómo se presenta el costo. No ponderamos la penalidad por la probabilidad de un año seco: con el precio de escasez inferior, una penalidad 87% menor, el portafolio casi no cambia (3.5) |
| 5. Contrastar la energía firme con los años secos observados | En parte: pidieron un escenario normal y uno de El Niño | **Implementada** (3.3 y 5.7): el año seco observado, con 1,05 veces la ENFICC hidráulica, cambia la respuesta de 2026-2027. Las pequeñas hidráulicas y las solares generaron en El Niño muy por encima de su factor firme: su energía firme no está sobreestimada |
| 6. Medir el error de la homologación en GWh y fuera de muestra | No en esa forma; profundiza la recomendación de darle más peso a RapidFuzz | **En parte** (3.1): escogemos el umbral con una mitad de las etiquetas y lo evaluamos con la otra, como también nos lo sugirió la profesora al revisar el notebook; la exhaustividad baja y las dos métricas dependen de la partición. Medir el error en GWh, ponderando por la OEF, queda como la mejora siguiente |
| Debilidades que quedan: horizonte, muestra de retrasos y alcance de la energía firme | No | Las reconocemos en 4.9 y 5.7 |

### 5.4 Mejora 1: la brecha como distribución

La sección 3.3 le aplica a cada proyecto el 53% de su OEF, pero ningún proyecto entrega el 53%: entra o no entra. Aquí cada simulación decide qué pasa con cada proyecto y sortea también las probabilidades y la demanda.

Cada simulación tiene cuatro piezas:
- **Las probabilidades también son inciertas,** porque salen de nueve plantas. Cada simulación las sortea de una distribución de Dirichlet con parámetros 1 + 9 × (las fracciones por OEF de la subasta de 2019): el "conteos + 1" que propone la IA, con los conteos medidos en OEF, como en la sección 3.1. Con nueve plantas, la fracción a tiempo se mueve unos ±27 puntos.
- **Todos los proyectos comparten las probabilidades de su simulación.** Si a uno le va mal, es más probable que a los demás también: los retrasos quedan correlacionados.
- **El retraso se cuenta en meses.** Quien entra tarde se demora 15,7, 18,8 o 27,6 meses, como las tres plantas que entraron tarde en 2019, y en cada período cuenta los meses que opera. La sección 3.3 redondea ese retraso a dos períodos completos.
- **La demanda se sortea** dentro de la banda de 68% de la UPME: media + *z* × (alta − media), con *z* normal estándar y el mismo *z* en los cuatro períodos, porque una demanda alta suele seguir alta.

In [59]:
FRACCIONES_2019 = resumen_2019["fraccion_oef"].reindex(["entró a tiempo", "entró tarde", "no ha entrado"]).to_numpy()
RETRASOS_2019 = tarde.to_numpy()                                   # meses de las tres plantas que entraron tarde
por_periodo = demanda.pivot(index="periodo", columns="escenario_demanda", values="demanda_gwh").reindex(PERIODOS)
DEMANDA_MEDIA, DEMANDA_ALTA = por_periodo["media"].to_numpy(), por_periodo["alta"].to_numpy()
base_33 = balance("base").set_index("periodo").reindex(PERIODOS)
SIN_NUEVOS = (base_33["oferta_firme_gwh"] - base_33["nuevas_gwh"]).to_numpy()   # oferta sin proyectos nuevos (GWh)


def operando(retraso, k, meses=True):
    """Fracción de cada período en que opera cada proyecto (simulación × proyecto × período).

    retraso: meses de retraso de cada proyecto en cada simulación (np.inf si no entra). k: período comprometido.
    Con meses=False solo cuentan los períodos completos, como en la regla de la sección 3.3.
    """
    i = np.arange(len(PERIODOS))[None, None, :] - k[None, :, None]     # períodos desde el compromiso
    fraccion = np.clip((12 * (i + 1) - retraso[:, :, None]) / 12, 0, 1) * (i >= 0)
    return fraccion if meses else np.floor(fraccion)

In [60]:
def simular_brecha(n=20_000, semilla=SEMILLA, tamano_muestra=9, demanda_incierta=True, meses=True):
    """Simula n veces la brecha firme del caso base (GWh por período), con la entrada de cada proyecto sorteada.

    tamano_muestra: plantas detrás de las probabilidades; cada simulación las sortea de una Dirichlet con
    parámetros 1 + tamano_muestra × fracciones de 2019. Con None quedan fijas, las de MODELO_RETRASOS.
    demanda_incierta: sortea la demanda en la banda de 68% de la UPME, con el mismo z en los cuatro períodos.
    meses: retraso en meses, de las plantas de 2019; con False, dos períodos completos (sección 3.3).
    """
    rng = np.random.default_rng(semilla)
    if tamano_muestra is None:
        p = np.tile([MODELO_RETRASOS["a_tiempo"], MODELO_RETRASOS["tarde"]], (n, 1))
    else:
        p = rng.dirichlet(1 + tamano_muestra * FRACCIONES_2019, size=n)[:, :2]
    azar = rng.random((n, len(nuevos)))                  # un sorteo por proyecto, con las p de su simulación
    a_tiempo = azar < p[:, [0]]
    entra_tarde = ~a_tiempo & (azar < p[:, [0]] + p[:, [1]])
    meses_tarde = (rng.choice(RETRASOS_2019, size=azar.shape) if meses
                   else np.full(azar.shape, 12.0 * MODELO_RETRASOS["periodos_retraso"]))
    retraso = np.where(a_tiempo, 0.0, np.where(entra_tarde, meses_tarde, np.inf))
    k = nuevos["periodo_entrada"].map(PERIODOS.index).to_numpy()
    oef = nuevos[COLUMNAS_OEF].to_numpy()[None]          # GWh-día de cada proyecto por período
    nuevas = (operando(retraso, k, meses) * oef).sum(axis=1) * dias.to_numpy()
    z = rng.standard_normal((n, 1)) if demanda_incierta else np.zeros((n, 1))
    demanda_simulada = DEMANDA_MEDIA + z * (DEMANDA_ALTA - DEMANDA_MEDIA)
    return pd.DataFrame(demanda_simulada - SIN_NUEVOS - nuevas, columns=PERIODOS)

Antes de usarla, una prueba: con las probabilidades fijas, el retraso de dos períodos completos y la demanda media, la media de las simulaciones tiene que reproducir la brecha de la sección 3.3. Lo que queda de diferencia es ruido del azar.

In [61]:
prueba = simular_brecha(tamano_muestra=None, demanda_incierta=False, meses=False)
diferencia = (prueba.mean() - base_33["brecha_gwh"]).abs().max()
assert diferencia < 30, f"La simulación no reproduce la brecha de la sección 3.3 (diferencia de {diferencia:,.0f} GWh)"
print(f"✓ Sin la incertidumbre nueva, la media reproduce la brecha de la sección 3.3 (diferencia máxima {es(diferencia, 1)} GWh)")

sim = simular_brecha()
percentiles = pd.DataFrame({"valor esperado (sección 3.3)": base_33["brecha_gwh"], "P5": sim.quantile(0.05),
                            "P50": sim.median(), "P80": sim.quantile(0.8), "P95": sim.quantile(0.95)}).round(0)
percentiles["probabilidad de que no falte"] = (sim <= 0).mean().round(2)
percentiles

✓ Sin la incertidumbre nueva, la media reproduce la brecha de la sección 3.3 (diferencia máxima 6,7 GWh)


,valor esperado (sección 3.3),P5,P50,P80,P95,probabilidad de que no falte
2026-2027,1.770,-2.696,1.765,4.062,6.234,"0,26"
2027-2028,2.809,-3.210,2.930,6.103,9.139,"0,22"
2028-2029,5.168,-2.439,5.077,8.947,12.679,"0,13"
2029-2030,4.585,-4.077,4.809,9.427,13.877,"0,19"


In [62]:
fig = go.Figure()
for (bajo, alto), opacidad, nombre in [((0.05, 0.95), 0.18, "90% central de las simulaciones"),
                                       ((0.2, 0.8), 0.38, "60% central de las simulaciones")]:
    fig.add_trace(go.Scatter(x=PERIODOS + PERIODOS[::-1], y=list(sim.quantile(alto)) + list(sim.quantile(bajo))[::-1],
                             mode="lines", fill="toself", fillcolor=f"rgba(42,120,214,{opacidad})",
                             line=dict(width=0), name=nombre, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=PERIODOS, y=sim.median(), mode="lines+markers", name="Mediana",
                         line=dict(color="#184f95", width=2)))
fig.add_trace(go.Scatter(x=PERIODOS, y=base_33["brecha_gwh"], mode="markers", name="Valor esperado (sección 3.3)",
                         marker=dict(symbol="diamond", size=11, color="#eb6834")))
fig.add_hline(y=0, line_color="#888", line_width=1)
fig.update_layout(title="La brecha firme del caso base como distribución (20.000 simulaciones)",
                  xaxis_title="Período del cargo", yaxis_title="Brecha firme (GWh)", legend=LEYENDA, height=460,
                  separators=",.", yaxis_tickformat=",.0f")
fig.show()

**Lectura.** La prueba sale bien: sin la incertidumbre nueva, la media de la simulación reproduce la brecha de la sección 3.3 (diferencia máxima de 6,7 GWh). Con ella, la brecha deja de ser una cifra. En 2029-2030 la mediana es de 4.809 GWh, cerca de los 4.585 del valor esperado, pero entre el 5% y el 95% de las simulaciones va de −4.077 a 13.877 GWh. En lo central, la IA tenía razón: la incertidumbre es mayor que la brecha. En el 19% de las simulaciones no falta energía firme en 2029-2030, y en el 13% no falta en 2028-2029, el peor período.

La mayor parte de ese ancho viene de la demanda: en 2026-2027, cuando no entra ningún proyecto nuevo, la brecha va de −2.696 a 6.234 GWh solo por ella. La mediana se mueve poco frente al valor esperado, y en sentidos opuestos, como anticipó la IA. Sube en 2027-2028 y 2029-2030, porque la Dirichlet con nueve plantas y el «+1» acercan las tres probabilidades a un tercio: la fracción a tiempo baja en promedio del 53% al 48%. Baja en 2028-2029, porque contar el retraso por meses le suma a ese período la parte del año en que ya operan los proyectos que entraron tarde.

### 5.5 El plan del caso base frente a la distribución

El plan de la sección 3.4 se dimensionó para el valor esperado de la brecha. Aquí se mide con qué probabilidad la cubre y se resuelve el mismo MILP para los percentiles 50, 80 y 95.

In [63]:
def evaluar_plan(det, sim):
    """Energía firme nueva de un plan en cada período y, frente a la brecha simulada, la probabilidad de cubrirla
    y la energía esperada sin cubrir (GWh). det: el detalle que devuelve correr()."""
    firme = (det["x"] * det["pr"]["F"]).sum(axis=0)
    falta = np.maximum(sim.to_numpy() - firme, 0)
    return pd.DataFrame({"energía firme nueva (GWh)": firme, "probabilidad de cubrir": (sim.to_numpy() <= firme).mean(axis=0),
                         "energía esperada sin cubrir (GWh)": falta.mean(axis=0)}, index=PERIODOS)


evaluar_plan(corridas["base"][1], sim).round(2)

,energía firme nueva (GWh),probabilidad de cubrir,energía esperada sin cubrir (GWh)
2026-2027,0,"0,26","2.187,5"
2027-2028,0,"0,22","3.408,23"
2028-2029,"5.147,38","0,51","1.807,59"
2029-2030,"5.147,38","0,53","2.025,82"


**Lectura.** El plan de 840 MW de la sección 3.4 aporta 5.147 GWh firmes desde 2028-2029. Frente a la brecha simulada, cubre 2028-2029 en el 51% de los casos y 2029-2030 en el 53%: es un plan para la mediana, no para el riesgo. La energía esperada sin cubrir de esos dos períodos suma 3.833 GWh, cuando el valor esperado decía que quedaban 20. La IA estimó del orden de 1.900 GWh esperados sin cubrir en 2029-2030, solo por la demanda; con los retrasos también sorteados, son 2.026. Es su punto central: como la energía sin cubrir es max(0, brecha − plan), promediar la brecha antes de optimizar la subestima. En 2026-2028 no hay plan posible, y la energía esperada sin cubrir es la brecha misma: 5.596 GWh entre los dos períodos, frente a los 4.579 del valor esperado.

In [64]:
planes, niveles = {}, [("Valor esperado (3.4)", None), ("P50", 0.5), ("P80", 0.8), ("P95", 0.95)]
for nivel, q in niveles:
    resumen, det = correr("base", brecha_extra=0 if q is None else sim.quantile(q) - base_33["brecha_gwh"])
    evaluacion = evaluar_plan(det, sim)
    mw = {NOMBRES_TECNOLOGIA[k[3:]]: v for k, v in resumen.items() if k.startswith("mw_")}
    planes[nivel] = mw | {"MW nuevos": sum(mw.values()), "costo de los bloques (MUSD, VP)": resumen["costo_bloques_vp_musd"],
                          "probabilidad de cubrir 2029-2030": evaluacion.at["2029-2030", "probabilidad de cubrir"],
                          "energía esperada sin cubrir 2028-2030 (GWh)":
                              evaluacion["energía esperada sin cubrir (GWh)"].iloc[2:].sum()}
planes = pd.DataFrame(planes).T
planes = planes.loc[:, (planes != 0).any()]
planes.round(2)

,Bagazo,Eólica,"Gas, ciclo abierto",Solar FV,MW nuevos,"costo de los bloques (MUSD, VP)",probabilidad de cubrir 2029-2030,energía esperada sin cubrir 2028-2030 (GWh)
Valor esperado (3.4),40,0,600,200,840,"422,88","0,53","3.833,41"
P50,40,0,600,200,840,"422,88","0,53","3.833,41"
P80,40,0,1.200,100,1.340,"754,93","0,82","1.070,34"
P95,40,400,1.400,1.200,3.040,"1.148,62","0,93","266,28"


In [65]:
tecnologias = [t for t in NOMBRES_TECNOLOGIA.values() if t in planes]
largo = planes[tecnologias].rename_axis("nivel").reset_index().melt(id_vars="nivel", var_name="tecnología", value_name="MW")
fig = px.bar(largo, x="nivel", y="MW", color="tecnología", color_discrete_map=COLORES_TECNOLOGIA,
             title="MW nuevos del caso base según la brecha que se quiere cubrir",
             labels={"nivel": "Brecha que se cubre", "MW": "MW nuevos en 2029-2030", "tecnología": "Tecnología"})
for nivel, fila in planes.iterrows():
    texto = (f"{fila['costo de los bloques (MUSD, VP)']:,.0f} MUSD".replace(",", ".")
             + f"<br>cubre 2029-2030 en el {fila['probabilidad de cubrir 2029-2030']:.0%}")
    fig.add_annotation(x=nivel, y=fila["MW nuevos"], text=texto, showarrow=False, yshift=22)
fig.update_layout(height=480, legend=LEYENDA, separators=",.")
fig.update_yaxes(range=[0, planes["MW nuevos"].max() * 1.22], tickformat=",.0f")
fig.show()

**Lectura.** El plan para la mediana (P50) es el mismo de la sección 3.4: 840 MW por 423 MUSD. Cubrir la brecha con 80% de confianza pide 1.340 MW (1.200 de ciclo abierto, 100 solares y 40 de bagazo) por 755 MUSD, un 79% más; ese plan cubre 2029-2030 en el 82% de las simulaciones y deja 1.070 GWh esperados sin cubrir en 2028-2030. Con 95% de confianza hacen falta 3.040 MW y 1.149 MUSD: el ciclo abierto llena su cupo de 1.400 MW y entran 1.200 MW solares y 400 eólicos, más caros por MWh firme. Aun así, el plan cubre 2029-2030 en el 93% de los casos, porque los cupos y el tope solar ya no dejan construir más. Pasar del 80% al 95% cuesta más que pasar de la mediana al 80%. La respuesta deja de ser «600 MW de gas» y pasa a ser una curva: cuántos MW hacen falta según el riesgo que se esté dispuesto a correr.

### 5.6 Mejora 2: los bloques nuevos con el mismo riesgo

Un proyecto con OEF entra con retraso, pero un bloque del MILP llega completo y a tiempo. La versión corta de la sugerencia 2 le aplica a cada bloque la regla de los proyectos: 53% de su energía firme en sus dos primeros períodos y 72% desde el tercero.

El bloque se paga completo aunque no llegue, así que el MILP compensa contratando más. La tabla separa los MW que habría que contratar de la energía firme que se espera de ellos. Es un caso pesimista: la cohorte de 2019 era sobre todo eólica en La Guajira, con problemas de transmisión y de consultas previas, y su tasa de fracaso exagera la de una térmica de ciclo abierto.

In [66]:
def con_riesgo(pr, modelo=MODELO_RETRASOS):
    """Copia del MILP en la que un bloque que entra en τ aporta en t solo la fracción esperada de su energía firme:
    la de entrar a tiempo en sus primeros `periodos_retraso` períodos y, desde ahí, también la de entrar tarde."""
    A, nt = pr["A"].copy(), len(PERIODOS)
    nb = (A.shape[1] - nt) // nt
    for t in range(nt):                                   # las primeras nt filas son las de la energía firme
        for tau in range(t + 1):
            fraccion = modelo["a_tiempo"] + (modelo["tarde"] if t - tau >= modelo["periodos_retraso"] else 0)
            A[t, [k * nt + tau for k in range(nb)]] *= fraccion
    return pr | {"A": A}


riesgo, nt = {}, len(PERIODOS)
for nivel, q in [("Valor esperado", None), ("P80", 0.8)]:
    det = correr("base", brecha_extra=0 if q is None else sim.quantile(q) - base_33["brecha_gwh"])[1]
    for bloques_llegan, pr in [("a tiempo (3.4)", det["pr"]), ("con el riesgo de 2019", con_riesgo(det["pr"]))]:
        res = resolver(pr)
        y = np.round(res.x[:-nt])
        x = y.reshape(-1, nt).cumsum(axis=1)
        mw = x[:, -1] * det["bl"]["tamano_mw"].to_numpy()
        riesgo[(nivel, bloques_llegan)] = {
            "MW a contratar": mw.sum(), "de ellos, ciclo abierto": mw[(det["bl"]["tecnologia"] == "gas_ciclo_abierto").to_numpy()].sum(),
            "energía firme esperada 2029-2030 (GWh)": (pr["A"][:nt, :-nt] @ y)[-1],
            "costo de los bloques (MUSD, VP)": float(pr["delta"] @ (pr["C"] @ x)), "energía sin cubrir (GWh)": res.x[-nt:].sum()}
pd.DataFrame(riesgo).T.rename_axis(["brecha", "los bloques llegan"]).round(0)

MW a contratar  de ellos, ciclo abierto  energía firme esperada 2029-2030 (GWh)  costo de los bloques (MUSD, VP)  \
brecha         los bloques llegan                                                                                                                        
Valor esperado a tiempo (3.4)                    840                      600                                   5.147                              423   
               con el riesgo de 2019           1.340                    1.200                                   5.175                              788   
P80            a tiempo (3.4)                  1.340                    1.200                                   9.764                              755   
               con el riesgo de 2019           3.040                    1.400                                   6.934                            1.149   

                                      energía sin cubrir (GWh)  
brecha         los bloques llegan                               
Valor esperado a tiempo (3.4)                            4.599  
               con el riesgo de 2019                     4.579  
P80            a tiempo (3.4)                           10.165  
               con el riesgo de 2019                    14.991

**Lectura.** La IA anticipó bien el efecto. Si los bloques nuevos se retrasan como los proyectos de 2019, cubrir el valor esperado de la brecha pide contratar 1.340 MW en lugar de 840 (1.200 de ciclo abierto en lugar de 600), y el costo de los bloques pasa de 423 a 788 MUSD, un 86% más. La energía firme que se espera de ellos en 2029-2030 es casi la misma (5.175 frente a 5.147 GWh): se contrata más para recibir lo mismo. Con la brecha del P80 ya no alcanza: el MILP contrata 3.040 MW, con el cupo de gas lleno, y solo espera 6.934 GWh firmes en 2029-2030, frente a una brecha de 9.427. Las dos mejoras apuntan al mismo lado: el plan de la sección 3.4 es el mínimo, no el prudente.

### 5.7 Lo que descartamos y por qué

Una respuesta de IA no se copia: se revisa. Aquí explicamos qué no aplicaba a nuestros datos, qué no podía ver la IA y qué simplificamos al implementar.

- **Lo que no aplicaba a nuestros datos.** La IA razonó sobre la descripción del método, sin los datos, y dos de sus críticas a las reglas no cambian ninguna cifra: ningún proyecto está vencido y ninguno tiene compromiso de entrada en diciembre de 2026 (23 entran en 2027-2028, 2 en 2028-2029 y 16 en 2029-2030). Tampoco aplican otras dos sospechas. El tope solar no se llena con MW nominales: ya usa los MW esperados con retrasos (sección 3.4). Y el combustible no se carga por toda la energía firme, sino por la generación esperada, con un factor de planta de 0,32 para el ciclo abierto y de 0,15 para el diésel; por eso el diésel sale más barato por MWh firme, como sospechaba la IA. El año de 366 días ya se contaba.
- **Dos errores de nuestro prompt.** No mencionamos el cupo de gas: 1.400 MW compartidos entre ciclo abierto y ciclo combinado (PIEG 2025-2039, Tabla 23). Por eso la IA supuso que el gas no tenía límite. Su punto de fondo sigue en pie: el modelo limita los MW de gas, no el combustible, y en el P95 ese cupo ya se llena. Además, el prompt decía que los 205 pares de la homologación se habían revisado a mano, y no era exacto: las etiquetas las propuso una IA y las revisó otro agente de IA, y nosotros revisamos solo los 24 pares que deciden las métricas (sección 3.1). Eso le da más peso a la sugerencia 6 de la IA: medir el error de la homologación fuera de muestra, que hicimos en la sección 3.1, y verificar a mano los proyectos con OEF.
- **Lo que sí se podía hacer sin datos nuevos.** Dos sugerencias tenían una versión con los datos que ya teníamos, y la hicimos: el barrido de los cupos de gas y de diésel (sugerencia 3) y el contraste del factor firme de las plantas menores con lo que generaron en El Niño (sugerencia 5). Están en las dos celdas siguientes.
- **Lo que la IA no podía ver.** Las nueve plantas de 2019 no pesan lo mismo: Tesorito tiene el 53% de la OEF, así que el 53% que entró a tiempo es, en la práctica, una sola planta. Con el tamaño efectivo de Kish, la muestra equivale a 3,2 plantas iguales, y la incertidumbre de los retrasos es mayor que la que supusimos con nueve. El ejercicio 3 del notebook de ejercicios mide cuánto cambia el P80.
- **Lo que simplificamos.** Cada percentil se calcula período por período, así que el plan del P80 cubre el P80 de cada período, no el 80% de los horizontes completos. Los retrasos en meses salen de solo tres plantas. El «+1» de la Dirichlet acerca las probabilidades a un tercio y mueve la mediana. Y la mejora 2 está en su versión corta: el riesgo de los bloques entra como valor esperado, no sorteado junto con la mejora 1.
- **Lo que adoptamos para el alcance.** La IA dijo que su primera mejora «permite retirar buena parte de las sensibilidades, lo que ayuda con el alcance». Lo hicimos: de los 17 casos del prompt, contando el central, quedan 8: el central, los que mueven la energía sin cubrir y los costos bajos y altos (sección 3.5). Los demás siguen en el pipeline del proyecto.
- **Lo que no usamos.** Las cifras que la IA dio de memoria: casi cuatro años entre la adjudicación y el inicio de la obligación en las últimas subastas, el gas que consume el ciclo abierto, la demanda nacional de gas y las plantas de las subastas de 2008 y 2011-2012. No las verificamos, así que no entran en ningún cálculo.
- **Cómo se presenta el costo.** La IA notó que los 423 MUSD no son la inversión, y tiene razón: son el valor presente de los costos anuales dentro del horizonte. La inversión del portafolio base ronda los 803 MUSD, y la sección 4.8 ahora lo dice.

In [67]:
# Sugerencia 3, sin datos nuevos: el estrés con otros cupos de gas y de diésel (el central es gas 1.400 y diésel 0)
filas = []
for gas, diesel in [(0, 0), (400, 0), (800, 0), (1200, 0), (1400, 0), (1400, 200), (1400, 500), (1400, 1000)]:
    resumen, _ = correr("estres", cupo={"gas": gas, "diesel_motores": diesel})
    filas.append({"cupo de gas (MW)": float(gas), "cupo de diésel (MW)": float(diesel),
                  **{f"{v} (MW)": resumen[f"mw_{k}"] for k, v in NOMBRES_TECNOLOGIA.items()},
                  "costo de los bloques (MUSD, VP)": resumen["costo_bloques_vp_musd"],
                  "energía sin cubrir (GWh)": resumen["no_cubierta_gwh"]})
barrido = pd.DataFrame(filas)
barrido.loc[:, (barrido != 0).any()].round(0)

,cupo de gas (MW),cupo de diésel (MW),Solar FV (MW),Eólica (MW),Bagazo (MW),"Gas, ciclo abierto (MW)",Motores diésel (MW),"costo de los bloques (MUSD, VP)",energía sin cubrir (GWh)
0,0,0,1.200,400,40,0,0,256,27.210
1,400,0,1.200,400,40,400,0,511,20.903
2,800,0,1.200,400,40,800,0,766,14.596
3,1.200,0,200,100,40,1.200,0,810,11.025
4,1.400,0,200,100,40,1.200,0,810,11.025
5,1.400,200,300,0,40,1.000,200,866,9.479
6,1.400,500,300,100,40,700,500,961,7.160
7,1.400,1.000,0,0,40,300,960,1.028,4.493


In [68]:
# Sugerencia 5: el factor firme de las menores frente a su factor de planta en los meses de El Niño (XM, 2015-2026)
fases = leer("xm_factores_planta_fase.csv").pivot(index="tecnologia", columns="fase", values="factor_planta")
grupos = {"pch": ("Pequeñas hidráulicas (no despachadas)", "hidraulica"), "solar_fv": ("Solar de 9 MW o más", "solar"),
          "biomasa_bagazo": ("Cogeneración con bagazo", "biomasa")}
pd.DataFrame({"factor firme del balance": [FACTORES_MENORES[g] for _, g in grupos.values()],
              "factor de planta en El Niño": fases.loc[list(grupos), "nino"].to_numpy(),
              "factor de planta en meses neutros": fases.loc[list(grupos), "neutro"].to_numpy()},
             index=[n for n, _ in grupos.values()]).round(2)

,factor firme del balance,factor de planta en El Niño,factor de planta en meses neutros
Pequeñas hidráulicas (no despachadas),"0,15","0,49","0,61"
Solar de 9 MW o más,"0,13","0,25","0,24"
Cogeneración con bagazo,"0,54","0,6","0,55"


**Lectura.** Sin cupo de gas, el estrés no se cubre: aun con la solar y la eólica en lo máximo que permiten su tope y su cupo (1.200 y 400 MW), quedan 27.210 GWh sin cubrir, frente a 11.025 con el cupo actual. Por debajo de 800 MW, cada 400 MW de gas menos dejan unos 6.300 GWh más sin cubrir. El cupo de 1.400 MW no se llena (el MILP se queda en 1.200), así que lo que pesa no es el cupo sino el calendario. El diésel es la única opción que reduce la energía sin cubrir antes de 2028, porque puede llegar en 2027-2028: con 1.000 MW de cupo solo quedan sin cubrir los 4.493 GWh de 2026-2027, cuando nada puede llegar. Es la «opción rápida» que señalaba la IA, con dos reparos: su primer período es optimista (sección 3.2) y no hay proyectos a líquidos en curso.

En los meses de El Niño de 2015 a 2026, las pequeñas hidráulicas generaron al 49% de su capacidad y las solares de 9 MW o más al 25%, muy por encima de los factores firmes del balance (0,15 y 0,13); la cogeneración con bagazo, al 60%, algo por encima del suyo (0,54). Contra lo que esperaba la IA, la energía firme de las menores no parece sobreestimada: el factor del PIEG es conservador. Dos reparos: son promedios de meses, no el peor año, y las demás térmicas menores no se pueden contrastar así, porque su generación mide el despacho, no la disponibilidad.

### 5.8 La respuesta, con las mejoras

Las dos mejoras no cambian la pregunta: cambian la forma de la respuesta. Esta es la respuesta de la sección 4.8, con la brecha tratada como una distribución.

**Cuánta y cuándo.** Con la hidrología crítica, la brecha del caso base tiene un valor esperado de 4.585 GWh en 2029-2030, pero entre el 5% y el 95% de las simulaciones va de −4.077 a 13.877 GWh. La mayor parte de ese rango viene de la demanda. Con un año seco como los observados desde 2010, la brecha sería unos 2.200 GWh menor (sección 3.3).

**Qué se construye y cuánto cuesta, según el riesgo.** El plan de la sección 3.4 (840 MW por 423 MUSD) cubre 2029-2030 en la mitad de los casos (53%). Para cubrirlo con 80% de confianza hacen falta 1.340 MW, sobre todo de ciclo abierto, por 755 MUSD. Con 95%, 3.040 MW y 1.149 MUSD, y ni así se cubre del todo, porque se llenan los cupos. Si además los bloques nuevos se retrasan como los proyectos de 2019, el plan del valor esperado ya pide 1.340 MW y 788 MUSD.

**Lo que no cambia.** Con los cupos actuales, antes de diciembre de 2028 no hay con qué: solo un cupo de diésel, que hoy no existe, lo cambiaría (sección 5.7). Lo que más mueve la respuesta son los plazos. La expansión de mínimo costo sigue siendo sobre todo gas de ciclo abierto: las mejoras cambian cuánto hay que construir según el riesgo que se acepte, no con qué.

**Lo que sigue.** Cierran el notebook la tabla de cómo atendimos la evaluación de la propuesta, la declaración de uso de IA y las referencias. Los ejercicios, que repiten y extienden lo hecho, están en el notebook `Ejercicios_Suficiencia_SIN.ipynb`, al lado de este en el .zip.

<a id="evaluacion" name="evaluacion"></a>
## Cómo atendimos la evaluación de la propuesta

Los profesores evaluaron la propuesta con 85 sobre 100 y dejaron comentarios por criterio y cuatro recomendaciones. Esta tabla dice qué hicimos con cada uno y dónde se ve en este notebook.

| Comentario de la evaluación | Qué hicimos | Dónde se ve |
|---|---|---|
| La pregunta tenía dos objetivos, y un MILP de mínimo costo no produce el precio de bolsa | Una sola pregunta, de suficiencia y expansión, sin el precio de bolsa | Sección 1 (la pregunta) y 4.8 |
| El horizonte («corto y mediano plazo») no estaba delimitado | Los cuatro períodos del Cargo por Confiabilidad, de diciembre de 2026 a noviembre de 2030 | Sección 1 (qué abarca) |
| Recomendación: preguntar cuánta energía y para cuándo, con un escenario normal y uno de El Niño | La brecha de cada período con tres hidrologías. El Niño se mide con dos varas: la hidrología crítica de la ENFICC, que es el criterio del cargo, y un año seco observado, como los tres períodos secos desde 2010, todos con El Niño. El año normal sale de la misma generación hidráulica observada. La expansión se dimensiona con la ENFICC | Secciones 3.3, 4.2 y 4.8 |
| Con estos datos no se puede modelar el precio de bolsa | El precio queda como contexto descriptivo, por condición hidrológica y fase de El Niño, y se lee como asociación, no como causalidad | Secciones 2.5 y 4.9 |
| Recomendación: el precio de bolsa puede analizarse con gráficos de su evolución histórica y de su relación con indicadores de suficiencia, sin presentarlo como una variable que el modelo minimiza | Una figura con la evolución del precio en pesos constantes y su relación con el margen de energía firme y con el embalse. El precio no está en la pregunta ni en el MILP | Sección 2.5 |
| Para escoger bloques de mínimo costo hacen falta costos por tecnología, con su fuente | Cada parámetro de cada bloque tiene su fuente, su tabla o página, su URL y su conversión en `fuentes_supuestos.csv`. La sección 3.2 muestra la fuente de cada uno, reproduce desde el JSON del Catálogo los costos del gas y de la eólica y comprueba que los costos de los bloques salen de esos parámetros | Secciones 3.2 y 4.4 |
| Recomendación: si mantienen el MILP, documentar y justificar los costos y los demás supuestos de cada tecnología | Lo anterior, más la fuente de los factores firmes (PIEG, Tabla 11, o XM), del primer período (el plazo de obra de la fuente), de los cupos (PIEG, Tabla 23; UPME; Asocaña), del tope solar (récord de entradas en XM), de la tasa (PIEG) y de la penalidad (UPME). Los supuestos que mueven la respuesta se prueban en las sensibilidades | Secciones 3.2, 3.4 y 3.5 |
| El plan para obtener y usar los datos: muy detallado y honesto sobre la falta de llave común y el desfase entre año calendario y periodo dic.–nov. | La homologación con RapidFuzz empareja los proyectos con OEF con las plantas de XM: con ella se sabe qué proyectos ya operan, y su energía firme no se cuenta dos veces. Todo se agrega por período del cargo | Secciones 3.1, 3.3 y 2.5 |
| Recomendación: la homologación con RapidFuzz es un muy buen aporte; darle más peso | Subsección propia: normalización, puntajes, revisión de atributos y umbral escogido con 205 pares etiquetados y comprobado fuera de muestra, con una mitad de las etiquetas para escoger y la otra para evaluar. Sus parejas deciden qué proyectos ya están en el parque | Secciones 3.1 y 3.3; ejercicio 1 del notebook de ejercicios |
| El alcance era demasiado grande para un tutorial | El notebook responde la pregunta con el método central y deja por fuera lo que no aporta a la respuesta: el pipeline completo, un simulador, las tres consultas en vivo a la API de XM, las dos animaciones y 9 de los 16 casos de sensibilidad. Los ejercicios pasaron a un notebook aparte, y las tablas de funciones y una de las animaciones, a la carpeta `docs/` del repositorio. La app del proyecto queda como un enlace | Todo el notebook; secciones 3.5 y 4.7; `Ejercicios_Suficiencia_SIN.ipynb`; `docs/` del repositorio |
| La fuente debe ser pública y verificable; el gestor normativo de la CREG es normativa, no datos. Al revisar la nota, los profesores aceptaron que la CREG es necesaria para entender el mecanismo | Los datos son de XM, la UPME, NOAA y el DANE, y los supuestos, del PIEG y el Catálogo de la UPME, CNE, EPE, NREL, Asocaña, FRED y la Superintendencia Financiera, cada uno con su URL; todo se verifica con su huella SHA-256 y se revisa tabla por tabla. La CREG entra como contexto y por sus definiciones | Secciones 2.2, 2.4 y 3.2 |
| Bien que declaren el uso de IA y cómo se valida | Declaración de uso de IA, comparación de cada resultado con el pipeline (`comprobar()`) y una sección sobre cómo usamos la IA para ir más allá de la metodología | Secciones 3.2 a 3.5 y 5; declaración de uso de IA |
| La página de la propuesta estaba muy cargada y decía «Phyton» | Una introducción de máximo tres líneas en cada sección y una revisión de ortografía de todo el texto | Todo el notebook |

<a id="ia" name="ia"></a>
## Declaración de uso de IA

Usamos IA generativa (Claude, de Anthropic) en todo el proyecto, por dos vías y bajo la dirección del equipo: Claude Code, para el código, los datos y los borradores de los textos, y el chat de Claude, para la consulta de la sección 5 y para revisar la redacción.

- **Código.** Claude escribió la mayor parte del código del pipeline del proyecto y de este notebook, a partir del plan de trabajo y de nuestras instrucciones. Este notebook rehace con funciones propias el método central del pipeline. El equipo definió la pregunta, las fuentes y las decisiones de modelación, y revisó los resultados.
- **Fuentes.** Claude buscó fuentes de costos y referencias (XM, UPME, CREG y prensa) y transcribió datos, siempre con URL, página y estado de verificación. Las cifras de prensa se identifican como tales. Antes de la entrega, Claude contrastó las cifras de `fuentes_supuestos.csv` con los documentos originales (el PIEG, el Catálogo de la UPME, CNE, EPE y la CREG).
- **Homologación.** Claude sugirió las etiquetas de los 205 pares de la sección 3.1. Después, a pedido del equipo, un agente de Claude las revisó de forma independiente contra los atributos de las fuentes, par por par y con evidencia. El 22 de noviembre de 2026 revisamos los 24 pares que deciden las métricas y le pedimos a Claude contrastarlos con fuentes públicas (resultados de las subastas, listado y energía firme por planta de XM, bitácora de SIMEM, informes y registro de la UPME, PARATEC, CREG y ANLA). Ninguna etiqueta cambió, y la evidencia de cada par quedó en el archivo de etiquetas del paquete completo (`data/manual/equivalencias_validadas.csv`).
- **Plantas, proyectos y cupos.** A pedido del equipo, agentes de Claude verificaron con fuentes primarias (UPME, XM, SIMEM, CREG, ANLA y promotores) los MW de los proyectos con OEF, las ampliaciones con obras y los cupos de bagazo, diésel y eólica. El equipo escogió los valores y aprobó los supuestos.
- **Redacción.** Claude escribió el primer borrador de los textos del notebook: el contexto, las lecturas de los resultados, los ejercicios y la sección 5. El equipo los revisó y corrigió con su propio criterio, y las interpretaciones y conclusiones son nuestras.
- **Ir más allá de la metodología.** La consulta a la IA sobre cómo mejorar el método, con el prompt, la respuesta completa y las dos mejoras que implementamos, está en la [sección 5](#mas-alla).

**Cómo se controla:**
- Toda cifra de resultados se calcula en Python desde los datos. Los parámetros transcritos (costos, cupos y factores firmes) traen su fuente y su página en `fuentes_supuestos.csv`, y las cifras de referencia, incluidas las de prensa, en `referencias_suficiencia.csv`.
- Este notebook compara sus resultados con los del pipeline, que los calculó con su propio código (`comprobar()`, en 3.3, 3.4 y 3.5). Como Claude escribió buena parte de ambos códigos, esta comparación detecta errores de implementación, no de método; el método lo revisamos nosotros.
- Los datos del paquete se verifican con su huella SHA-256, y los datos transcritos traen fuente y página.
- Cada parámetro de los bloques y cada supuesto tiene su fuente en `fuentes_supuestos.csv`, y la sección 3.2 comprueba que los costos salen de ellos.

<a id="referencias" name="referencias"></a>
## Referencias

**Académicas**

- Cramton, P. y Stoft, S. (2007). *Colombia Firm Energy Market*. Proceedings of the 40th Annual Hawaii International Conference on System Sciences (HICSS'07). IEEE. [doi.org/10.1109/HICSS.2007.133](https://doi.org/10.1109/HICSS.2007.133) ([versión de los autores](https://www.cramton.umd.edu/papers2005-2009/cramton-stoft-colombia-firm-energy-market.pdf)).
- Cramton, P., Ockenfels, A. y Stoft, S. (2013). *Capacity Market Fundamentals*. Economics of Energy & Environmental Policy, 2(2), 27-46. [doi.org/10.5547/2160-5890.2.2.2](https://doi.org/10.5547/2160-5890.2.2.2)
- Fellegi, I. P. y Sunter, A. B. (1969). *A Theory for Record Linkage*. Journal of the American Statistical Association, 64(328), 1183-1210. [doi.org/10.1080/01621459.1969.10501049](https://doi.org/10.1080/01621459.1969.10501049)
- Harbord, D. y Pagnozzi, M. (2014). *Britain's Electricity Capacity Auctions: Lessons from Colombia and New England*. The Electricity Journal, 27(5), 54-62. [doi.org/10.1016/j.tej.2014.05.004](https://doi.org/10.1016/j.tej.2014.05.004)
- Juvinao Navarro, J. C. (2021). *Lecciones del Cargo por Confiabilidad en Colombia como un mecanismo de incentivo a la generación de energía eléctrica*. Desarrollo y Sociedad, (87), 113-148. Universidad de los Andes. [doi.org/10.13043/dys.87.4](https://doi.org/10.13043/dys.87.4)
- Koltsaklis, N. E. y Dagoumas, A. S. (2018). *State-of-the-art generation expansion planning: A review*. Applied Energy, 230, 563-589. [doi.org/10.1016/j.apenergy.2018.08.087](https://doi.org/10.1016/j.apenergy.2018.08.087)
- Poveda, G., Álvarez, D. M. y Rueda, Ó. A. (2011). *Hydro-climatic variability over the Andes of Colombia associated with ENSO: a review of climatic processes and their impact on one of the Earth's most important biodiversity hotspots*. Climate Dynamics, 36(11-12), 2233-2249. [doi.org/10.1007/s00382-010-0931-y](https://doi.org/10.1007/s00382-010-0931-y)
- Ríos, D., Pérez, A., Carabalí, J. y Meneses, L. (2024). *Market power and adverse weather events in a hydro-dominated electricity market*. Borradores de Economía, 1266. Banco de la República. [doi.org/10.32468/be.1266](https://doi.org/10.32468/be.1266)

**Regulación, planeación y datos**

- CREG (2006). *Resolución 071 de 2006: metodología para la remuneración del Cargo por Confiabilidad en el Mercado Mayorista de Energía*. [gestornormativo.creg.gov.co](https://gestornormativo.creg.gov.co/gestor/entorno/docs/resolucion_creg_0071_2006.htm)
- UPME (2026). *Plan Indicativo de Expansión de la Generación 2025-2039*. [docs.upme.gov.co](https://docs.upme.gov.co/SIMEC/Energia%20Electrica/PIEG/2025-2039/PIEG_2025_2039_Version_final.pdf)
- UPME (2026). *Proyección de la demanda de energía eléctrica y potencia máxima 2026-2040*, rev. julio de 2026 (tablas de agosto de 2026). [upme.gov.co](https://www.upme.gov.co/simec/planeacion-energetica/proyeccion-de-demanda-v2/)
- MinEnergía, UPME y Agencia Danesa de Energía (2025). *Catálogo Tecnológico Colombiano*, API abierta. [api.upme.gov.co](https://api.upme.gov.co/catalogo_tecnologias/api/techs_year/)
- XM (2026). *XM informa los resultados de la subasta de asignación de Obligaciones de Energía Firme, OEF, para la vigencia diciembre 2029 – noviembre 2030* (22 de mayo de 2026). [xm.com.co](https://www.xm.com.co/noticias/9118-xm-informa-los-resultados-de-la-subasta-de-asignacion-de-obligaciones-de-energia)
- XM. API de datos (`servapibi.xm.com.co`) y SIMEM, consultados en octubre y noviembre de 2026 con `requests` y [pydataxm](https://github.com/EquipoAnaliticaXM/API_XM).
- NOAA Climate Prediction Center. *Oceanic Niño Index (ONI)*. [cpc.ncep.noaa.gov](https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt)
- FRED (CPI de EE. UU.), DANE (IPC) y datos.gov.co (TRM), para llevar los costos y precios a una misma moneda y año.
- CREG (2010). *Resolución 005 de 2010: requisitos y condiciones técnicas de la cogeneración*. [gestornormativo.creg.gov.co](https://gestornormativo.creg.gov.co/gestor/entorno/docs/resolucion_creg_0005_2010.htm)
- CREG (2021). *Resolución 174 de 2021: autogeneración a pequeña escala y generación distribuida*. [gestornormativo.creg.gov.co](https://gestornormativo.creg.gov.co/gestor/entorno/docs/resolucion_creg_0174_2021.htm)
- CREG (2025). *Documento CREG 901 202 de 2025*, soporte de la Resolución CREG 101 079 de 2025. [gestornormativo.creg.gov.co](https://gestornormativo.creg.gov.co/gestor/entorno/docs/pdf/doc_creg_0901_202_2025.pdf)
- UPME (2025). *Proyección de precios de los energéticos 2024-2050*, anexo de la Circular UPME 105 de 2025. [docs.upme.gov.co](https://docs.upme.gov.co/SIMEC/Planeacion-energetica/Documentos/Anexo_de_Proyeccion_de_precios_2024-2050.xlsx)
- UPME. *Costo incremental operativo de racionamiento*, datos abiertos (conjunto ptj2-kj7i). [datos.gov.co](https://www.datos.gov.co/resource/ptj2-kj7i.json)
- UPME (2026). *Informe de registros activos de proyectos de generación*, corte al 30 de septiembre de 2026. [docs.upme.gov.co](https://docs.upme.gov.co/SIMEC/Energia%20Electrica/Informes_Registro_Proyectos_Generacion/Informe_registros_activos_de_proyectos_generacion_electrica_septiembre_2026.xlsx)
- Asocaña (2026). *Informe Anual 2025-2026* (p. 39 y Cuadro 5, p. 63). [asocana.org](https://www.asocana.org/documentos/362026-9AEF4DC0-00FF00,000A000,878787,C3C3C3,0F0F0F,B4B4B4,FF00FF,FFFFFF,2D2D2D,A3C4B5.pdf).

**Costos de tecnologías**

- National Laboratory of the Rockies, antes NREL (2025). *Annual Technology Baseline 2025*: escenarios bajo y alto del CAPEX. [atb.nlr.gov](https://atb.nlr.gov/electricity/2025/index)
- CNE Chile (2026). *Informe de Costos de Tecnologías de Generación y Almacenamiento* (mayo de 2026). [cne.cl](https://www.cne.cl/wp-content/uploads/2026/06/ICTGA-2026.pdf)
- EPE y MME (2025). *PDE 2035: Caderno de Parâmetros de Custos de Geração e Transmissão* (revisión del 10 de noviembre de 2025). [epe.gov.br](https://www.epe.gov.br/pt/publicacoes-dados-abertos/publicacoes/plano-decenal-de-expansao-de-energia-2035)

**Prensa**

- Portafolio (2026). Nota sobre el Boletín 364 de XM: faltantes de energía firme de hasta 7.669 GWh en 2026-2027. [portafolio.co](https://www.portafolio.co/energia/colombia-podria-enfrentar-un-deficit-de-energia-en-2027-xm-advierte-faltantes-de-hasta-7-669-gwh-504633)

La documentación oficial de cada paquete está en la tabla de la sección 2.1. La fuente de cada parámetro y de cada supuesto, con su tabla o página, su URL y su conversión, está en `datos_tutorial/fuentes_supuestos.csv`.